# Submission 25

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:

from __future__ import annotations

import ast
import pickle
import types

import os
import sys
import time
import math
import json
import random
import hashlib
import inspect
import pkgutil
import importlib
import subprocess
import traceback
import warnings
import copy
import re
import logging
import importlib.util
import heapq
from pathlib import Path
from collections import defaultdict, deque
from dataclasses import dataclass, field, is_dataclass, asdict
from typing import Any, Optional

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SUBMISSION_PATH = WORKDIR / "submission.parquet"

# Early placeholder so Kaggle always sees the file, even if something fails later.
pd.DataFrame(
    [{"row_id": "0_0", "game_id": "0", "end_of_game": False, "score": 0.0}]
).to_parquet(SUBMISSION_PATH, index=False)


def _first_existing(paths: list[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def find_competition_root() -> Optional[Path]:
    direct = _first_existing(
        [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3"),
        ]
    )
    if direct is not None:
        return direct

    root = Path("/kaggle/input")
    if root.exists():
        for wheels in root.rglob("arc_agi_3_wheels"):
            return wheels.parent
    return None


COMP_ROOT = find_competition_root()
if COMP_ROOT is None:
    raise FileNotFoundError("Could not locate ARC-AGI-3 competition input directory.")

WHEELS_DIR = COMP_ROOT / "arc_agi_3_wheels"
ENVIRONMENTS_DIR = COMP_ROOT / "environment_files"
AGENTS_DIR = COMP_ROOT / "ARC-AGI-3-Agents"

if not WHEELS_DIR.exists():
    raise FileNotFoundError(f"Could not locate wheels directory: {WHEELS_DIR}")


def install_local_wheels(wheels_dir: Path) -> None:
    wheel_files = sorted(str(p) for p in wheels_dir.glob("*.whl"))
    if not wheel_files:
        raise FileNotFoundError(f"No wheel files found in {wheels_dir}")
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        str(wheels_dir),
        *wheel_files,
    ]
    subprocess.check_call(cmd)


# Packages are installed in the first notebook cell as required by the competition.


def find_symbol(package_name: str, symbol_name: str) -> Any:
    package = importlib.import_module(package_name)
    if hasattr(package, symbol_name):
        return getattr(package, symbol_name)
    if hasattr(package, "__path__"):
        for modinfo in pkgutil.walk_packages(package.__path__, package.__name__ + "."):
            try:
                module = importlib.import_module(modinfo.name)
            except Exception:
                continue
            if hasattr(module, symbol_name):
                return getattr(module, symbol_name)
    raise AttributeError(f"Could not find symbol '{symbol_name}' inside package '{package_name}'")


Arcade = find_symbol("arc_agi", "Arcade")
OperationMode = find_symbol("arc_agi", "OperationMode")
try:
    GameAction = find_symbol("arcengine", "GameAction")
    GameState = find_symbol("arcengine", "GameState")
except Exception:
    GameAction = find_symbol("arc_agi", "GameAction")
    GameState = find_symbol("arc_agi", "GameState")


def resolve_mode(prefer_competition: bool) -> Any:
    members = getattr(OperationMode, "__members__", {})
    ordered_names: list[str] = []
    if prefer_competition:
        ordered_names.extend(["COMPETITION", "COMPETITON", "ONLINE", "NORMAL", "LOCAL", "OFFLINE"])
    else:
        ordered_names.extend(["OFFLINE", "NORMAL", "LOCAL", "ONLINE", "COMPETITION", "COMPETITON"])
    for name in ordered_names:
        if name in members:
            return members[name]
    try:
        members_list = list(OperationMode)
        if members_list:
            return members_list[0]
    except Exception:
        pass
    return "online" if prefer_competition else "offline"


ACTION_MEMBERS = getattr(GameAction, "__members__", {})
RESET_ACTION = ACTION_MEMBERS.get("RESET", "RESET")
ACTION_ID_TO_ENUM: dict[int, Any] = {}
for i in range(1, 8):
    ACTION_ID_TO_ENUM[i] = ACTION_MEMBERS.get(f"ACTION{i}", f"ACTION{i}")

IS_RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

os.environ.setdefault("SCHEME", "http")
os.environ.setdefault("HOST", "gateway")
os.environ.setdefault("PORT", "8001")
os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
os.environ.setdefault("ARC_API_KEY", "test-key-123")
os.environ.setdefault("OPERATION_MODE", "COMPETITION" if IS_RERUN else "OFFLINE")

ROOT_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001/").rstrip("/")
HEADERS = {
    "X-API-Key": os.environ.get("ARC_API_KEY", ""),
    "Accept": "application/json",
}


def wait_for_gateway(base_url: str, headers: dict[str, str], timeout_s: int = 180) -> bool:
    if not IS_RERUN:
        return False
    deadline = time.time() + timeout_s
    last_err = None
    url = base_url.rstrip("/") + "/api/games"
    while time.time() < deadline:
        try:
            r = requests.get(url, headers=headers, timeout=5)
            if r.ok:
                return True
            last_err = f"{r.status_code}: {r.text[:200]}"
        except Exception as exc:
            last_err = repr(exc)
        time.sleep(2)
    print(f"Gateway unavailable, falling back to local environments when possible: {last_err}")
    return False


GATEWAY_READY = wait_for_gateway(ROOT_URL, HEADERS, timeout_s=600)


def make_arcade() -> Any:
    prefer_competition = IS_RERUN and GATEWAY_READY
    kwargs: dict[str, Any] = {}
    sig = inspect.signature(Arcade)
    op_mode = resolve_mode(prefer_competition)
    if "operation_mode" in sig.parameters:
        kwargs["operation_mode"] = op_mode
    if "arc_api_key" in sig.parameters and IS_RERUN:
        kwargs["arc_api_key"] = os.environ.get("ARC_API_KEY", "")
    if "arc_base_url" in sig.parameters and IS_RERUN:
        kwargs["arc_base_url"] = os.environ.get("ARC_BASE_URL", ROOT_URL)
    if "environments_dir" in sig.parameters and ENVIRONMENTS_DIR.exists():
        kwargs["environments_dir"] = str(ENVIRONMENTS_DIR)
    return Arcade(**kwargs)


def maybe_get(obj: Any, *names: str) -> Any:
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj[name]
        if hasattr(obj, name):
            return getattr(obj, name)
    return None


def deep_find(obj: Any, names: tuple[str, ...], max_depth: int = 3) -> Any:
    seen: set[int] = set()

    def rec(x: Any, depth: int) -> Any:
        if depth < 0 or x is None:
            return None
        try:
            oid = id(x)
            if oid in seen:
                return None
            seen.add(oid)
        except Exception:
            pass

        got = maybe_get(x, *names)
        if got is not None:
            return got

        if isinstance(x, dict):
            for v in x.values():
                found = rec(v, depth - 1)
                if found is not None:
                    return found
        elif isinstance(x, (list, tuple)):
            for v in x:
                found = rec(v, depth - 1)
                if found is not None:
                    return found
        else:
            d = getattr(x, "__dict__", None)
            if isinstance(d, dict):
                for v in d.values():
                    found = rec(v, depth - 1)
                    if found is not None:
                        return found
        return None

    return rec(obj, max_depth)


def to_plain(obj: Any, depth: int = 6) -> Any:
    if depth < 0:
        return None
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, dict):
        return {k: to_plain(v, depth - 1) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_plain(v, depth - 1) for v in obj]
    if is_dataclass(obj):
        return to_plain(asdict(obj), depth - 1)
    if hasattr(obj, "model_dump"):
        try:
            return to_plain(obj.model_dump(), depth - 1)
        except Exception:
            pass
    if hasattr(obj, "dict"):
        try:
            return to_plain(obj.dict(), depth - 1)
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        try:
            return {k: to_plain(v, depth - 1) for k, v in vars(obj).items() if not k.startswith("_")}
        except Exception:
            pass
    return repr(obj)


def normalize_action_id(action: Any) -> Optional[int]:
    if action is None:
        return None
    if isinstance(action, (int, np.integer)):
        return int(action)
    name = None
    if isinstance(action, str):
        name = action
    elif hasattr(action, "name"):
        name = str(action.name)
    elif hasattr(action, "value") and isinstance(action.value, str):
        name = action.value
    if name is not None:
        name = name.split(".")[-1].upper()
        if name == "RESET":
            return 0
        if name.startswith("ACTION"):
            try:
                return int(name.replace("ACTION", "").replace("_", ""))
            except Exception:
                return None
    if hasattr(action, "value") and isinstance(action.value, (int, np.integer)):
        return int(action.value)
    return None


def unique_preserve_order(items: list[Any]) -> list[Any]:
    out = []
    seen = set()
    for item in items:
        key = repr(item)
        if key in seen:
            continue
        seen.add(key)
        out.append(item)
    return out


def unwrap_observation(step_output: Any) -> Any:
    out = step_output
    if isinstance(out, tuple) and len(out) > 0:
        out = out[0]
    frame_data = maybe_get(out, "frame_data", "observation")
    if frame_data is not None:
        if deep_find(frame_data, ("frame", "frames", "grid", "grids"), 2) is not None:
            out = frame_data
    return out


def state_name(obs: Any) -> str:
    st = deep_find(obs, ("state", "game_state"), 2)
    if st is None:
        return ""
    if isinstance(st, str):
        return st.split(".")[-1].upper()
    if hasattr(st, "name"):
        return str(st.name).upper()
    return str(st).split(".")[-1].upper()


def levels_completed(obs: Any) -> int:
    val = deep_find(obs, ("levels_completed", "score"), 2)
    if val is None:
        return 0
    try:
        return int(val)
    except Exception:
        try:
            return int(float(val))
        except Exception:
            return 0


def extract_available_action_ids(obs: Any, env: Any = None) -> set[int]:
    raw = deep_find(obs, ("available_actions", "actions"), 2)
    if raw is None and env is not None:
        raw = maybe_get(env, "available_actions", "action_space")
        if raw is not None:
            raw = maybe_get(raw, "available_actions") or raw

    if raw is None:
        return {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}

    if isinstance(raw, dict):
        raw = raw.keys()

    ids: set[int] = set()
    try:
        iterable = list(raw)
    except Exception:
        iterable = []

    for a in iterable:
        aid = normalize_action_id(a)
        if aid is not None and aid != 0:
            ids.add(aid)

    if not ids:
        ids = {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}
    return ids


def extract_latest_frame(obs: Any) -> tuple[np.ndarray, int]:
    frame_obj = deep_find(obs, ("frame", "frames", "grid", "grids"), 3)
    if frame_obj is None:
        raise ValueError("Could not extract frame/grid from observation.")
    arr = np.asarray(frame_obj, dtype=np.uint8)
    if arr.ndim == 3:
        return arr[-1].copy(), int(arr.shape[0])
    if arr.ndim == 2:
        return arr.copy(), 1
    if arr.ndim == 1:
        side = int(round(math.sqrt(arr.size)))
        if side * side == arr.size:
            return arr.reshape(side, side).astype(np.uint8), 1
    raise ValueError(f"Unexpected frame shape: {arr.shape}")


def step_env(env: Any, action_token: Any, data: Optional[dict[str, int]] = None, reasoning: Optional[str] = None) -> Any:
    patterns = [
        ((action_token,), {"data": data, "reasoning": reasoning}),
        ((action_token,), {"data": data}),
        ((action_token,), {"reasoning": reasoning}),
        ((action_token,), {}),
    ]
    if data is not None:
        patterns.extend(
            [
                ((action_token, data, reasoning), {}),
                ((action_token, data), {}),
            ]
        )

    last_exc = None
    for args, kwargs in patterns:
        try:
            return env.step(*args, **{k: v for k, v in kwargs.items() if v is not None})
        except TypeError as exc:
            last_exc = exc
        except Exception:
            raise
    raise last_exc or RuntimeError("Failed to call env.step")


def take_action(env: Any, action_id: int, data: Optional[dict[str, int]] = None, reasoning: Optional[str] = None) -> Any:
    action_enum = RESET_ACTION if action_id == 0 else ACTION_ID_TO_ENUM.get(action_id, action_id)
    tokens = []
    if action_enum is not None:
        tokens.append(action_enum)
        name = getattr(action_enum, "name", None)
        if name:
            tokens.append(name)
        value = getattr(action_enum, "value", None)
        if value is not None:
            tokens.append(value)
    tokens.append(action_id)
    tokens = unique_preserve_order(tokens)

    last_exc = None
    for tok in tokens:
        try:
            out = step_env(env, tok, data=data, reasoning=reasoning)
            return unwrap_observation(out)
        except TypeError as exc:
            last_exc = exc
        except Exception:
            raise
    raise last_exc or RuntimeError(f"Could not step environment with action {action_id}")


def perform_reset(env: Any, reason: str = "reset", max_attempts: int = 3) -> tuple[Any, int]:
    obs = None
    used = 0
    for _ in range(max_attempts):
        obs = take_action(env, 0, data=None, reasoning=reason)
        used += 1
        if state_name(obs) not in {"NOT_PLAYED", ""}:
            break
    return obs, used


def normalize_game_ids(items: Any) -> list[str]:
    ids: list[str] = []
    for item in items or []:
        gid = None
        if isinstance(item, str):
            gid = item
        elif isinstance(item, dict):
            gid = item.get("game_id") or item.get("id")
        else:
            gid = getattr(item, "game_id", None) or getattr(item, "id", None)
            if gid is None:
                text = str(item)
                if text and text != object.__repr__(item):
                    gid = text
        if gid is not None:
            ids.append(str(gid))
    return ids


INFINITY = np.iinfo(np.int32).max
EDGE_DTYPE = np.dtype(
    [
        ("group", "i4"),
        ("result", "i4"),
        ("target", "U64"),
        ("distance", "i4"),
        ("errors", "i4"),
    ]
)


@dataclass
class NodeInfo:
    name: str
    total_candidates: int
    num_groups: int
    group2remaining_candidate_ids: list[set[int]]
    error_threshold: int = 3
    closed: bool = False
    distance: int = INFINITY
    edge_data: np.ndarray = field(init=False)

    def __post_init__(self) -> None:
        self.group2remaining_candidate_ids = [set(s) for s in self.group2remaining_candidate_ids]
        self.edge_data = np.zeros(self.total_candidates, dtype=EDGE_DTYPE)
        for group_id, candidate_ids in enumerate(self.group2remaining_candidate_ids):
            if candidate_ids:
                self.edge_data["group"][list(candidate_ids)] = group_id

    def has_open_group(self, group_id: int) -> bool:
        for gid in range(min(group_id, self.num_groups - 1) + 1):
            if self.group2remaining_candidate_ids[gid]:
                return True
        return False

    def record_test(self, edge_idx: int, success: int, target_node: Optional[str] = None) -> bool:
        if edge_idx < 0 or edge_idx >= self.total_candidates:
            return False

        current_result = int(self.edge_data["result"][edge_idx])
        if current_result != 0:
            if success == 1 and target_node and str(target_node) != str(self.edge_data["target"][edge_idx]):
                # keep the shorter path if the node is being repaired later
                pass
            else:
                return False

        edge_group_id = int(self.edge_data["group"][edge_idx])

        if success == -1:
            self.edge_data["errors"][edge_idx] += 1
            if self.edge_data["errors"][edge_idx] < self.error_threshold:
                return False
            self.edge_data["errors"][edge_idx] = 0
            new_group_id = edge_group_id + 1
            self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)
            if new_group_id <= self.num_groups - 1:
                self.edge_data["group"][edge_idx] = new_group_id
                self.group2remaining_candidate_ids[new_group_id].add(edge_idx)
                return False
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
            return True

        self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)

        if success == 1:
            self.edge_data["result"][edge_idx] = 1
            self.edge_data["target"][edge_idx] = str(target_node or "")
            self.edge_data["distance"][edge_idx] = -1
        else:
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
        return True


class GraphExplorer:
    def __init__(self, n_groups: int = 5, verbose: int = 0):
        self._n_groups = max(1, int(n_groups))
        self._verbose = verbose
        self.reset()

    def reset(self) -> None:
        self._nodes: dict[str, NodeInfo] = {}
        self._G: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._G_rev: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._frontier: set[str] = set()
        self._dist: dict[str, int] = {}
        self._next: dict[str, tuple[int, str]] = {}
        self._active_group = 0
        self._empty = True
        self.suspicious_transitions: dict[tuple[str, int, str], int] = {}
        self.suspicious_transitions_threshold = 3

    @property
    def active_group(self) -> int:
        return self._active_group

    @property
    def frontier(self) -> set[str]:
        return self._frontier

    def distance(self, node: str) -> int:
        return int(self._dist.get(node, INFINITY))

    def has_node(self, node: str) -> bool:
        return node in self._nodes

    def initialize(self, start_node: str, num_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        self._add_new_node(start_node, num_candidates, group2remaining_candidate_ids)

    def _add_new_node(self, node: str, n_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        if node in self._nodes:
            return
        self._nodes[node] = NodeInfo(
            name=node,
            total_candidates=n_candidates,
            num_groups=self._n_groups,
            group2remaining_candidate_ids=group2remaining_candidate_ids,
        )
        self._G[node] = set()
        self._G_rev[node] = set()
        self._empty = False
        if self._nodes[node].has_open_group(self._active_group):
            self._frontier.add(node)
        else:
            self._close_node(node)

    def _remove_edge(self, node: str, edge_idx: int) -> None:
        for existing_idx, target in list(self._G.get(node, set())):
            if int(existing_idx) == int(edge_idx):
                self._G[node].discard((existing_idx, target))
                self._G_rev[target].discard((existing_idx, node))


    def _close_node(self, node: str) -> None:
        info = self._nodes[node]
        if info.closed:
            return
        info.closed = True
        self._frontier.discard(node)
        self._rebuild_distances()

    def _rebuild_distances(self) -> None:
        self._dist.clear()
        self._next.clear()

        dq = deque()
        for node, info in self._nodes.items():
            info.distance = INFINITY
            self._dist[node] = INFINITY

        for node in self._frontier:
            self._dist[node] = 0
            self._nodes[node].distance = 0
            dq.append(node)

        while dq:
            v = dq.popleft()
            v_dist = self._dist.get(v, INFINITY)
            for edge_idx, u in self._G_rev.get(v, ()):
                cand_dist = v_dist + 1
                self._nodes[u].edge_data["distance"][edge_idx] = cand_dist
                if cand_dist < self._dist.get(u, INFINITY):
                    self._dist[u] = cand_dist
                    self._nodes[u].distance = cand_dist
                    self._next[u] = (edge_idx, v)
                    dq.append(u)

    def _maybe_advance_group(self, current_node: str) -> None:
        while self.distance(current_node) >= INFINITY and self._active_group < self._n_groups - 1:
            self._active_group += 1
            self._frontier.clear()
            for node, info in self._nodes.items():
                info.closed = False
                if info.has_open_group(self._active_group):
                    self._frontier.add(node)
            self._rebuild_distances()

    def record_test(
        self,
        node: str,
        edge_idx: int,
        success: int,
        target_node: Optional[str] = None,
        target_num_candidates: Optional[int] = None,
        group2remaining_candidate_ids: Optional[list[set[int]]] = None,
        suspicious_transition: bool = False,
    ) -> None:
        if node not in self._nodes:
            return

        if suspicious_transition and success == 1 and target_node is not None:
            key = (node, edge_idx, target_node)
            self.suspicious_transitions[key] = self.suspicious_transitions.get(key, 0) + 1
            if self.suspicious_transitions[key] < self.suspicious_transitions_threshold:
                return

        node_info = self._nodes[node]
        wrote = node_info.record_test(edge_idx, int(success), target_node)
        if not wrote:
            return

        if success == 1 and target_node is not None:
            self._remove_edge(node, edge_idx)
            if target_node not in self._nodes:
                if target_num_candidates is None or group2remaining_candidate_ids is None:
                    return
                self._add_new_node(target_node, target_num_candidates, group2remaining_candidate_ids)
            self._G[node].add((edge_idx, target_node))
            self._G_rev[target_node].add((edge_idx, node))

            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)

            if self._nodes[target_node].has_open_group(self._active_group):
                self._rebuild_distances()
            else:
                self._close_node(target_node)
                self._maybe_advance_group(target_node)
        else:
            self._remove_edge(node, edge_idx)
            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)
                self._maybe_advance_group(node)

    def open_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        if not node_info.has_open_group(self._active_group):
            return []
        out: list[int] = []
        for gid in range(self._active_group + 1):
            out.extend(sorted(node_info.group2remaining_candidate_ids[gid]))
        return out

    def successful_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        lowest_dist = self.distance(node)
        if lowest_dist >= INFINITY:
            return []
        return [
            idx
            for idx, row in enumerate(node_info.edge_data)
            if int(row["result"]) == 1
            and int(row["group"]) <= self._active_group
            and int(row["distance"]) <= lowest_dist
        ]

    def edge_status(self, node: str, edge_idx: int) -> int:
        if node not in self._nodes:
            return 0
        node_info = self._nodes[node]
        if edge_idx < 0 or edge_idx >= node_info.total_candidates:
            return 0
        return int(node_info.edge_data["result"][edge_idx])

    def edge_distance(self, node: str, edge_idx: int) -> int:
        if node not in self._nodes:
            return INFINITY
        node_info = self._nodes[node]
        if edge_idx < 0 or edge_idx >= node_info.total_candidates:
            return INFINITY
        return int(node_info.edge_data["distance"][edge_idx])

    def choose_edge(self, node: str) -> Optional[int]:
        open_ids = self.open_candidate_ids(node)
        if open_ids:
            return random.choice(open_ids)
        options = self.successful_candidate_ids(node)
        if options:
            return random.choice(options)
        return None


@dataclass
class BetaCounter:
    success: float = 1.0
    failure: float = 1.0

    def mean(self) -> float:
        denom = self.success + self.failure
        return float(self.success / denom) if denom > 0 else 0.5

    @property
    def count(self) -> float:
        return max(0.0, self.success + self.failure - 2.0)

    def update(self, success_inc: float, failure_inc: float) -> None:
        self.success += max(0.0, float(success_inc))
        self.failure += max(0.0, float(failure_inc))


class FeatureMemory:
    def __init__(self) -> None:
        self._stats: dict[tuple[Any, ...], BetaCounter] = {}
        self.total_updates = 0.0

    def _get(self, key: tuple[Any, ...]) -> BetaCounter:
        if key not in self._stats:
            self._stats[key] = BetaCounter()
        return self._stats[key]

    def mean(self, key: tuple[Any, ...]) -> float:
        return self._get(key).mean()

    def count(self, key: tuple[Any, ...]) -> float:
        return self._get(key).count

    def update(self, key: tuple[Any, ...], success_inc: float, failure_inc: float) -> None:
        self._get(key).update(success_inc, failure_inc)
        self.total_updates += max(0.0, float(success_inc)) + max(0.0, float(failure_inc))

    def _candidate_weighted_keys(self, candidate: dict[str, Any]) -> list[tuple[tuple[Any, ...], float]]:
        feats = candidate.get("features") or {}
        kind = str(candidate.get("kind", ""))
        action_id = int(candidate.get("action_id", -1))

        if kind == "simple":
            return [
                (("kind", "simple"), 0.35),
                (("aid", action_id), 0.65),
            ]

        if kind == "click":
            source = str(feats.get("source", "click"))
            group = int(candidate.get("group", feats.get("group", 4)))
            color = int(feats.get("color", -1))
            area_bucket = int(feats.get("area_bucket", -1))
            rect = int(feats.get("rect", 0))
            edge_mask = int(feats.get("edge_mask", 0))
            rep_idx = int(feats.get("rep_idx", -1))
            bin_x = int(feats.get("bin_x", -1))
            bin_y = int(feats.get("bin_y", -1))
            return [
                (("kind", "click"), 0.10),
                (("aid", action_id), 0.18),
                (("group", group), 0.18),
                (("source", source), 0.08),
                (("color", color), 0.12),
                (("group_color", group, color), 0.10),
                (("bin", bin_x, bin_y), 0.10),
                (("rep", rep_idx), 0.04),
                (("area", area_bucket), 0.04),
                (("rect", rect), 0.03),
                (("edge", edge_mask), 0.03),
            ]

        return [(("kind", kind or "other"), 1.0)]

    def estimate(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        if not weighted:
            return 0.5
        total_w = 0.0
        total_v = 0.0
        for key, w in weighted:
            total_w += float(w)
            total_v += float(w) * self.mean(key)
        return float(total_v / total_w) if total_w > 0 else 0.5

    def exposure(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        if not weighted:
            return 0.0
        total_w = 0.0
        total_v = 0.0
        for key, w in weighted:
            total_w += float(w)
            total_v += float(w) * self.count(key)
        return float(total_v / total_w) if total_w > 0 else 0.0

    def update_candidate(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if candidate.get("kind") == "meta_reset":
            return

        if level_up:
            success_inc = 1.20
            failure_inc = 0.00
        elif changed and not suspicious:
            success_inc = 0.65
            failure_inc = 0.35
        elif changed and suspicious:
            success_inc = 0.20
            failure_inc = 0.80
        elif game_over or errored:
            success_inc = 0.00
            failure_inc = 1.25
        else:
            success_inc = 0.00
            failure_inc = 1.00

        for key, _ in self._candidate_weighted_keys(candidate):
            self.update(key, success_inc, failure_inc)

    def kind_bias(self) -> float:
        return float(self.mean(("kind", "click")) - self.mean(("kind", "simple")))


GLOBAL_FEATURE_MEMORY = FeatureMemory()


@dataclass
class FrameView:
    node_id: str
    hash_id: str
    raw_frame: np.ndarray
    masked_frame_for_segmentation: np.ndarray
    hash_frame_input: np.ndarray
    num_frames: int
    state: str
    levels_completed: int
    available_action_ids: set[int]
    undo_available: bool
    segments: list[dict[str, Any]]
    candidates: list[dict[str, Any]]
    groups: list[set[int]]


class FrameProcessor:
    OFFSETS4 = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self) -> None:
        self.frame_shape = (64, 64)
        self.status_bar_color = 16
        self.salient_colors = {6, 7, 8, 9, 10, 11, 12, 13, 14, 15}
        self.non_salient_colors = {0, 1, 2, 3, 4, 5}
        self.status_bar_ratio_threshold = 5.0
        self.status_bar_twins_threshold = 3

    def _set_shape(self, shape: tuple[int, int]) -> None:
        self.frame_shape = tuple(int(v) for v in shape)

    @property
    def status_bar_distance_threshold(self) -> int:
        return max(1, min(3, min(self.frame_shape) // 12 + 1))

    @property
    def minimal_width(self) -> int:
        return 2 if min(self.frame_shape) >= 8 else 1

    @property
    def maximal_width(self) -> int:
        return max(2, min(self.frame_shape) // 2)

    def segment_frame(self, frame: np.ndarray) -> tuple[np.ndarray, list[dict[str, Any]]]:
        frame = np.asarray(frame, dtype=np.uint8)
        self._set_shape(tuple(frame.shape))
        h, w = frame.shape
        label_map = np.full((h, w), -1, dtype=np.int32)
        components: list[dict[str, Any]] = []
        cid = -1

        for y in range(h):
            for x in range(w):
                if label_map[y, x] != -1:
                    continue
                cid += 1
                color = int(frame[y, x])
                q = deque([(y, x)])
                label_map[y, x] = cid

                min_x = max_x = x
                min_y = max_y = y
                area = 0
                points: list[tuple[int, int]] = []

                while q:
                    cy, cx = q.popleft()
                    points.append((cy, cx))
                    area += 1
                    min_x = min(min_x, cx)
                    max_x = max(max_x, cx)
                    min_y = min(min_y, cy)
                    max_y = max(max_y, cy)
                    for dy, dx in self.OFFSETS4:
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < h and 0 <= nx < w and label_map[ny, nx] == -1 and int(frame[ny, nx]) == color:
                            label_map[ny, nx] = cid
                            q.append((ny, nx))

                rect_area = (max_x - min_x + 1) * (max_y - min_y + 1)
                components.append(
                    {
                        "bounding_box": (min_x, min_y, max_x, max_y),
                        "color": color,
                        "area": area,
                        "is_rectangle": area == rect_area,
                        "points": points,
                    }
                )

        buckets: dict[tuple[int, bool, int], list[int]] = defaultdict(list)
        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            buckets[key].append(i)

        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            twins = [j for j in buckets[key] if j != i]
            comp["number_of_twins"] = len(twins)
            comp["twin_ids"] = twins

        return label_map, components

    def check_segment_fully_on_edge(self, segment: dict[str, Any], edges: Optional[list[str]] = None) -> list[str]:
        x1, y1, x2, y2 = segment["bounding_box"]
        thr = self.status_bar_distance_threshold
        if edges is None:
            edges = ["any"]
        result = []
        if "left" in edges or "any" in edges:
            if max(x1, x2) < thr:
                result.append("left")
        if "right" in edges or "any" in edges:
            if min(x1, x2) >= self.frame_shape[1] - thr:
                result.append("right")
        if "top" in edges or "any" in edges:
            if max(y1, y2) < thr:
                result.append("top")
        if "bottom" in edges or "any" in edges:
            if min(y1, y2) >= self.frame_shape[0] - thr:
                result.append("bottom")
        return result

    def check_segment_ratio(self, segment: dict[str, Any], direction: str = "any") -> bool:
        x1, y1, x2, y2 = segment["bounding_box"]
        x_len = x2 - x1 + 1
        y_len = y2 - y1 + 1
        ratio = x_len / max(y_len, 1)
        if ratio >= self.status_bar_ratio_threshold and direction in ("any", "horizontal"):
            return True
        if ratio <= 1.0 / self.status_bar_ratio_threshold and direction in ("any", "vertical"):
            return True
        return False

    def segment_twins_on_edge(self, segment: dict[str, Any], frame_segments: list[dict[str, Any]], edges: Optional[list[str]] = None) -> list[int]:
        if edges is None:
            edges = self.check_segment_fully_on_edge(segment, ["any"])
            if not edges:
                return []
        twins = []
        for twin_id in segment["twin_ids"]:
            twin_edges = self.check_segment_fully_on_edge(frame_segments[twin_id], edges)
            if twin_edges:
                twins.append(twin_id)
        return twins

    def identify_status_bars(self, segmented_frame: np.ndarray, frame_segments: list[dict[str, Any]]) -> np.ndarray:
        self._set_shape(tuple(segmented_frame.shape))
        checked: set[int] = set()
        status_segment_groups: list[list[int]] = []

        for i, segment in enumerate(frame_segments):
            if i in checked:
                continue
            checked.add(i)
            on_edge = self.check_segment_fully_on_edge(segment, ["any"])
            if not on_edge:
                continue

            dirs = []
            if "left" in on_edge or "right" in on_edge:
                dirs.append("vertical")
            if "top" in on_edge or "bottom" in on_edge:
                dirs.append("horizontal")
            direction = "any" if len(dirs) == 2 else dirs[0]

            candidate_ids = [i]
            is_long = self.check_segment_ratio(segment, direction)
            if not is_long:
                twin_ids = self.segment_twins_on_edge(segment, frame_segments)
                for twin_id in twin_ids:
                    checked.add(twin_id)
                if len(twin_ids) + 1 < self.status_bar_twins_threshold:
                    continue
                candidate_ids.extend(twin_ids)

            status_segment_groups.append(candidate_ids)

        mask = np.zeros(segmented_frame.shape, dtype=bool)
        for group in status_segment_groups:
            for seg_id in group:
                mask[segmented_frame == seg_id] = True
        return mask

    def hash_frame(self, frame: np.ndarray) -> str:
        frame = np.asarray(frame, dtype=np.uint8, order="C")
        flat = frame.ravel()
        if flat.size % 2 == 1:
            flat = np.concatenate([flat, np.zeros(1, dtype=np.uint8)])
        packed = (flat[0::2] << 4) | (flat[1::2] & 0x0F)
        return hashlib.blake2b(
            packed.tobytes(),
            digest_size=16,
            person=repr(frame.shape).encode(),
        ).hexdigest()

    def segment_group(self, segment: dict[str, Any]) -> int:
        x1, y1, x2, y2 = segment["bounding_box"]
        xw = x2 - x1 + 1
        yw = y2 - y1 + 1
        color = int(segment["color"])
        is_salient = color in self.salient_colors
        is_medium = self.minimal_width <= xw <= self.maximal_width and self.minimal_width <= yw <= self.maximal_width
        is_status_bar = color == self.status_bar_color

        if is_salient and is_medium:
            return 0
        if is_medium:
            return 1
        if is_salient:
            return 2
        if not is_status_bar:
            return 3
        return 4


    def representative_points(self, segment: dict[str, Any]) -> list[tuple[int, int]]:
        pts = segment["points"]
        if not pts:
            return []
        x1, y1, x2, y2 = segment["bounding_box"]
        cy = 0.5 * (y1 + y2)
        cx = 0.5 * (x1 + x2)

        def dist2(p: tuple[int, int], tx: float, ty: float) -> float:
            py, px = p
            return (py - ty) ** 2 + (px - tx) ** 2

        def nearest(tx: float, ty: float) -> tuple[int, int]:
            return min(pts, key=lambda p: dist2(p, tx, ty))

        center_point = nearest(cx, cy)
        first_point = min(pts)
        last_point = max(pts)

        out = [center_point]
        if len(pts) >= 12:
            out.append(first_point)
        if len(pts) >= 32:
            out.append(last_point)

        width = x2 - x1 + 1
        height = y2 - y1 + 1
        if len(pts) >= 24 and max(width, height) >= (2 * min(width, height) + 2):
            if width >= height:
                out.append(nearest(x1 + 0.25 * (x2 - x1), cy))
                out.append(nearest(x1 + 0.75 * (x2 - x1), cy))
            else:
                out.append(nearest(cx, y1 + 0.25 * (y2 - y1)))
                out.append(nearest(cx, y1 + 0.75 * (y2 - y1)))

        dedup = []
        seen = set()
        for py, px in out:
            key = (int(px), int(py))
            if key not in seen:
                seen.add(key)
                dedup.append((int(px), int(py)))
        return dedup

    def edge_mask(self, segment: dict[str, Any]) -> int:
        mask = 0
        for edge in self.check_segment_fully_on_edge(segment, ["any"]):
            if edge == "left":
                mask |= 1
            elif edge == "right":
                mask |= 2
            elif edge == "top":
                mask |= 4
            elif edge == "bottom":
                mask |= 8
        return int(mask)

    def area_bucket(self, area: int) -> int:
        return int(min(7, max(0, int(math.log2(max(1, int(area)))))))

    def candidate_features(
        self,
        *,
        kind: str,
        action_id: int,
        group: int,
        x: Optional[int] = None,
        y: Optional[int] = None,
        raw_frame: Optional[np.ndarray] = None,
        segment: Optional[dict[str, Any]] = None,
        source: str = "simple",
        rep_idx: int = 0,
    ) -> dict[str, Any]:
        out: dict[str, Any] = {
            "kind": str(kind),
            "action_id": int(action_id),
            "group": int(group),
            "source": str(source),
        }
        if kind == "simple":
            return out

        h, w = raw_frame.shape if raw_frame is not None else self.frame_shape
        xi = int(0 if x is None else x)
        yi = int(0 if y is None else y)
        out["x"] = xi
        out["y"] = yi
        out["bin_x"] = int(min(3, (4 * xi) // max(1, w - 1))) if w > 1 else 0
        out["bin_y"] = int(min(3, (4 * yi) // max(1, h - 1))) if h > 1 else 0
        out["rep_idx"] = int(rep_idx)

        if segment is not None:
            out["color"] = int(segment["color"])
            out["area_bucket"] = self.area_bucket(int(segment["area"]))
            out["rect"] = int(bool(segment["is_rectangle"]))
            out["edge_mask"] = self.edge_mask(segment)
        else:
            color = int(raw_frame[yi, xi]) if raw_frame is not None else -1
            out["color"] = color
            out["area_bucket"] = 0
            out["rect"] = 1
            edge_mask = 0
            if xi <= 1:
                edge_mask |= 1
            if xi >= w - 2:
                edge_mask |= 2
            if yi <= 1:
                edge_mask |= 4
            if yi >= h - 2:
                edge_mask |= 8
            out["edge_mask"] = int(edge_mask)
        return out

    def coarse_grid_points(self, frame: np.ndarray, invalid_mask: np.ndarray, grid_n: int = 4) -> list[tuple[int, int]]:
        h, w = frame.shape
        ys = np.linspace(0, h - 1, grid_n).round().astype(int).tolist()
        xs = np.linspace(0, w - 1, grid_n).round().astype(int).tolist()

        points: list[tuple[int, int]] = []
        seen = set()
        for y in ys:
            for x in xs:
                if not invalid_mask[y, x]:
                    key = (x, y)
                    if key not in seen:
                        seen.add(key)
                        points.append(key)
                    continue
                found = None
                for radius in range(1, 4):
                    for dy in range(-radius, radius + 1):
                        for dx in range(-radius, radius + 1):
                            ny, nx = y + dy, x + dx
                            if 0 <= ny < h and 0 <= nx < w and not invalid_mask[ny, nx]:
                                found = (nx, ny)
                                break
                        if found is not None:
                            break
                    if found is not None:
                        break
                if found is not None and found not in seen:
                    seen.add(found)
                    points.append(found)
        return points



class LevelExplorerAgent:
    INVERSE_ACTION = {1: 2, 2: 1, 3: 4, 4: 3}

    def __init__(self, game_id: str, n_groups: int = 5, global_memory: Optional[FeatureMemory] = None) -> None:
        self.game_id = str(game_id)
        self.n_groups = n_groups
        self.fp = FrameProcessor()
        self.graph = GraphExplorer(n_groups=n_groups)
        self.global_memory = global_memory if global_memory is not None else GLOBAL_FEATURE_MEMORY
        seed_bytes = hashlib.blake2b(self.game_id.encode(), digest_size=4).digest()
        seed_int = int.from_bytes(seed_bytes, "big") ^ SEED
        self.py_rng = random.Random(seed_int)
        self.np_rng = np.random.default_rng(seed_int)
        self.reset_level_state()

    def reset_level_state(self) -> None:
        self.level_memory = FeatureMemory()
        self.status_bar_mask: Optional[np.ndarray] = None
        self.start_node: Optional[str] = None
        self.view_cache: dict[str, FrameView] = {}
        self.graph.reset()
        self.steps_in_level = 0
        self.resets_in_level = 0
        self.steps_since_new_node = 0
        self.steps_since_hash_change = 0
        self.last_unique_nodes = 0
        self.undo_stack: list[str] = []
        self.last_changed_action_id: Optional[int] = None

    def remember(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if candidate.get("kind") in {"meta_reset", "undo"}:
            return
        self.level_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )
        self.global_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )

    def candidate_priority(self, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        kind = str(candidate.get("kind", ""))
        aid = int(candidate.get("action_id", -1))

        if kind == "undo":
            score = -0.35 - 0.06 * float(distance)
            score += 1e-6 * self.py_rng.random()
            return float(score)

        local_est = self.level_memory.estimate(candidate)
        global_est = self.global_memory.estimate(candidate)
        prior = 0.72 * local_est + 0.28 * global_est

        local_exposure = self.level_memory.exposure(candidate)
        global_exposure = min(16.0, self.global_memory.exposure(candidate))
        exposure = 0.82 * local_exposure + 0.18 * global_exposure
        novelty = 1.0 / math.sqrt(1.0 + exposure)

        group = int(candidate.get("group", 0))
        feats = candidate.get("features") or {}
        source = str(feats.get("source", ""))

        if path_mode:
            score = 1.62 * prior + 0.07 * novelty - 0.21 * float(distance)
        else:
            score = 1.32 * prior + 0.13 * novelty - 0.20 * float(group)

        kind_bias = 0.60 * self.level_memory.kind_bias() + 0.20 * self.global_memory.kind_bias()
        if kind == "click":
            score += 0.08 * kind_bias
            if source == "segment":
                score += 0.03
            color = int(feats.get("color", -1))
            if color in self.fp.salient_colors:
                score += 0.04
        elif kind == "simple":
            score -= 0.05 * kind_bias
            if aid == 5:
                score += 0.03
            if aid in (1, 2, 3, 4):
                if self.last_changed_action_id == aid:
                    score += 0.08
                inv = self.INVERSE_ACTION.get(self.last_changed_action_id)
                if inv is not None and inv == aid:
                    score -= 0.10

        score += 1e-6 * self.py_rng.random()
        return float(score)

    def _segment_priority_key(self, seg: dict[str, Any]) -> tuple[Any, ...]:
        group = self.fp.segment_group(seg)
        color = int(seg["color"])
        salient = 0 if color in self.fp.salient_colors else 1
        status = 1 if color == self.fp.status_bar_color else 0
        area = int(seg["area"])
        x1, y1, x2, y2 = seg["bounding_box"]
        width = x2 - x1 + 1
        height = y2 - y1 + 1
        mediumish = 0 if (self.fp.minimal_width <= width <= self.fp.maximal_width and self.fp.minimal_width <= height <= self.fp.maximal_width) else 1
        return (group, status, salient, mediumish, -min(area, 256), y1, x1)

    def _build_view(self, obs: Any, env: Any) -> FrameView:
        raw_frame, num_frames = extract_latest_frame(obs)
        state = state_name(obs)
        lvl_done = levels_completed(obs)
        available_action_ids = extract_available_action_ids(obs, env)
        undo_available = 7 in available_action_ids

        if self.status_bar_mask is None or self.status_bar_mask.shape != raw_frame.shape:
            initial_labels, initial_segments = self.fp.segment_frame(raw_frame)
            self.status_bar_mask = self.fp.identify_status_bars(initial_labels, initial_segments)

        masked_for_seg = raw_frame.copy()
        masked_for_seg[self.status_bar_mask] = self.fp.status_bar_color
        hash_input = masked_for_seg.copy()
        hash_input[hash_input == self.fp.status_bar_color] = 0
        base_hash = self.fp.hash_frame(hash_input)
        action_sig = ",".join(str(a) for a in sorted(available_action_ids))
        node_id = f"{base_hash}|a:{action_sig}"

        if node_id in self.view_cache:
            cached = self.view_cache[node_id]
            return FrameView(
                node_id=cached.node_id,
                hash_id=cached.hash_id,
                raw_frame=raw_frame,
                masked_frame_for_segmentation=cached.masked_frame_for_segmentation,
                hash_frame_input=cached.hash_frame_input,
                num_frames=num_frames,
                state=state,
                levels_completed=lvl_done,
                available_action_ids=set(available_action_ids),
                undo_available=undo_available,
                segments=cached.segments,
                candidates=cached.candidates,
                groups=cached.groups,
            )

        labels, segments = self.fp.segment_frame(masked_for_seg)
        candidates: list[dict[str, Any]] = []
        candidate_groups: list[set[int]] = [set() for _ in range(self.n_groups)]

        if 6 in available_action_ids:
            seen_clicks: set[tuple[int, int]] = set()
            per_group_caps = [18, 14, 10, 8, 4]
            per_group_counts = [0 for _ in range(self.n_groups)]

            ordered_segments = sorted(enumerate(segments), key=lambda item: self._segment_priority_key(item[1]))
            for seg_id, seg in ordered_segments:
                base_group = self.fp.segment_group(seg)
                reps = self.fp.representative_points(seg)
                if int(seg["color"]) == self.fp.status_bar_color:
                    reps = reps[:1]
                    base_group = self.n_groups - 1

                for rep_idx, (x, y) in enumerate(reps):
                    group = min(base_group + (1 if rep_idx > 0 else 0), self.n_groups - 1)
                    if per_group_counts[group] >= per_group_caps[group]:
                        continue
                    key = (int(x), int(y))
                    if key in seen_clicks:
                        continue
                    seen_clicks.add(key)
                    cand_idx = len(candidates)
                    candidates.append(
                        {
                            "kind": "click",
                            "action_id": 6,
                            "data": {"x": int(x), "y": int(y)},
                            "group": group,
                            "tag": ("seg", seg_id, rep_idx, int(x), int(y)),
                            "features": self.fp.candidate_features(
                                kind="click",
                                action_id=6,
                                group=group,
                                x=int(x),
                                y=int(y),
                                raw_frame=raw_frame,
                                segment=seg,
                                source="segment",
                                rep_idx=rep_idx,
                            ),
                        }
                    )
                    candidate_groups[group].add(cand_idx)
                    per_group_counts[group] += 1

            invalid = self.status_bar_mask.copy()
            for x, y in seen_clicks:
                invalid[int(y), int(x)] = True
            click_only = set(available_action_ids).issubset({6, 7})
            grid_n = 5 if click_only else 4
            coarse = self.fp.coarse_grid_points(raw_frame, invalid_mask=invalid, grid_n=grid_n)
            coarse_cap = 10 if click_only else 6
            for j, (x, y) in enumerate(coarse[:coarse_cap]):
                group = self.n_groups - 2 if j < max(4, coarse_cap // 2) else self.n_groups - 1
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "click",
                        "action_id": 6,
                        "data": {"x": int(x), "y": int(y)},
                        "group": group,
                        "tag": ("grid", j, int(x), int(y)),
                        "features": self.fp.candidate_features(
                            kind="click",
                            action_id=6,
                            group=group,
                            x=int(x),
                            y=int(y),
                            raw_frame=raw_frame,
                            segment=None,
                            source="grid",
                            rep_idx=0,
                        ),
                    }
                )
                candidate_groups[group].add(cand_idx)

        for aid in sorted(available_action_ids):
            if aid in {1, 2, 3, 4, 5}:
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "simple",
                        "action_id": int(aid),
                        "data": None,
                        "group": 0,
                        "tag": ("simple", int(aid)),
                        "features": self.fp.candidate_features(
                            kind="simple",
                            action_id=int(aid),
                            group=0,
                            source="simple",
                        ),
                    }
                )
                candidate_groups[0].add(cand_idx)
            elif aid == 7:
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "undo",
                        "action_id": 7,
                        "data": None,
                        "group": self.n_groups - 1,
                        "tag": ("undo", 7),
                        "features": {
                            "kind": "undo",
                            "action_id": 7,
                            "group": self.n_groups - 1,
                            "source": "undo",
                        },
                    }
                )
                candidate_groups[self.n_groups - 1].add(cand_idx)

        view = FrameView(
            node_id=node_id,
            hash_id=base_hash,
            raw_frame=raw_frame,
            masked_frame_for_segmentation=masked_for_seg,
            hash_frame_input=hash_input,
            num_frames=num_frames,
            state=state,
            levels_completed=lvl_done,
            available_action_ids=set(available_action_ids),
            undo_available=undo_available,
            segments=segments,
            candidates=candidates,
            groups=candidate_groups,
        )
        self.view_cache[node_id] = view
        return view

    def start_level(self, obs: Any, env: Any) -> FrameView:
        self.reset_level_state()
        view = self._build_view(obs, env)
        self.start_node = view.node_id
        self.graph.initialize(view.node_id, len(view.candidates), view.groups)
        self.last_unique_nodes = len(self.graph._nodes)
        return view

    def observe_transition(
        self,
        prev_view: FrameView,
        candidate_idx: int,
        next_obs: Any,
        env: Any,
        game_over: bool = False,
        level_up: bool = False,
    ) -> Optional[FrameView]:
        self.steps_in_level += 1
        candidate = prev_view.candidates[candidate_idx]

        if game_over:
            self.graph.record_test(prev_view.node_id, candidate_idx, 0, None)
            self.remember(candidate, changed=False, game_over=True)
            self.steps_since_hash_change += 1
            self.steps_since_new_node += 1
            return None

        if level_up:
            self.remember(candidate, changed=True, level_up=True)
            self.steps_since_hash_change = 0
            self.steps_since_new_node = 0
            aid = int(candidate.get("action_id", -1))
            if aid not in {7, -1}:
                self.last_changed_action_id = aid
            return None

        next_view = self._build_view(next_obs, env)
        changed = next_view.node_id != prev_view.node_id
        suspicious = next_view.node_id == self.start_node and next_view.num_frames > 1

        self.graph.record_test(
            prev_view.node_id,
            candidate_idx,
            1 if changed else 0,
            target_node=next_view.node_id if changed else None,
            target_num_candidates=len(next_view.candidates),
            group2remaining_candidate_ids=next_view.groups,
            suspicious_transition=suspicious,
        )
        self.remember(candidate, changed=changed, suspicious=suspicious)

        node_count = len(self.graph._nodes)
        if node_count > self.last_unique_nodes:
            self.steps_since_new_node = 0
            self.last_unique_nodes = node_count
        else:
            self.steps_since_new_node += 1

        if changed:
            self.steps_since_hash_change = 0
            aid = int(candidate.get("action_id", -1))
            if aid == 7 or candidate.get("kind") == "undo":
                if self.undo_stack:
                    if next_view.node_id == self.undo_stack[-1]:
                        self.undo_stack.pop()
                    elif next_view.node_id in self.undo_stack:
                        while self.undo_stack and self.undo_stack[-1] != next_view.node_id:
                            self.undo_stack.pop()
                        if self.undo_stack and self.undo_stack[-1] == next_view.node_id:
                            self.undo_stack.pop()
                self.last_changed_action_id = 7
            else:
                if prev_view.node_id != next_view.node_id:
                    if not self.undo_stack or self.undo_stack[-1] != prev_view.node_id:
                        self.undo_stack.append(prev_view.node_id)
                        if len(self.undo_stack) > 96:
                            self.undo_stack = self.undo_stack[-96:]
                self.last_changed_action_id = aid
        else:
            self.steps_since_hash_change += 1
        return next_view

    def note_external_reset(self, count: int = 1) -> None:
        self.resets_in_level += int(count)
        self.steps_in_level += int(count)
        self.steps_since_hash_change += int(count)
        self.steps_since_new_node += int(count)
        self.undo_stack.clear()
        self.last_changed_action_id = None

    def _undo_candidate_idx(self, view: FrameView) -> Optional[int]:
        for i, cand in enumerate(view.candidates):
            if int(cand.get("action_id", -1)) == 7:
                return i
        return None

    def choose_candidate(self, view: FrameView) -> Optional[dict[str, Any]]:
        if not view.candidates:
            return None

        undo_idx = self._undo_candidate_idx(view)
        undo_status = self.graph.edge_status(view.node_id, undo_idx) if undo_idx is not None else -1
        can_try_undo = (
            undo_idx is not None
            and bool(view.undo_available)
            and bool(self.undo_stack)
            and undo_status != -1
        )

        current_dist = self.graph.distance(view.node_id)
        unreachable = (
            view.node_id != self.start_node
            and view.node_id not in self.graph.frontier
            and current_dist >= INFINITY
        )

        level_idx = max(1, int(view.levels_completed) + 1)
        new_node_limit = 16 if level_idx <= 2 else (22 if level_idx <= 4 else 30)
        hash_change_limit = 10 if level_idx <= 2 else (14 if level_idx <= 4 else 20)
        stagnated = self.steps_since_new_node >= new_node_limit or self.steps_since_hash_change >= hash_change_limit

        if unreachable:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to recover reachable frontier from {view.node_id}"
                return out
            if view.node_id != self.start_node and self.resets_in_level < 12:
                return {"kind": "meta_reset", "reasoning": f"reset to recover frontier from {view.node_id}"}
            return None

        if stagnated and view.node_id != self.start_node:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to escape stagnation from {view.node_id}"
                return out
            if self.resets_in_level < 12:
                return {"kind": "meta_reset", "reasoning": f"reset after stagnation at {view.node_id}"}

        if view.node_id == self.start_node and not self.graph.frontier:
            return None

        open_ids = self.graph.open_candidate_ids(view.node_id)
        open_non_undo = [idx for idx in open_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if open_non_undo:
            edge_idx = max(open_non_undo, key=lambda idx: self.candidate_priority(view.candidates[idx], path_mode=False))
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if can_try_undo:
            out = dict(view.candidates[undo_idx])
            out["candidate_idx"] = int(undo_idx)
            out["reasoning"] = f"undo to parent/frontier from {view.node_id}"
            return out

        path_ids = self.graph.successful_candidate_ids(view.node_id)
        path_non_undo = [idx for idx in path_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if path_non_undo:
            edge_idx = max(
                path_non_undo,
                key=lambda idx: self.candidate_priority(
                    view.candidates[idx],
                    path_mode=True,
                    distance=self.graph.edge_distance(view.node_id, idx),
                ),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if view.node_id != self.start_node and self.resets_in_level < 12:
            return {"kind": "meta_reset", "reasoning": f"reset because no reachable frontier from {view.node_id}"}
        return None



try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    TORCH_AVAILABLE = True
except Exception:
    torch = None
    nn = None
    F = None
    optim = None
    TORCH_AVAILABLE = False

SIMPLE_MODEL_ACTION_IDS = [1, 2, 3, 4, 5, 7]
SIMPLE_MODEL_INDEX = {aid: idx for idx, aid in enumerate(SIMPLE_MODEL_ACTION_IDS)}


if TORCH_AVAILABLE:
    class ActionChangeModel(nn.Module):
        def __init__(self, in_channels: int = 17, width: int = 32, n_simple: int = len(SIMPLE_MODEL_ACTION_IDS)) -> None:
            super().__init__()
            w1 = int(width)
            w2 = int(width * 2)
            self.conv1 = nn.Conv2d(in_channels, w1, kernel_size=3, padding=1)
            self.conv2 = nn.Conv2d(w1, w2, kernel_size=3, padding=1)
            self.conv3 = nn.Conv2d(w2, w2, kernel_size=3, padding=1)
            self.simple_pool = nn.AdaptiveAvgPool2d((4, 4))
            self.simple_head = nn.Sequential(
                nn.Flatten(),
                nn.Linear(w2 * 4 * 4, 128),
                nn.ReLU(),
                nn.Linear(128, n_simple),
            )
            self.click_head = nn.Sequential(
                nn.Conv2d(w2, w1, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(w1, 1, kernel_size=1),
            )

        def forward(self, x: "torch.Tensor") -> tuple["torch.Tensor", "torch.Tensor"]:
            x = F.relu(self.conv1(x))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            simple_logits = self.simple_head(self.simple_pool(x))
            click_logits = self.click_head(x).flatten(1)
            return simple_logits, click_logits
else:
    ActionChangeModel = None


class HybridLevelExplorerAgent(LevelExplorerAgent):
    def __init__(self, game_id: str, n_groups: int = 5, global_memory: Optional[FeatureMemory] = None) -> None:
        self.device = None
        if TORCH_AVAILABLE:
            try:
                self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            except Exception:
                self.device = torch.device("cpu")
        self.model = None
        self.optimizer = None
        self.model_width = 40 if TORCH_AVAILABLE and self.device is not None and self.device.type == "cuda" else 24
        self.batch_size = 64 if TORCH_AVAILABLE and self.device is not None and self.device.type == "cuda" else 24
        self.train_frequency = 6 if TORCH_AVAILABLE and self.device is not None and self.device.type == "cuda" else 10
        self.min_train = max(24, self.batch_size)
        self.replay = deque(maxlen=24000)
        self.replay_seen: dict[str, int] = defaultdict(int)
        self.model_cache_node: Optional[str] = None
        self.model_cache_simple: Optional[np.ndarray] = None
        self.model_cache_click: Optional[np.ndarray] = None
        self.game_step_counter = 0
        self.node_candidate_failures: dict[tuple[str, int], int] = defaultdict(int)
        self.start_node_simple_untried: set[int] = set()
        self.start_node_click_untried: list[int] = []
        super().__init__(game_id, n_groups=n_groups, global_memory=global_memory)
        self._ensure_model()

    def _ensure_model(self) -> None:
        if not TORCH_AVAILABLE or ActionChangeModel is None:
            return
        if self.model is None:
            self.model = ActionChangeModel(width=self.model_width).to(self.device)
            self.optimizer = optim.Adam(self.model.parameters(), lr=3e-4)
            self.model.eval()

    def reset_level_state(self) -> None:
        super().reset_level_state()
        self.model_cache_node = None
        self.model_cache_simple = None
        self.model_cache_click = None
        self.node_candidate_failures = defaultdict(int)
        self.start_node_simple_untried = set()
        self.start_node_click_untried = []

    def _pad_frame(self, frame: np.ndarray) -> tuple[np.ndarray, tuple[int, int]]:
        frame = np.asarray(frame, dtype=np.uint8)
        h, w = frame.shape
        h = min(int(h), 64)
        w = min(int(w), 64)
        out = np.zeros((64, 64), dtype=np.uint8)
        out[:h, :w] = frame[:h, :w]
        return out, (h, w)

    def _frame_to_tensor_from_raw(self, frame: np.ndarray) -> Optional["torch.Tensor"]:
        if not TORCH_AVAILABLE or self.model is None:
            return None
        padded, (h, w) = self._pad_frame(frame)
        x = torch.zeros((17, 64, 64), dtype=torch.float32, device=self.device)
        frame_t = torch.from_numpy(padded.astype(np.int64)).to(self.device)
        x.scatter_(0, frame_t.unsqueeze(0), 1.0)
        x[16, :h, :w] = 1.0
        return x

    def _frame_to_tensor_from_exp(self, padded: np.ndarray, shape_hw: tuple[int, int]) -> Optional["torch.Tensor"]:
        if not TORCH_AVAILABLE or self.model is None:
            return None
        h, w = int(shape_hw[0]), int(shape_hw[1])
        h = max(0, min(64, h))
        w = max(0, min(64, w))
        x = torch.zeros((17, 64, 64), dtype=torch.float32, device=self.device)
        frame_t = torch.from_numpy(np.asarray(padded, dtype=np.int64)).to(self.device)
        x.scatter_(0, frame_t.unsqueeze(0), 1.0)
        if h > 0 and w > 0:
            x[16, :h, :w] = 1.0
        return x

    def _get_model_predictions(self, view: FrameView) -> tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        if not TORCH_AVAILABLE or self.model is None:
            return None, None
        if self.model_cache_node == view.node_id and self.model_cache_simple is not None and self.model_cache_click is not None:
            return self.model_cache_simple, self.model_cache_click
        try:
            x = self._frame_to_tensor_from_raw(view.raw_frame)
            if x is None:
                return None, None
            self.model.eval()
            with torch.no_grad():
                simple_logits, click_logits = self.model(x.unsqueeze(0))
                simple_probs = torch.sigmoid(simple_logits[0]).detach().float().cpu().numpy()
                click_probs = torch.sigmoid(click_logits[0]).detach().float().cpu().numpy().reshape(64, 64)
            self.model_cache_node = view.node_id
            self.model_cache_simple = simple_probs
            self.model_cache_click = click_probs
            return simple_probs, click_probs
        except Exception:
            self.model_cache_node = None
            self.model_cache_simple = None
            self.model_cache_click = None
            return None, None

    def _candidate_model_score(self, view: FrameView, candidate: dict[str, Any]) -> float:
        simple_probs, click_probs = self._get_model_predictions(view)
        if simple_probs is None or click_probs is None:
            return 0.5
        aid = int(candidate.get("action_id", -1))
        if aid in SIMPLE_MODEL_INDEX:
            return float(simple_probs[SIMPLE_MODEL_INDEX[aid]])
        if aid == 6:
            data = candidate.get("data") or {}
            x = int(data.get("x", 0))
            y = int(data.get("y", 0))
            if 0 <= x < 64 and 0 <= y < 64:
                return float(click_probs[y, x])
        return 0.5

    def _candidate_to_exp_target(self, prev_view: FrameView, candidate: dict[str, Any], next_view: Optional[FrameView], *, game_over: bool, level_up: bool) -> float:
        if level_up:
            return 1.0
        if game_over:
            return 0.0
        if next_view is None:
            return 0.0
        changed = next_view.node_id != prev_view.node_id
        suspicious = next_view.node_id == self.start_node and next_view.num_frames > 1
        if changed and not suspicious:
            return 0.80
        if changed and suspicious:
            return 0.35
        return 0.0

    def _candidate_to_exp_index(self, candidate: dict[str, Any]) -> Optional[tuple[str, int]]:
        aid = int(candidate.get("action_id", -1))
        if aid in SIMPLE_MODEL_INDEX:
            return ("simple", int(SIMPLE_MODEL_INDEX[aid]))
        if aid == 6:
            data = candidate.get("data") or {}
            x = int(data.get("x", 0))
            y = int(data.get("y", 0))
            if 0 <= x < 64 and 0 <= y < 64:
                return ("click", int(y * 64 + x))
        return None

    def _record_experience(self, prev_view: FrameView, candidate: dict[str, Any], next_view: Optional[FrameView], *, game_over: bool, level_up: bool) -> None:
        if not TORCH_AVAILABLE or self.model is None:
            return
        idx_info = self._candidate_to_exp_index(candidate)
        if idx_info is None:
            return
        kind, idx = idx_info
        target = float(self._candidate_to_exp_target(prev_view, candidate, next_view, game_over=game_over, level_up=level_up))
        padded, shape_hw = self._pad_frame(prev_view.raw_frame)
        key_payload = padded.tobytes() + str(shape_hw).encode("utf-8") + f"|{kind}|{idx}".encode("utf-8")
        key = hashlib.blake2b(key_payload, digest_size=16).hexdigest()
        cap = 2 if target <= 0.0 else 3
        if self.replay_seen.get(key, 0) >= cap:
            return
        self.replay.append(
            {
                "frame": padded.copy(),
                "shape_hw": tuple(shape_hw),
                "kind": kind,
                "idx": int(idx),
                "target": float(target),
            }
        )
        self.replay_seen[key] += 1

    def _train_model(self, train_steps: int = 1) -> None:
        if not TORCH_AVAILABLE or self.model is None or self.optimizer is None:
            return
        if len(self.replay) < self.min_train:
            return
        try:
            self.model.train()
            replay_size = len(self.replay)
            batch_n = min(self.batch_size, replay_size)
            for _ in range(max(1, int(train_steps))):
                batch_indices = self.np_rng.choice(replay_size, size=batch_n, replace=False)
                batch = [self.replay[int(i)] for i in batch_indices]
                xs = []
                for exp in batch:
                    x = self._frame_to_tensor_from_exp(exp["frame"], exp["shape_hw"])
                    if x is not None:
                        xs.append(x)
                if len(xs) != len(batch) or not xs:
                    continue
                x_batch = torch.stack(xs, dim=0)
                y_batch = torch.tensor([float(exp["target"]) for exp in batch], dtype=torch.float32, device=self.device)
                simple_logits, click_logits = self.model(x_batch)

                selected_logits = []
                for i, exp in enumerate(batch):
                    if exp["kind"] == "simple":
                        selected_logits.append(simple_logits[i, int(exp["idx"])])
                    else:
                        selected_logits.append(click_logits[i, int(exp["idx"])])
                selected_logits = torch.stack(selected_logits, dim=0)

                loss = F.binary_cross_entropy_with_logits(selected_logits, y_batch)
                # Mild exploration regularization to avoid total collapse early in the game.
                loss = loss - 2e-4 * torch.sigmoid(simple_logits).mean() - 2e-5 * torch.sigmoid(click_logits).mean()

                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
            self.model.eval()
            self.model_cache_node = None
            self.model_cache_simple = None
            self.model_cache_click = None
        except Exception:
            try:
                self.model.eval()
            except Exception:
                pass

    def start_level(self, obs: Any, env: Any) -> FrameView:
        view = super().start_level(obs, env)
        self.start_node_simple_untried = {
            idx for idx, cand in enumerate(view.candidates)
            if cand.get("kind") == "simple" and int(cand.get("action_id", -1)) != 7
        }
        self.start_node_click_untried = [
            idx for idx, cand in enumerate(view.candidates)
            if cand.get("kind") == "click" and int(cand.get("group", self.n_groups - 1)) <= 1
        ][:8]
        return view

    def note_external_reset(self, count: int = 1) -> None:
        super().note_external_reset(count)
        self.model_cache_node = None
        self.model_cache_simple = None
        self.model_cache_click = None

    def _node_candidate_key(self, view: FrameView, candidate_idx: int) -> tuple[str, int]:
        return (str(view.node_id), int(candidate_idx))

    def hybrid_score(self, view: FrameView, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        base = super().candidate_priority(candidate, path_mode=path_mode, distance=distance)
        base_prob = 1.0 / (1.0 + math.exp(-1.45 * float(base)))
        model_prob = self._candidate_model_score(view, candidate)
        local_prior = self.level_memory.estimate(candidate)
        global_prior = self.global_memory.estimate(candidate)
        prior = 0.72 * local_prior + 0.28 * global_prior

        node_failures = self.node_candidate_failures.get(self._node_candidate_key(view, int(candidate.get("candidate_idx", -1))), 0)
        fail_penalty = 0.06 * min(4, int(node_failures))

        kind = str(candidate.get("kind", ""))
        aid = int(candidate.get("action_id", -1))
        warmup_bonus = 0.0
        if not path_mode and view.node_id == self.start_node:
            if kind == "simple" and aid != 7 and self.steps_in_level <= max(4, len(view.available_action_ids)):
                warmup_bonus += 0.08
            elif kind == "click" and self.steps_in_level <= 8 and int(candidate.get("group", self.n_groups - 1)) <= 1:
                warmup_bonus += 0.04

        if path_mode:
            score = 0.54 * base_prob + 0.14 * model_prob + 0.32 * prior - 0.018 * float(distance) - fail_penalty
        else:
            score = 0.37 * base_prob + 0.38 * model_prob + 0.25 * prior + warmup_bonus - fail_penalty

        score += 1e-6 * self.py_rng.random()
        return float(score)

    def observe_transition(
        self,
        prev_view: FrameView,
        candidate_idx: int,
        next_obs: Any,
        env: Any,
        game_over: bool = False,
        level_up: bool = False,
    ) -> Optional[FrameView]:
        candidate = prev_view.candidates[candidate_idx]
        next_view = super().observe_transition(
            prev_view,
            candidate_idx,
            next_obs,
            env,
            game_over=game_over,
            level_up=level_up,
        )
        try:
            self._record_experience(prev_view, candidate, next_view, game_over=game_over, level_up=level_up)
        except Exception:
            pass

        key = self._node_candidate_key(prev_view, candidate_idx)
        changed = bool(level_up) or (next_view is not None and next_view.node_id != prev_view.node_id)
        if changed:
            self.node_candidate_failures[key] = max(0, int(self.node_candidate_failures.get(key, 0)) - 1)
        else:
            self.node_candidate_failures[key] += 1

        self.game_step_counter += 1
        if TORCH_AVAILABLE and self.model is not None and self.game_step_counter % self.train_frequency == 0:
            extra = 2 if self.device is not None and self.device.type == "cuda" and len(self.replay) >= 4 * self.batch_size else 1
            self._train_model(train_steps=extra)
        return next_view

    def choose_candidate(self, view: FrameView) -> Optional[dict[str, Any]]:
        if not view.candidates:
            return None

        # Cache predictions once for the current view so repeated scoring is cheap.
        self._get_model_predictions(view)

        undo_idx = self._undo_candidate_idx(view)
        undo_status = self.graph.edge_status(view.node_id, undo_idx) if undo_idx is not None else -1
        can_try_undo = (
            undo_idx is not None
            and bool(view.undo_available)
            and bool(self.undo_stack)
            and undo_status != -1
        )

        current_dist = self.graph.distance(view.node_id)
        unreachable = (
            view.node_id != self.start_node
            and view.node_id not in self.graph.frontier
            and current_dist >= INFINITY
        )

        level_idx = max(1, int(view.levels_completed) + 1)
        max_resets = 8 if level_idx <= 3 else 10
        new_node_limit = 14 if level_idx <= 2 else (20 if level_idx <= 4 else 28)
        hash_change_limit = 9 if level_idx <= 2 else (13 if level_idx <= 4 else 18)
        stagnated = self.steps_since_new_node >= new_node_limit or self.steps_since_hash_change >= hash_change_limit

        if unreachable:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to recover reachable frontier from {view.node_id}"
                return out
            if view.node_id != self.start_node and self.resets_in_level < max_resets:
                return {"kind": "meta_reset", "reasoning": f"reset to recover frontier from {view.node_id}"}
            return None

        if stagnated and view.node_id != self.start_node:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to escape stagnation from {view.node_id}"
                return out
            if self.resets_in_level < max_resets:
                return {"kind": "meta_reset", "reasoning": f"reset after stagnation at {view.node_id}"}

        if view.node_id == self.start_node and not self.graph.frontier:
            return None

        open_ids = self.graph.open_candidate_ids(view.node_id)
        open_non_undo = [idx for idx in open_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        # Warm start on the first node: probe simple actions before aggressive clicks.
        if view.node_id == self.start_node and open_non_undo:
            simple_candidates = [idx for idx in open_non_undo if idx in self.start_node_simple_untried]
            if simple_candidates and self.steps_in_level <= max(4, len(view.available_action_ids)):
                edge_idx = max(
                    simple_candidates,
                    key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                )
                self.start_node_simple_untried.discard(edge_idx)
                cand = view.candidates[edge_idx]
                out = dict(cand)
                out["candidate_idx"] = int(edge_idx)
                return out
            if self.start_node_click_untried and self.steps_in_level <= 8:
                click_candidates = [idx for idx in self.start_node_click_untried if idx in open_non_undo]
                if click_candidates:
                    edge_idx = max(
                        click_candidates,
                        key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                    )
                    self.start_node_click_untried = [idx for idx in self.start_node_click_untried if idx != edge_idx]
                    cand = view.candidates[edge_idx]
                    out = dict(cand)
                    out["candidate_idx"] = int(edge_idx)
                    return out

        if open_non_undo:
            edge_idx = max(
                open_non_undo,
                key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if can_try_undo:
            out = dict(view.candidates[undo_idx])
            out["candidate_idx"] = int(undo_idx)
            out["reasoning"] = f"undo to parent/frontier from {view.node_id}"
            return out

        path_ids = self.graph.successful_candidate_ids(view.node_id)
        path_non_undo = [idx for idx in path_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if path_non_undo:
            edge_idx = max(
                path_non_undo,
                key=lambda idx: self.hybrid_score(
                    view,
                    {**view.candidates[idx], "candidate_idx": int(idx)},
                    path_mode=True,
                    distance=self.graph.edge_distance(view.node_id, idx),
                ),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if view.node_id != self.start_node and self.resets_in_level < max_resets:
            return {"kind": "meta_reset", "reasoning": f"reset because no reachable frontier from {view.node_id}"}
        return None

def level_budget(level_idx: int) -> int:
    table = {
        1: 80,
        2: 120,
        3: 180,
        4: 260,
        5: 360,
        6: 480,
        7: 640,
    }
    return table.get(level_idx, 760)


def game_budget(level_idx: int) -> int:
    return 1100 + 200 * max(0, level_idx - 1)


def make_env(arc: Any, game_id: str) -> Any:
    sig = inspect.signature(arc.make)
    kwargs: dict[str, Any] = {}
    if "save_recording" in sig.parameters:
        kwargs["save_recording"] = False
    if "include_frame_data" in sig.parameters:
        kwargs["include_frame_data"] = True
    if "render_mode" in sig.parameters:
        kwargs["render_mode"] = None
    return arc.make(game_id, **kwargs)




# ==================== Local source planner ====================

try:
    ActionInput = find_symbol("arcengine", "ActionInput")
except Exception:
    try:
        ActionInput = find_symbol("arc_agi", "ActionInput")
    except Exception:
        ActionInput = None


class _FallbackActionInput:
    def __init__(self, id: Any = None, action: Any = None, data: Optional[dict[str, Any]] = None, **kwargs: Any) -> None:
        self.id = id if id is not None else action
        self.action = action if action is not None else id
        self.data = data
        for k, v in kwargs.items():
            setattr(self, k, v)


PLANNER_LOG = logging.getLogger("arc_source_planner")
if not PLANNER_LOG.handlers:
    logging.basicConfig(level=logging.WARNING)
PLANNER_LOG.setLevel(logging.INFO if os.getenv("ARC_DEBUG", "0") == "1" else logging.WARNING)

ENV_SOURCE_CACHE: dict[str, tuple[Optional[str], Optional[str]]] = {}
GAME_CLASS_CACHE: dict[tuple[str, str], Any] = {}
ENV_FILE_INDEX: dict[str, list[Path]] = defaultdict(list)
if ENVIRONMENTS_DIR.exists():
    for _py_path in ENVIRONMENTS_DIR.rglob("*.py"):
        if _py_path.name.startswith("_"):
            continue
        ENV_FILE_INDEX[_py_path.stem.lower()].append(_py_path)

NOISY_FIELD_TOKENS = {
    "logger", "logging", "recorder", "record", "recording",
    "screen", "surface", "image", "images", "sprite", "sprites",
    "render", "renderer", "display", "window", "sound", "audio",
    "clock", "timer", "rng", "random", "seed", "api", "client",
    "socket", "session", "history", "trace", "debug", "cache",
    "memo", "frames", "frame", "pixels", "grid_cache", "tile_cache",
    "guid", "uuid", "id_map", "path_cache", "log", "profile",
}
NOISY_FIELD_EXACT = {
    "_action_count", "action_count", "_last_action", "last_action",
    "_action_complete", "_full_reset", "_elapsed", "_elapsed_steps",
    "_frame_index", "_tick", "tick", "_guid",
}
SEMANTIC_PATH_TOKENS = {
    "goal": 8,
    "target": 8,
    "exit": 8,
    "finish": 8,
    "flag": 7,
    "home": 7,
    "portal": 7,
    "key": 6,
    "door": 6,
    "switch": 6,
    "button": 6,
    "cursor": 6,
    "player": 6,
    "agent": 6,
    "avatar": 6,
    "hero": 6,
    "enemy": 5,
    "monster": 5,
    "object": 4,
    "box": 4,
    "gem": 4,
    "coin": 4,
    "selected": 5,
    "holding": 5,
    "held": 5,
    "mode": 5,
    "phase": 5,
    "state": 4,
    "index": 2,
    "count": 2,
    "pos": 3,
    "coord": 3,
    "x": 1,
    "y": 1,
    "row": 1,
    "col": 1,
    "cx": 1,
    "cy": 1,
}


def planner_time_budget_seconds(level_idx: int, has_transfer: bool = False) -> float:
    table = {
        0: 12.0,
        1: 18.0,
        2: 28.0,
        3: 42.0,
        4: 58.0,
        5: 78.0,
        6: 98.0,
    }
    budget = table.get(int(level_idx), 110.0)
    if has_transfer:
        budget *= 1.15
    return float(budget)


def _path_text(path: tuple[Any, ...]) -> str:
    return ".".join(str(p).lower() for p in path)


def _semantic_path_score(path: tuple[Any, ...]) -> int:
    text = _path_text(path)
    score = 0
    for token, bonus in SEMANTIC_PATH_TOKENS.items():
        if token in text:
            score += int(bonus)
    return int(score)


def _is_noisy_field(name: Any) -> bool:
    s = str(name).strip().lower()
    if s in NOISY_FIELD_EXACT:
        return True
    if s.startswith("__") and s.endswith("__"):
        return True
    for tok in NOISY_FIELD_TOKENS:
        if tok in s:
            return True
    return False


def _normalize_number(v: Any) -> Any:
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, bool):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        if math.isnan(float(v)) or math.isinf(float(v)):
            return None
        return round(float(v), 4)
    return v


def _pack_small_value(v: Any) -> Any:
    if v is None:
        return None
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, (bool, int, np.integer, float, np.floating)):
        return _normalize_number(v)
    if isinstance(v, str):
        if len(v) <= 48:
            return v
        return v[:48]
    if isinstance(v, bytes):
        if len(v) <= 16:
            return tuple(int(x) for x in v)
        return None
    if isinstance(v, np.ndarray):
        if v.size <= 16:
            arr = np.asarray(v)
            flat = tuple(_normalize_number(x) for x in arr.flatten().tolist())
            return ("nd", tuple(int(x) for x in arr.shape), flat)
        return None
    if isinstance(v, (list, tuple)):
        if len(v) <= 6:
            packed = []
            for item in v:
                pv = _pack_small_value(item)
                if pv is None:
                    return None
                packed.append(pv)
            return tuple(packed)
        return None
    if isinstance(v, dict):
        if len(v) <= 8:
            packed_items = []
            for key in sorted(v.keys(), key=lambda z: str(z))[:8]:
                pv = _pack_small_value(v[key])
                if pv is None:
                    continue
                packed_items.append((str(key), pv))
            if packed_items:
                return tuple(packed_items)
        return None
    if hasattr(v, "name") and isinstance(getattr(v, "name"), str):
        return str(v.name)
    return None


def _flatten_small_state(obj: Any, depth: int = 2, max_items: int = 160) -> dict[tuple[Any, ...], Any]:
    out: dict[tuple[Any, ...], Any] = {}
    seen: set[int] = set()

    def rec(x: Any, path: tuple[Any, ...], d: int) -> None:
        if d < 0 or x is None or len(out) >= max_items:
            return
        try:
            oid = id(x)
            if oid in seen:
                return
            seen.add(oid)
        except Exception:
            pass

        packed = _pack_small_value(x)
        if packed is not None:
            out[path] = packed
            return

        if isinstance(x, dict):
            items = list(x.items())[:12]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), d - 1)
            return

        if isinstance(x, (list, tuple)):
            if len(x) <= 8:
                for i, v in enumerate(x[:8]):
                    rec(v, path + (int(i),), d - 1)
            return

        attrs = getattr(x, "__dict__", None)
        if isinstance(attrs, dict):
            items = sorted(attrs.items(), key=lambda kv: str(kv[0]))[:24]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), d - 1)

    rec(obj, tuple(), depth)
    return out


def _get_value_by_path(obj: Any, path: tuple[Any, ...]) -> Any:
    x = obj
    for key in path:
        if x is None:
            return None
        try:
            if isinstance(key, int):
                if isinstance(x, (list, tuple)) and 0 <= key < len(x):
                    x = x[key]
                else:
                    return None
            else:
                if isinstance(x, dict):
                    if key in x:
                        x = x[key]
                    elif str(key) in x:
                        x = x[str(key)]
                    else:
                        return None
                else:
                    if not hasattr(x, key):
                        return None
                    x = getattr(x, key)
        except Exception:
            return None
    return _pack_small_value(x)


def _hidden_coord_points_from_obj(obj: Any, shape_hw: tuple[int, int], max_points: int = 24) -> list[tuple[float, int, int, str]]:
    h, w = int(shape_hw[0]), int(shape_hw[1])
    out: list[tuple[float, int, int, str]] = []
    seen_pts: set[tuple[int, int]] = set()
    seen_objs: set[int] = set()

    def add_pt(x: Any, y: Any, score: float, source: str) -> None:
        try:
            xi = int(round(float(_normalize_number(x))))
            yi = int(round(float(_normalize_number(y))))
        except Exception:
            return
        if not (0 <= xi < w and 0 <= yi < h):
            return
        key = (xi, yi)
        if key in seen_pts:
            return
        seen_pts.add(key)
        out.append((float(score), xi, yi, source))

    def rec(x: Any, path: tuple[Any, ...], depth: int) -> None:
        if depth < 0 or x is None or len(out) >= max_points:
            return
        try:
            oid = id(x)
            if oid in seen_objs:
                return
            seen_objs.add(oid)
        except Exception:
            pass

        path_score = _semantic_path_score(path)
        text = _path_text(path)

        if isinstance(x, dict):
            lower_map = {str(k).lower(): k for k in x.keys()}
            if "x" in lower_map and "y" in lower_map:
                add_pt(x[lower_map["x"]], x[lower_map["y"]], 80 + path_score, f"hidden:{text or 'xy'}")
            if "col" in lower_map and "row" in lower_map:
                add_pt(x[lower_map["col"]], x[lower_map["row"]], 78 + path_score, f"hidden:{text or 'rc'}")
            if "cx" in lower_map and "cy" in lower_map:
                add_pt(x[lower_map["cx"]], x[lower_map["cy"]], 76 + path_score, f"hidden:{text or 'cxy'}")
            items = list(x.items())[:12]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), depth - 1)
            return

        if isinstance(x, (list, tuple)):
            if len(x) >= 2 and all(isinstance(v, (int, float, np.integer, np.floating, bool)) for v in x[:2]):
                a, b = x[0], x[1]
                add_pt(a, b, 64 + path_score, f"hidden:{text or 'pair'}")
                add_pt(b, a, 60 + path_score, f"hidden:{text or 'pair_rev'}")
            if len(x) <= 8:
                for i, v in enumerate(x[:8]):
                    rec(v, path + (int(i),), depth - 1)
            return

        attrs = getattr(x, "__dict__", None)
        if isinstance(attrs, dict):
            lower_map = {str(k).lower(): k for k in attrs.keys()}
            if "x" in lower_map and "y" in lower_map:
                add_pt(getattr(x, lower_map["x"]), getattr(x, lower_map["y"]), 80 + path_score, f"hidden:{text or 'xy'}")
            if "col" in lower_map and "row" in lower_map:
                add_pt(getattr(x, lower_map["col"]), getattr(x, lower_map["row"]), 78 + path_score, f"hidden:{text or 'rc'}")
            if "cx" in lower_map and "cy" in lower_map:
                add_pt(getattr(x, lower_map["cx"]), getattr(x, lower_map["cy"]), 76 + path_score, f"hidden:{text or 'cxy'}")
            if "position" in lower_map:
                rec(getattr(x, lower_map["position"]), path + ("position",), depth - 1)
            items = sorted(attrs.items(), key=lambda kv: str(kv[0]))[:24]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), depth - 1)

    rec(obj, tuple(), 3)
    out.sort(key=lambda t: (-t[0], t[2], t[1]))
    return out[:max_points]


def _visible_objects(frame: np.ndarray) -> list[dict[str, Any]]:
    frame = np.asarray(frame, dtype=np.uint8)
    bg = int(np.bincount(frame.reshape(-1), minlength=16).argmax())
    objs = []
    for color in range(16):
        if color == bg:
            continue
        mask = frame == color
        count = int(mask.sum())
        if count < 2:
            continue
        ys, xs = np.where(mask)
        objs.append(
            {
                "color": int(color),
                "cx": float(xs.mean()),
                "cy": float(ys.mean()),
                "n": int(count),
            }
        )
    objs.sort(key=lambda o: (o["color"], -o["n"], o["cx"], o["cy"]))
    return objs


def _estimate_click_offset(prev_frame: np.ndarray, curr_frame: np.ndarray) -> tuple[float, float]:
    objs_prev = _visible_objects(prev_frame)
    objs_curr = _visible_objects(curr_frame)
    if not objs_prev or not objs_curr:
        return 0.0, 0.0
    matched = []
    for op in objs_prev:
        best = None
        best_dist = float("inf")
        for oc in objs_curr:
            if oc["color"] != op["color"]:
                continue
            if abs(oc["n"] - op["n"]) > max(op["n"], oc["n"]) * 0.55:
                continue
            d = abs(oc["cx"] - op["cx"]) + abs(oc["cy"] - op["cy"])
            if d < best_dist:
                best_dist = d
                best = oc
        if best is not None:
            matched.append((op, best))
    if not matched:
        return 0.0, 0.0
    dx = float(np.mean([m[1]["cx"] - m[0]["cx"] for m in matched]))
    dy = float(np.mean([m[1]["cy"] - m[0]["cy"] for m in matched]))
    if abs(dx) > curr_frame.shape[1] or abs(dy) > curr_frame.shape[0]:
        return 0.0, 0.0
    return dx, dy


def _find_game_source_and_class(game_id: str, env: Any = None) -> tuple[Optional[str], Optional[str]]:
    if game_id in ENV_SOURCE_CACHE:
        return ENV_SOURCE_CACHE[game_id]

    base = str(game_id).split("-")[0]
    suffix = str(game_id).split("-", 1)[1] if "-" in str(game_id) else ""
    candidates: list[tuple[int, Path]] = []

    def add_candidate(path_like: Any, bonus: int = 0) -> None:
        if path_like is None:
            return
        path = Path(path_like)
        if not path.exists():
            return
        score = int(bonus)
        path_text = str(path).lower()
        if path.stem.lower() == base.lower():
            score += 30
        if suffix and suffix.lower() in path_text:
            score += 18
        if base.lower() in path_text:
            score += 8
        score += max(0, 6 - len(path.parts))
        candidates.append((score, path))

    local_dir = None
    try:
        env_info = maybe_get(env, "environment_info")
        local_dir = maybe_get(env_info, "local_dir")
    except Exception:
        local_dir = None
    if local_dir:
        local_dir = Path(local_dir)
        add_candidate(local_dir / f"{base}.py", bonus=40)
        for py in local_dir.glob("*.py"):
            add_candidate(py, bonus=25)

    exact_candidates = [
        ENVIRONMENTS_DIR / base / suffix / f"{base}.py",
        ENVIRONMENTS_DIR / base / f"{base}.py",
        ENVIRONMENTS_DIR / f"{base}.py",
    ]
    for cand in exact_candidates:
        add_candidate(cand, bonus=20)

    for py in ENV_FILE_INDEX.get(base.lower(), []):
        add_candidate(py, bonus=10)

    if not candidates:
        ENV_SOURCE_CACHE[game_id] = (None, None)
        return (None, None)

    candidates.sort(key=lambda t: (-t[0], len(str(t[1]))))
    chosen = candidates[0][1]
    class_name = None
    try:
        text = chosen.read_text(encoding="utf-8", errors="ignore")[:20000]
        m = re.search(r"class\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(\s*ARCBaseGame", text)
        if m:
            class_name = m.group(1)
        else:
            class_candidates = re.findall(r"class\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(", text)
            if class_candidates:
                for cand in class_candidates:
                    if cand.lower().startswith(base.lower()):
                        class_name = cand
                        break
                if class_name is None:
                    class_name = class_candidates[0]
    except Exception:
        class_name = None

    if class_name is None:
        class_name = base[:1].upper() + base[1:]

    out = (str(chosen), str(class_name))
    ENV_SOURCE_CACHE[game_id] = out
    return out


def _load_game_class(source_path: str, class_name: str) -> Any:
    key = (str(source_path), str(class_name))
    if key in GAME_CLASS_CACHE:
        return GAME_CLASS_CACHE[key]

    module_name = f"_arc_local_{hashlib.md5(str(source_path).encode()).hexdigest()[:12]}"
    parent = str(Path(source_path).parent)
    grandparent = str(Path(source_path).parent.parent)
    if parent not in sys.path:
        sys.path.insert(0, parent)
    if grandparent not in sys.path:
        sys.path.insert(0, grandparent)
    if str(ENVIRONMENTS_DIR) not in sys.path:
        sys.path.insert(0, str(ENVIRONMENTS_DIR))

    spec = importlib.util.spec_from_file_location(module_name, source_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {source_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)

    if hasattr(module, class_name):
        cls = getattr(module, class_name)
    else:
        cls = None
        for name, obj in vars(module).items():
            if inspect.isclass(obj) and name.lower().startswith(Path(source_path).stem.lower()):
                cls = obj
                break
        if cls is None:
            for name, obj in vars(module).items():
                if inspect.isclass(obj) and hasattr(obj, "perform_action"):
                    cls = obj
                    break
    if cls is None:
        raise AttributeError(f"Could not resolve game class '{class_name}' from {source_path}")

    GAME_CLASS_CACHE[key] = cls
    return cls


def _game_action_token(action_id: int) -> Any:
    try:
        if hasattr(GameAction, "from_id"):
            return GameAction.from_id(int(action_id))
    except Exception:
        pass
    if int(action_id) == 0:
        return RESET_ACTION
    return ACTION_ID_TO_ENUM.get(int(action_id), int(action_id))


def _make_local_action_input(action_id: int, data: Optional[dict[str, Any]] = None) -> Any:
    payload = None
    if data is not None:
        payload = {}
        for k, v in data.items():
            try:
                payload[str(k)] = int(v) if isinstance(v, (int, np.integer, bool)) else float(v) if isinstance(v, (float, np.floating)) else v
            except Exception:
                payload[str(k)] = v
    token = _game_action_token(action_id)
    cls = ActionInput or _FallbackActionInput

    attempts: list[tuple[tuple[Any, ...], dict[str, Any]]] = []
    if payload and "x" in payload and "y" in payload:
        attempts.extend(
            [
                (tuple(), {"id": token, "x": payload["x"], "y": payload["y"], "data": payload}),
                (tuple(), {"action": token, "x": payload["x"], "y": payload["y"], "data": payload}),
            ]
        )
    attempts.extend(
        [
            (tuple(), {"id": token, "data": payload}),
            (tuple(), {"action": token, "data": payload}),
            ((token, payload), {}),
            ((token,), {"data": payload}),
            ((token,), {}),
        ]
    )

    for args, kwargs in attempts:
        try:
            return cls(*args, **{k: v for k, v in kwargs.items() if v is not None})
        except Exception:
            continue
    return _FallbackActionInput(id=token, data=payload)


def _local_step(game: Any, action_id: int, data: Optional[dict[str, Any]] = None) -> Any:
    fn = getattr(game, "perform_action", None)
    if fn is None:
        fn = getattr(game, "step", None)
    if fn is None:
        raise AttributeError("Local game object has neither perform_action nor step.")

    token = _game_action_token(action_id)
    ai = _make_local_action_input(action_id, data=data)

    patterns: list[tuple[tuple[Any, ...], dict[str, Any]]] = []
    patterns.extend(
        [
            ((ai,), {"raw": True}),
            ((ai,), {}),
        ]
    )
    if data is not None:
        patterns.extend(
            [
                ((token,), {"data": data, "raw": True}),
                ((token,), {"data": data}),
                ((int(action_id),), {"data": data, "raw": True}),
                ((int(action_id),), {"data": data}),
                ((token, data), {}),
                ((int(action_id), data), {}),
            ]
        )
    patterns.extend(
        [
            ((token,), {"raw": True}),
            ((token,), {}),
            ((int(action_id),), {"raw": True}),
            ((int(action_id),), {}),
        ]
    )

    last_exc = None
    for args, kwargs in patterns:
        try:
            return fn(*args, **{k: v for k, v in kwargs.items() if v is not None})
        except TypeError as exc:
            last_exc = exc
            continue
        except Exception as exc:
            last_exc = exc
            continue
    if last_exc is None:
        raise RuntimeError(f"Could not apply local action {action_id}")
    raise last_exc


def _local_frame_from_result(result: Any, game: Any) -> Optional[np.ndarray]:
    frame_obj = maybe_get(result, "frame", "frames", "grid", "grids")
    if frame_obj is None:
        frame_obj = deep_find(result, ("frame", "frames", "grid", "grids"), 3)
    if frame_obj is not None:
        try:
            arr = np.asarray(frame_obj, dtype=np.uint8)
            if arr.ndim == 3:
                return arr[-1].copy()
            if arr.ndim == 2:
                return arr.copy()
        except Exception:
            pass
    if hasattr(game, "get_pixels"):
        try:
            arr = np.asarray(game.get_pixels(0, 0, 64, 64), dtype=np.uint8)
            if arr.ndim == 3:
                return arr[-1].copy()
            if arr.ndim == 2:
                return arr.copy()
        except Exception:
            return None
    return None


def _local_state_name(result: Any, game: Any) -> str:
    st = maybe_get(result, "state", "game_state")
    if st is None:
        st = deep_find(result, ("state", "game_state"), 2)
    if st is None:
        st = maybe_get(game, "_game_state", "game_state", "state")
    if st is None:
        return ""
    if isinstance(st, str):
        return st.split(".")[-1].upper()
    if hasattr(st, "name"):
        return str(st.name).upper()
    return str(st).split(".")[-1].upper()


def _local_levels_completed(result: Any, game: Any) -> int:
    val = maybe_get(result, "levels_completed", "score")
    if val is None:
        val = deep_find(result, ("levels_completed", "score"), 2)
    if val is None:
        val = maybe_get(game, "_score", "score")
    if val is None:
        return 0
    try:
        return int(val)
    except Exception:
        try:
            return int(float(val))
        except Exception:
            return 0


def _local_current_level_index(game: Any, result: Any = None) -> Optional[int]:
    val = maybe_get(game, "_current_level_index", "current_level_index", "level_index")
    if val is None and result is not None:
        val = maybe_get(result, "current_level_index", "level_index")
    if val is None:
        return None
    try:
        return int(val)
    except Exception:
        return None


def _local_available_ids(result: Any, game: Any) -> set[int]:
    raw = maybe_get(result, "available_actions", "actions")
    if raw is None:
        raw = deep_find(result, ("available_actions", "actions"), 2)
    if raw is None:
        raw = maybe_get(game, "_available_actions", "available_actions")
    if raw is None:
        return {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}
    if isinstance(raw, dict):
        raw = list(raw.keys())
    ids: set[int] = set()
    try:
        iterable = list(raw)
    except Exception:
        iterable = []
    for item in iterable:
        aid = normalize_action_id(item)
        if aid is not None and aid != 0:
            ids.add(int(aid))
    if not ids:
        ids = {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}
    return ids


def _clone_game(game: Any) -> Any:
    try:
        return copy.deepcopy(game)
    except Exception:
        return copy.deepcopy(game)


class SourcePlanner:
    def __init__(self, game_id: str, source_path: str, class_name: str) -> None:
        self.game_id = str(game_id)
        self.source_path = str(source_path)
        self.class_name = str(class_name)
        self.game_cls = None
        self.loaded = False
        self.fp = FrameProcessor()
        self.solution_cache: dict[int, list[tuple[int, Optional[dict[str, int]]]]] = {}
        self.root_frames: dict[int, np.ndarray] = {}
        self.tracked_paths_by_level: dict[int, list[tuple[Any, ...]]] = {}
        self.transfer_click_points: dict[int, list[tuple[int, int]]] = {}
        self._proposal_cache: dict[tuple[int, str], list[dict[str, Any]]] = {}

    def load(self) -> bool:
        try:
            self.game_cls = _load_game_class(self.source_path, self.class_name)
            self.loaded = self.game_cls is not None
        except Exception as exc:
            self.loaded = False
            PLANNER_LOG.warning("Planner failed to load %s (%s): %s", self.game_id, self.source_path, exc)
        return bool(self.loaded)

    def _new_game(self, level_idx: int) -> tuple[Any, Any, Optional[np.ndarray]]:
        if self.game_cls is None:
            raise RuntimeError("Planner is not loaded.")
        game = self.game_cls()
        if hasattr(game, "set_level"):
            try:
                game.set_level(int(level_idx))
            except Exception:
                pass
        result = None
        frame = None
        for _ in range(3):
            try:
                result = _local_step(game, 0, data=None)
            except Exception:
                result = None
            frame = _local_frame_from_result(result, game)
            if frame is not None:
                break
        return game, result, frame

    def _state_hash(self, game: Any, frame: np.ndarray, tracked_paths: list[tuple[Any, ...]]) -> str:
        frame_hash = self.fp.hash_frame(np.asarray(frame, dtype=np.uint8))
        if not tracked_paths:
            return frame_hash
        extras = []
        for path in tracked_paths:
            value = _get_value_by_path(game, path)
            if value is not None:
                extras.append((_path_text(path), value))
        if not extras:
            return frame_hash
        digest = hashlib.blake2b(repr(tuple(extras)).encode("utf-8"), digest_size=10).hexdigest()
        return frame_hash + "|" + digest

    def _level_solved(self, level_idx: int, result: Any, game: Any) -> bool:
        if _local_state_name(result, game) == "WIN":
            return True
        curr = _local_current_level_index(game, result)
        if curr is not None and curr > int(level_idx):
            return True
        lvl_done = _local_levels_completed(result, game)
        if lvl_done > int(level_idx):
            return True
        return False

    def _game_over(self, result: Any, game: Any) -> bool:
        return _local_state_name(result, game) == "GAME_OVER"

    def _discover_tracked_paths(
        self,
        root_game: Any,
        root_frame: np.ndarray,
        sample_proposals: list[dict[str, Any]],
        level_idx: int,
    ) -> list[tuple[Any, ...]]:
        base_flat = _flatten_small_state(root_game, depth=2, max_items=180)
        if not base_flat:
            return []

        changed: set[tuple[Any, ...]] = set()
        semantic: list[tuple[int, tuple[Any, ...]]] = []
        for path in base_flat.keys():
            score = _semantic_path_score(path)
            if score > 0:
                semantic.append((score, path))
        semantic.sort(key=lambda t: (-t[0], len(t[1]), _path_text(t[1])))

        for proposal in sample_proposals[:18]:
            try:
                g2 = _clone_game(root_game)
                result = _local_step(g2, int(proposal["action_id"]), proposal.get("data"))
                if result is None:
                    continue
                flat2 = _flatten_small_state(g2, depth=2, max_items=180)
            except Exception:
                continue
            for path, v0 in base_flat.items():
                v1 = flat2.get(path)
                if v1 is None:
                    continue
                if v1 != v0:
                    changed.add(path)

        ranked: list[tuple[int, tuple[Any, ...]]] = []
        for path in changed:
            score = 20 + _semantic_path_score(path) - len(path)
            ranked.append((score, path))
        for score, path in semantic:
            if path not in changed:
                ranked.append((3 + score, path))

        ranked.sort(key=lambda t: (-t[0], len(t[1]), _path_text(t[1])))
        out = [path for _, path in ranked[:48]]

        # Drop fields that behave like volatile counters but keep level index / score semantics.
        filtered = []
        for path in out:
            text = _path_text(path)
            if "action_count" in text or "last_action" in text:
                continue
            filtered.append(path)
        return filtered[:40]

    def _segment_click_candidates(self, frame: np.ndarray, game: Any, level_idx: int, max_points: int = 42) -> list[dict[str, Any]]:
        frame = np.asarray(frame, dtype=np.uint8)
        h, w = frame.shape
        bg = int(np.bincount(frame.reshape(-1), minlength=16).argmax())
        scores: dict[tuple[int, int], tuple[float, str]] = {}

        def add_pt(x: int, y: int, score: float, source: str) -> None:
            if not (0 <= int(x) < w and 0 <= int(y) < h):
                return
            key = (int(x), int(y))
            prev = scores.get(key)
            if prev is None or float(score) > float(prev[0]):
                scores[key] = (float(score), str(source))

        for rank, (score, x, y, source) in enumerate(_hidden_coord_points_from_obj(game, frame.shape, max_points=20)):
            add_pt(x, y, 90.0 + float(score) - 0.1 * rank, source)

        transfer_points = self.transfer_click_points.get(int(level_idx), [])
        for j, (x, y) in enumerate(transfer_points[:12]):
            add_pt(x, y, 88.0 - 0.2 * j, "transfer")

        labels, segments = self.fp.segment_frame(frame)
        total_area = int(frame.size)
        ordered_segments = []
        for seg_id, seg in enumerate(segments):
            area = int(seg["area"])
            color = int(seg["color"])
            if color == bg and area > total_area * 0.15:
                continue
            base_score = 72.0 - 6.0 * float(self.fp.segment_group(seg)) - 0.02 * float(area)
            if color != bg:
                base_score += 10.0
            if int(seg.get("number_of_twins", 0)) == 0:
                base_score += 2.0
            ordered_segments.append((base_score, seg_id, seg))
        ordered_segments.sort(key=lambda t: (-t[0], int(t[1])))

        for base_score, seg_id, seg in ordered_segments[:24]:
            reps = self.fp.representative_points(seg)
            x1, y1, x2, y2 = [int(v) for v in seg["bounding_box"]]
            cx = int(round(0.5 * (x1 + x2)))
            cy = int(round(0.5 * (y1 + y2)))

            for rep_rank, (x, y) in enumerate(reps[:3]):
                add_pt(int(x), int(y), base_score - 0.6 * rep_rank, f"segment:{seg_id}")
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = int(x + dx), int(y + dy)
                    if 0 <= nx < w and 0 <= ny < h and int(frame[ny, nx]) == bg:
                        add_pt(nx, ny, base_score - 2.0 - 0.3 * rep_rank, f"boundary:{seg_id}")

            for x, y in [
                (cx, cy),
                (x1 - 1, cy),
                (x2 + 1, cy),
                (cx, y1 - 1),
                (cx, y2 + 1),
                (x1, y1),
                (x2, y1),
                (x1, y2),
                (x2, y2),
            ]:
                add_pt(int(x), int(y), base_score - 2.5, f"bbox:{seg_id}")

        invalid = np.zeros(frame.shape, dtype=bool)
        for (x, y) in scores.keys():
            invalid[int(y), int(x)] = True

        try:
            coarse = self.fp.coarse_grid_points(frame, invalid_mask=invalid, grid_n=5 if max(h, w) >= 32 else 4)
        except Exception:
            coarse = []
        for j, (x, y) in enumerate(coarse[:20]):
            add_pt(int(x), int(y), 36.0 - 0.35 * j, "grid")

        specials = [
            (0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1),
            (w // 2, h // 2), (w // 2, 0), (0, h // 2),
            (w - 1, h // 2), (w // 2, h - 1),
        ]
        for j, (x, y) in enumerate(specials):
            add_pt(int(x), int(y), 18.0 - 0.15 * j, "special")

        ordered = sorted(
            [(score, source, x, y) for (x, y), (score, source) in scores.items()],
            key=lambda t: (-t[0], t[3], t[2]),
        )
        out = []
        for score, source, x, y in ordered[:max_points]:
            out.append(
                {
                    "action_id": 6,
                    "data": {"x": int(x), "y": int(y)},
                    "source": str(source),
                    "priority": float(score),
                }
            )
        return out

    def _raw_proposals(
        self,
        game: Any,
        frame: np.ndarray,
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
        depth: int = 0,
        current_path: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
    ) -> list[dict[str, Any]]:
        current_path = current_path or []
        avail = _local_available_ids(None, game)
        proposals: list[dict[str, Any]] = []

        preferred_simple = None
        if prev_solution and depth < len(prev_solution):
            if int(prev_solution[depth][0]) != 6:
                preferred_simple = int(prev_solution[depth][0])

        simple_ids = [aid for aid in sorted(avail) if int(aid) != 6]
        if preferred_simple in simple_ids:
            simple_ids = [preferred_simple] + [aid for aid in simple_ids if aid != preferred_simple]

        for aid in simple_ids:
            proposals.append(
                {
                    "action_id": int(aid),
                    "data": None,
                    "source": "simple",
                    "priority": 50.0 + (8.0 if aid == preferred_simple else 0.0),
                }
            )

        if 6 in avail:
            click_props = self._segment_click_candidates(frame, game, level_idx, max_points=42)
            proposals.extend(click_props)

        dedup = []
        seen = set()
        for cand in proposals:
            if cand["data"] is None:
                key = (int(cand["action_id"]), None)
            else:
                key = (int(cand["action_id"]), int(cand["data"]["x"]), int(cand["data"]["y"]))
            if key in seen:
                continue
            seen.add(key)
            dedup.append(cand)
        return dedup

    def _expand_node(
        self,
        game: Any,
        frame: np.ndarray,
        path: list[tuple[int, Optional[dict[str, int]]]],
        tracked_paths: list[tuple[Any, ...]],
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
        max_children: int = 18,
    ) -> list[dict[str, Any]]:
        current_hash = self._state_hash(game, frame, tracked_paths)
        proposals = self._raw_proposals(game, frame, level_idx, prev_solution=prev_solution, depth=len(path), current_path=path)

        children = []
        seen_next_hash: set[str] = set()

        for cand in proposals:
            aid = int(cand["action_id"])
            data = cand.get("data")
            try:
                g2 = _clone_game(game)
                result = _local_step(g2, aid, data=data)
                frame2 = _local_frame_from_result(result, g2)
                if frame2 is None:
                    continue
            except Exception:
                continue

            solved = self._level_solved(level_idx, result, g2)
            game_over = self._game_over(result, g2)
            next_hash = self._state_hash(g2, frame2, tracked_paths)

            if not solved and game_over:
                continue
            if not solved and next_hash == current_hash:
                continue
            if next_hash in seen_next_hash:
                continue
            seen_next_hash.add(next_hash)

            diff = int(np.count_nonzero(np.asarray(frame, dtype=np.uint8) != np.asarray(frame2, dtype=np.uint8)))
            reward = min(6.0, math.log1p(max(0, diff)))
            if diff == 0 and next_hash != current_hash:
                reward += 1.4
            reward += float(cand.get("priority", 0.0)) / 55.0
            if aid != 6:
                reward += 0.15
            if path and aid == int(path[-1][0]) and aid != 6:
                reward -= 0.08
            if solved:
                reward += 1000.0

            new_path = path + [(aid, None if data is None else {"x": int(data["x"]), "y": int(data["y"])})]
            children.append(
                {
                    "game": g2,
                    "frame": frame2,
                    "path": new_path,
                    "hash": next_hash,
                    "step_reward": float(reward),
                    "solved": bool(solved),
                }
            )

        children.sort(key=lambda c: (-float(c["step_reward"]), len(c["path"]), c["hash"]))
        return children[:max_children]

    def _replay_plan(
        self,
        root_game: Any,
        level_idx: int,
        plan: list[tuple[int, Optional[dict[str, int]]]],
    ) -> tuple[bool, list[tuple[int, Optional[dict[str, int]]]], Any, Optional[np.ndarray], bool]:
        g = _clone_game(root_game)
        hist: list[tuple[int, Optional[dict[str, int]]]] = []
        last_frame = _local_frame_from_result(None, g)
        terminal = False
        for aid, data in plan:
            try:
                result = _local_step(g, int(aid), data=data)
                last_frame = _local_frame_from_result(result, g)
                hist.append((int(aid), None if data is None else {"x": int(data["x"]), "y": int(data["y"])}))
            except Exception:
                break
            if self._level_solved(level_idx, result, g):
                return True, hist, g, last_frame, False
            if self._game_over(result, g):
                terminal = True
                break
        return False, hist, g, last_frame, terminal

    def _translate_solution(
        self,
        prev_solution: list[tuple[int, Optional[dict[str, int]]]],
        prev_frame: np.ndarray,
        curr_frame: np.ndarray,
    ) -> list[tuple[int, Optional[dict[str, int]]]]:
        dx, dy = _estimate_click_offset(prev_frame, curr_frame)
        if dx == 0.0 and dy == 0.0:
            return list(prev_solution)
        h, w = curr_frame.shape
        translated = []
        for aid, data in prev_solution:
            if data and "x" in data and "y" in data:
                nd = {
                    "x": int(max(0, min(w - 1, round(float(data["x"]) + dx)))),
                    "y": int(max(0, min(h - 1, round(float(data["y"]) + dy)))),
                }
                translated.append((int(aid), nd))
            else:
                translated.append((int(aid), None if data is None else dict(data)))
        return translated

    def _seed_roots_from_transfer(
        self,
        level_idx: int,
        root_game: Any,
        root_frame: np.ndarray,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]],
    ) -> tuple[Optional[list[tuple[int, Optional[dict[str, int]]]]], list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]]]:
        warm_roots: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = []
        self.transfer_click_points[int(level_idx)] = []
        if not prev_solution:
            return None, warm_roots

        direct_solved, direct_hist, direct_game, direct_frame, direct_terminal = self._replay_plan(root_game, level_idx, prev_solution)
        if direct_solved:
            return direct_hist, warm_roots
        if direct_hist and direct_frame is not None and not direct_terminal:
            warm_roots.append((direct_game, direct_frame, direct_hist, 0.4 * len(direct_hist)))

        prev_frame = self.root_frames.get(int(level_idx) - 1)
        translated = None
        if prev_frame is not None:
            translated = self._translate_solution(prev_solution, prev_frame, root_frame)
            translated_clicks = []
            for aid, data in translated:
                if int(aid) == 6 and data is not None:
                    translated_clicks.append((int(data["x"]), int(data["y"])))
            self.transfer_click_points[int(level_idx)] = translated_clicks[:12]

            trans_solved, trans_hist, trans_game, trans_frame, trans_terminal = self._replay_plan(root_game, level_idx, translated)
            if trans_solved:
                return trans_hist, warm_roots
            if trans_hist and trans_frame is not None and not trans_terminal:
                warm_roots.append((trans_game, trans_frame, trans_hist, 0.6 * len(trans_hist)))

        return None, warm_roots

    def _short_bfs(
        self,
        root_nodes: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]],
        tracked_paths: list[tuple[Any, ...]],
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]],
        deadline: float,
        max_depth: int,
        max_states: int,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        queue = deque()
        visited: dict[str, int] = {}
        states_examined = 0

        for game, frame, path, score in root_nodes:
            h = self._state_hash(game, frame, tracked_paths)
            if visited.get(h, 10 ** 9) <= len(path):
                continue
            visited[h] = len(path)
            queue.append((game, frame, path, float(score)))

        while queue and time.time() < deadline and states_examined < max_states:
            game, frame, path, score = queue.popleft()
            if len(path) >= max_depth:
                continue

            children = self._expand_node(
                game,
                frame,
                path,
                tracked_paths,
                level_idx,
                prev_solution=prev_solution,
                max_children=16,
            )
            states_examined += max(1, len(children))
            for child in children:
                if child["solved"]:
                    return child["path"]
            for child in children:
                h = str(child["hash"])
                plen = len(child["path"])
                if visited.get(h, 10 ** 9) <= plen:
                    continue
                visited[h] = plen
                queue.append((child["game"], child["frame"], child["path"], score + float(child["step_reward"])))
        return None

    def _best_first(
        self,
        root_nodes: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]],
        tracked_paths: list[tuple[Any, ...]],
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]],
        deadline: float,
        max_depth: int,
        max_states: int,
        max_heap: int,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        heap: list[tuple[float, int, int, Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = []
        visited_len: dict[str, int] = {}
        counter = 0

        for game, frame, path, score in root_nodes:
            h = self._state_hash(game, frame, tracked_paths)
            plen = len(path)
            if visited_len.get(h, 10 ** 9) <= plen:
                continue
            visited_len[h] = plen
            priority = -(float(score) - 0.02 * float(plen))
            heapq.heappush(heap, (priority, plen, counter, game, frame, path, float(score)))
            counter += 1

        explored = 0
        while heap and time.time() < deadline and explored < max_states:
            _, _, _, game, frame, path, node_score = heapq.heappop(heap)
            if len(path) >= max_depth:
                continue

            children = self._expand_node(
                game,
                frame,
                path,
                tracked_paths,
                level_idx,
                prev_solution=prev_solution,
                max_children=18,
            )
            explored += max(1, len(children))
            for child in children:
                if child["solved"]:
                    return child["path"]

            for child in children:
                h = str(child["hash"])
                plen = len(child["path"])
                if visited_len.get(h, 10 ** 9) <= plen:
                    continue
                visited_len[h] = plen
                child_score = float(node_score) + float(child["step_reward"])
                priority = -(child_score - 0.025 * float(plen))
                heapq.heappush(heap, (priority, plen, counter, child["game"], child["frame"], child["path"], child_score))
                counter += 1

            if len(heap) > max_heap * 2:
                heap = heapq.nsmallest(max_heap, heap)
                heapq.heapify(heap)

        return None

    def solve_level(
        self,
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
        time_budget_s: float = 20.0,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        if not self.loaded or self.game_cls is None:
            return None

        deadline = time.time() + max(1.0, float(time_budget_s))
        root_game, root_result, root_frame = self._new_game(level_idx)
        if root_frame is None:
            return None

        self.root_frames[int(level_idx)] = np.asarray(root_frame, dtype=np.uint8).copy()

        raw_samples = self._raw_proposals(root_game, root_frame, level_idx, prev_solution=prev_solution, depth=0, current_path=[])
        tracked_paths = self._discover_tracked_paths(root_game, root_frame, raw_samples, level_idx)
        self.tracked_paths_by_level[int(level_idx)] = tracked_paths

        solved_transfer, warm_transfer_roots = self._seed_roots_from_transfer(level_idx, root_game, root_frame, prev_solution)
        if solved_transfer is not None:
            self.solution_cache[int(level_idx)] = solved_transfer
            return solved_transfer

        root_nodes: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = [
            (root_game, root_frame, [], 0.0)
        ]
        root_nodes.extend(warm_transfer_roots)

        dedup_roots: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = []
        seen_hashes: dict[str, int] = {}
        for game, frame, path, score in root_nodes:
            h = self._state_hash(game, frame, tracked_paths)
            if seen_hashes.get(h, 10 ** 9) <= len(path):
                continue
            seen_hashes[h] = len(path)
            dedup_roots.append((game, frame, path, score))
        root_nodes = dedup_roots

        depth_table = {
            0: (10, 20, 18000, 90000, 1200),
            1: (12, 24, 30000, 130000, 1600),
            2: (14, 28, 42000, 180000, 2000),
            3: (16, 32, 65000, 250000, 2600),
            4: (18, 36, 90000, 340000, 3200),
            5: (20, 42, 130000, 460000, 3800),
        }
        short_depth, long_depth, short_states, long_states, max_heap = depth_table.get(
            int(level_idx), (22, 48, 180000, 600000, 4400)
        )

        plan = self._short_bfs(
            root_nodes,
            tracked_paths,
            level_idx,
            prev_solution,
            deadline=min(deadline, time.time() + max(2.0, 0.35 * float(time_budget_s))),
            max_depth=int(short_depth),
            max_states=int(short_states),
        )
        if plan is not None:
            self.solution_cache[int(level_idx)] = plan
            return plan

        if time.time() >= deadline:
            return None

        plan = self._best_first(
            root_nodes,
            tracked_paths,
            level_idx,
            prev_solution,
            deadline=deadline,
            max_depth=int(long_depth),
            max_states=int(long_states),
            max_heap=int(max_heap),
        )
        if plan is not None:
            self.solution_cache[int(level_idx)] = plan
            return plan
        return None

class DynamicsTracker:
    """Persistent per-game mechanics learner shared across levels."""
    def __init__(self, fp: FrameProcessor) -> None:
        self.fp = fp
        self.action_delta_counts: dict[int, dict[tuple[int, int], float]] = defaultdict(lambda: defaultdict(float))
        self.action_change_counts: dict[int, BetaCounter] = defaultdict(BetaCounter)
        self.action_nontranslation_counts: dict[int, BetaCounter] = defaultdict(BetaCounter)
        self.entity_move_strength: dict[tuple[Any, ...], float] = defaultdict(float)
        self.entity_actions: dict[tuple[Any, ...], set[int]] = defaultdict(set)
        self.entity_last_center: dict[tuple[Any, ...], tuple[float, float]] = {}
        self.recent_success_actions: deque[int] = deque(maxlen=16)
        self.last_avatar_key: Optional[tuple[Any, ...]] = None
        self.last_avatar_center: Optional[tuple[float, float]] = None

    def start_new_level(self) -> None:
        self.last_avatar_center = None
        self.recent_success_actions.clear()

    def note_reset(self) -> None:
        self.last_avatar_center = None

    def segment_signature(self, seg: dict[str, Any]) -> tuple[Any, ...]:
        x1, y1, x2, y2 = seg["bounding_box"]
        return (
            int(seg["color"]),
            int(seg["area"]),
            int(x2 - x1 + 1),
            int(y2 - y1 + 1),
            int(bool(seg["is_rectangle"])),
        )

    def segment_center(self, seg: dict[str, Any]) -> tuple[float, float]:
        x1, y1, x2, y2 = seg["bounding_box"]
        return (0.5 * (x1 + x2), 0.5 * (y1 + y2))

    def _pair_rank(self, sig: tuple[Any, ...], seg: dict[str, Any], *, dx: int, dy: int) -> float:
        color, area, width, height, rect = sig
        dist = abs(int(dx)) + abs(int(dy))
        score = 1.0
        if color in self.fp.salient_colors:
            score += 0.50
        if area <= 4:
            score += 0.65
        elif area <= 12:
            score += 0.45
        elif area <= 30:
            score += 0.20
        if max(width, height) <= 5:
            score += 0.25
        if not rect:
            score += 0.12
        score += 0.18 * len(self.entity_actions.get(sig, set()))
        if dist == 1:
            score += 0.30
        elif dist <= 3:
            score += 0.10
        score -= 0.02 * max(0, int(area) - 1)
        return float(score)

    def _match_translations(
        self,
        prev_segments: list[dict[str, Any]],
        next_segments: list[dict[str, Any]],
    ) -> list[dict[str, Any]]:
        prev_by_sig: dict[tuple[Any, ...], list[tuple[int, dict[str, Any]]]] = defaultdict(list)
        next_by_sig: dict[tuple[Any, ...], list[tuple[int, dict[str, Any]]]] = defaultdict(list)

        for i, seg in enumerate(prev_segments):
            if int(seg["color"]) == self.fp.status_bar_color:
                continue
            prev_by_sig[self.segment_signature(seg)].append((i, seg))
        for i, seg in enumerate(next_segments):
            if int(seg["color"]) == self.fp.status_bar_color:
                continue
            next_by_sig[self.segment_signature(seg)].append((i, seg))

        matches: list[dict[str, Any]] = []
        for sig in set(prev_by_sig).intersection(next_by_sig):
            prev_items = prev_by_sig[sig]
            next_items = next_by_sig[sig]
            if not prev_items or not next_items:
                continue
            if len(prev_items) > 5 or len(next_items) > 5:
                continue

            used_next: set[int] = set()
            ranked_prev = sorted(prev_items, key=lambda item: int(item[1]["area"]))
            for prev_idx, prev_seg in ranked_prev:
                pcx, pcy = self.segment_center(prev_seg)
                best: Optional[tuple[float, int, dict[str, Any], float, float]] = None
                for next_idx, next_seg in next_items:
                    if next_idx in used_next:
                        continue
                    ncx, ncy = self.segment_center(next_seg)
                    dist = abs(pcx - ncx) + abs(pcy - ncy)
                    if best is None or dist < best[0]:
                        best = (dist, next_idx, next_seg, ncx, ncy)
                if best is None:
                    continue
                dist, next_idx, next_seg, ncx, ncy = best
                if dist > 6.0:
                    continue
                used_next.add(next_idx)
                dx = int(round(ncx - pcx))
                dy = int(round(ncy - pcy))
                if dx == 0 and dy == 0:
                    continue
                matches.append(
                    {
                        "sig": sig,
                        "prev_idx": int(prev_idx),
                        "next_idx": int(next_idx),
                        "prev": prev_seg,
                        "next": next_seg,
                        "prev_center": (float(pcx), float(pcy)),
                        "next_center": (float(ncx), float(ncy)),
                        "dx": int(dx),
                        "dy": int(dy),
                        "rank": self._pair_rank(sig, prev_seg, dx=dx, dy=dy),
                    }
                )
        matches.sort(key=lambda m: float(m["rank"]), reverse=True)
        return matches

    def observe(
        self,
        prev_view: FrameView,
        candidate: dict[str, Any],
        next_view: Optional[FrameView],
        *,
        game_over: bool = False,
        level_up: bool = False,
    ) -> None:
        aid = int(candidate.get("action_id", -1))
        kind = str(candidate.get("kind", ""))

        if kind not in {"simple", "undo"}:
            return
        if aid == 7 or kind == "undo":
            self.note_reset()
            return

        if level_up:
            self.action_change_counts[aid].update(1.0, 0.0)
            self.recent_success_actions.append(aid)
            return

        if game_over:
            self.action_change_counts[aid].update(0.0, 1.0)
            return

        if next_view is None:
            self.action_change_counts[aid].update(0.0, 1.0)
            return

        changed = bool(next_view.node_id != prev_view.node_id)
        self.action_change_counts[aid].update(1.0 if changed else 0.0, 0.0 if changed else 1.0)
        if not changed:
            return

        matches = self._match_translations(prev_view.segments, next_view.segments)
        if not matches:
            self.action_nontranslation_counts[aid].update(1.0, 0.0)
            self.recent_success_actions.append(aid)
            return

        self.action_nontranslation_counts[aid].update(0.0, 1.0)
        top_matches = matches[: min(3, len(matches))]
        for k, match in enumerate(top_matches):
            sig = match["sig"]
            delta = (int(match["dx"]), int(match["dy"]))
            weight = max(0.15, float(match["rank"])) * (1.0 if k == 0 else 0.45)
            self.action_delta_counts[aid][delta] += weight
            self.entity_move_strength[sig] += weight
            self.entity_actions[sig].add(aid)
            self.entity_last_center[sig] = tuple(match["next_center"])

        best_key = self.best_entity_key()
        self.last_avatar_key = best_key
        if best_key is not None:
            self.last_avatar_center = self.entity_last_center.get(best_key, self.last_avatar_center)
        self.recent_success_actions.append(aid)

    def best_entity_key(self) -> Optional[tuple[Any, ...]]:
        best_key = None
        best_score = -1e18
        for sig, strength in self.entity_move_strength.items():
            color, area, width, height, rect = sig
            actions = len(self.entity_actions.get(sig, set()))
            score = float(strength)
            score += 0.95 * float(actions)
            if color in self.fp.salient_colors:
                score += 0.40
            if area <= 4:
                score += 0.60
            elif area <= 12:
                score += 0.35
            elif area <= 30:
                score += 0.12
            if max(width, height) <= 5:
                score += 0.20
            if not rect:
                score += 0.10
            score -= 0.018 * max(0, int(area) - 1)
            if score > best_score:
                best_score = score
                best_key = sig
        return best_key

    def locate_avatar(self, view: FrameView) -> Optional[dict[str, Any]]:
        key = self.best_entity_key()
        if key is None:
            return None
        candidates: list[dict[str, Any]] = []
        for seg in view.segments:
            sig = self.segment_signature(seg)
            if sig == key:
                candidates.append(seg)

        if not candidates:
            color, area, width, height, rect = key
            for seg in view.segments:
                x1, y1, x2, y2 = seg["bounding_box"]
                w = x2 - x1 + 1
                h = y2 - y1 + 1
                if (
                    int(seg["color"]) == int(color)
                    and abs(int(seg["area"]) - int(area)) <= 2
                    and abs(w - int(width)) <= 1
                    and abs(h - int(height)) <= 1
                ):
                    candidates.append(seg)

        if not candidates:
            return None

        if self.last_avatar_center is None:
            chosen = min(candidates, key=lambda seg: int(seg["area"]))
        else:
            ax, ay = self.last_avatar_center
            chosen = min(
                candidates,
                key=lambda seg: abs(self.segment_center(seg)[0] - ax) + abs(self.segment_center(seg)[1] - ay),
            )
        self.last_avatar_center = self.segment_center(chosen)
        return chosen

    def primary_delta(self, action_id: int) -> tuple[Optional[tuple[int, int]], float]:
        counts = self.action_delta_counts.get(int(action_id))
        if not counts:
            return None, 0.0
        best_delta, best_weight = max(counts.items(), key=lambda kv: float(kv[1]))
        total = float(sum(float(v) for v in counts.values()))
        conf = float(best_weight / max(1e-9, total))
        if best_weight < 0.35:
            return None, 0.0
        return (int(best_delta[0]), int(best_delta[1])), conf

    def action_change_rate(self, action_id: int) -> float:
        return float(self.action_change_counts[int(action_id)].mean())

    def interaction_propensity(self, action_id: int) -> float:
        aid = int(action_id)
        change_rate = self.action_change_counts[aid].mean()
        nontranslation_rate = self.action_nontranslation_counts[aid].mean()
        return float(0.55 * nontranslation_rate + 0.45 * change_rate)


class StrategicHybridAgent(HybridLevelExplorerAgent):
    """Graph explorer + per-game memory + lightweight mechanics inference."""
    def __init__(self, game_id: str, n_groups: int = 5, global_memory: Optional[FeatureMemory] = None) -> None:
        self.game_memory = FeatureMemory()
        self.game_dynamics: Optional[DynamicsTracker] = None
        super().__init__(game_id, n_groups=n_groups, global_memory=global_memory)
        self.game_dynamics = DynamicsTracker(self.fp)

    def reset_level_state(self) -> None:
        super().reset_level_state()
        if getattr(self, "game_dynamics", None) is not None:
            self.game_dynamics.start_new_level()

    def note_external_reset(self, count: int = 1) -> None:
        super().note_external_reset(count)
        if self.game_dynamics is not None:
            self.game_dynamics.note_reset()

    def remember(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if candidate.get("kind") in {"meta_reset", "undo"}:
            return
        self.level_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )
        self.game_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )
        self.global_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )

    def _majority_non_status_color(self, frame: np.ndarray) -> int:
        vals, counts = np.unique(np.asarray(frame, dtype=np.uint8), return_counts=True)
        pairs = [(int(v), int(c)) for v, c in zip(vals.tolist(), counts.tolist()) if int(v) != self.fp.status_bar_color]
        if not pairs:
            return int(vals[counts.argmax()]) if len(vals) else 0
        return int(max(pairs, key=lambda vc: vc[1])[0])

    def _compute_view_metadata(self, view: FrameView) -> None:
        if hasattr(view, "label_map") and hasattr(view, "color_hist") and hasattr(view, "segment_target_scores"):
            return
        label_map, _ = self.fp.segment_frame(view.masked_frame_for_segmentation)
        vals, counts = np.unique(np.asarray(view.raw_frame, dtype=np.uint8), return_counts=True)
        color_hist = {int(v): int(c) for v, c in zip(vals.tolist(), counts.tolist())}
        background_color = self._majority_non_status_color(view.raw_frame)

        target_scores: dict[int, float] = {}
        total_pixels = max(1, int(view.raw_frame.size))
        rare_thr = max(2, int(round(total_pixels * 0.012)))
        uncommon_thr = max(6, int(round(total_pixels * 0.03)))

        for seg_id, seg in enumerate(view.segments):
            color = int(seg["color"])
            area = int(seg["area"])
            score = 0.0
            if color == self.fp.status_bar_color:
                score = -1.0
            else:
                count = int(color_hist.get(color, area))
                if color in self.fp.salient_colors:
                    score += 1.05
                if count <= rare_thr:
                    score += 0.95
                elif count <= uncommon_thr:
                    score += 0.45
                if area <= 2:
                    score += 0.55
                elif area <= 9:
                    score += 0.35
                elif area <= 25:
                    score += 0.12
                if int(seg.get("number_of_twins", 0)) == 0:
                    score += 0.22
                if self.fp.edge_mask(seg) != 0:
                    score -= 0.18
                if color == background_color:
                    score -= 0.45
                if bool(seg.get("is_rectangle")) and area >= max(36, total_pixels // 8):
                    score -= 0.12
            target_scores[int(seg_id)] = float(score)

        setattr(view, "label_map", label_map)
        setattr(view, "color_hist", color_hist)
        setattr(view, "background_color", int(background_color))
        setattr(view, "segment_target_scores", target_scores)
        setattr(view, "target_points_cache", None)

        total_pixels = max(1, int(view.raw_frame.size))
        rare_thr = max(2, int(round(total_pixels * 0.012)))
        for cand in view.candidates:
            if str(cand.get("kind", "")) != "click":
                continue
            feats = cand.setdefault("features", {})
            data = cand.get("data") or {}
            x = int(data.get("x", 0))
            y = int(data.get("y", 0))
            if not (0 <= y < view.raw_frame.shape[0] and 0 <= x < view.raw_frame.shape[1]):
                continue
            seg_id = int(label_map[y, x]) if 0 <= int(label_map[y, x]) < len(view.segments) else -1
            seg = view.segments[seg_id] if seg_id >= 0 else None
            color = int(feats.get("color", int(view.raw_frame[y, x])))
            count = int(color_hist.get(color, 0))
            feats["color_count_bucket"] = int(min(7, max(0, int(math.log2(max(1, count))))))
            feats["rare_color"] = int(count <= rare_thr)
            feats["salient_color"] = int(color in self.fp.salient_colors)
            feats["bg_color"] = int(color == background_color)
            feats["target_like"] = int(seg_id >= 0 and target_scores.get(seg_id, 0.0) >= 0.75)
            feats["singleton_color"] = int(
                seg is not None and int(color_hist.get(color, 0)) <= int(seg["area"]) + 1 and int(seg.get("number_of_twins", 0)) == 0
            )

    def _outside_points_for_segment(
        self,
        view: FrameView,
        seg_id: int,
        seg: dict[str, Any],
        *,
        max_points: int = 4,
    ) -> list[tuple[int, int]]:
        self._compute_view_metadata(view)
        label_map = getattr(view, "label_map")
        h, w = view.raw_frame.shape
        x1, y1, x2, y2 = seg["bounding_box"]
        xs = [int(round(0.5 * (x1 + x2)))]
        ys = [int(round(0.5 * (y1 + y2)))]
        if x2 > x1:
            xs.extend([int(round(x1 + 0.25 * (x2 - x1))), int(round(x1 + 0.75 * (x2 - x1)))])
        if y2 > y1:
            ys.extend([int(round(y1 + 0.25 * (y2 - y1))), int(round(y1 + 0.75 * (y2 - y1)))])

        raw_points: list[tuple[int, int]] = []
        for x in xs:
            raw_points.append((x, y1 - 1))
            raw_points.append((x, y2 + 1))
        for y in ys:
            raw_points.append((x1 - 1, y))
            raw_points.append((x2 + 1, y))
        if int(seg["area"]) <= 4:
            raw_points.extend(
                [
                    (x1 - 1, y1),
                    (x1, y1 - 1),
                    (x2 + 1, y2),
                    (x2, y2 + 1),
                ]
            )

        out: list[tuple[int, int]] = []
        seen: set[tuple[int, int]] = set()
        for x, y in raw_points:
            if not (0 <= x < w and 0 <= y < h):
                continue
            if self.status_bar_mask is not None and bool(self.status_bar_mask[y, x]):
                continue
            if int(label_map[y, x]) == int(seg_id):
                continue
            key = (int(x), int(y))
            if key in seen:
                continue
            seen.add(key)
            out.append(key)
            if len(out) >= max_points:
                break
        return out

    def _clone_view_with_candidates(
        self,
        view: FrameView,
        candidates: list[dict[str, Any]],
        groups: list[set[int]],
    ) -> FrameView:
        new_view = FrameView(
            node_id=view.node_id,
            hash_id=view.hash_id,
            raw_frame=view.raw_frame,
            masked_frame_for_segmentation=view.masked_frame_for_segmentation,
            hash_frame_input=view.hash_frame_input,
            num_frames=view.num_frames,
            state=view.state,
            levels_completed=view.levels_completed,
            available_action_ids=set(view.available_action_ids),
            undo_available=bool(view.undo_available),
            segments=view.segments,
            candidates=candidates,
            groups=groups,
        )
        for attr in ["label_map", "color_hist", "background_color", "segment_target_scores", "target_points_cache"]:
            if hasattr(view, attr):
                setattr(new_view, attr, getattr(view, attr))
        setattr(new_view, "_strategic_augmented", True)
        return new_view

    def _build_view(self, obs: Any, env: Any) -> FrameView:
        view = super()._build_view(obs, env)
        self._compute_view_metadata(view)
        already_augmented = any(
            str((cand.get("features") or {}).get("source", "")) in {"neighbor", "model_top"}
            or (
                isinstance(cand.get("tag"), tuple)
                and cand.get("tag")
                and cand.get("tag")[0] in {"neighbor", "model_top"}
            )
            for cand in view.candidates
        )
        if getattr(view, "_strategic_augmented", False) or already_augmented:
            setattr(view, "_strategic_augmented", True)
            self.view_cache[view.node_id] = view
            return view
        if 6 not in view.available_action_ids:
            setattr(view, "_strategic_augmented", True)
            self.view_cache[view.node_id] = view
            return view

        candidates = [dict(c) for c in view.candidates]
        groups = [set(g) for g in view.groups]
        existing_clicks: set[tuple[int, int]] = set()
        for cand in candidates:
            if str(cand.get("kind", "")) != "click":
                continue
            data = cand.get("data") or {}
            existing_clicks.add((int(data.get("x", 0)), int(data.get("y", 0))))

        ordered_segments = sorted(
            enumerate(view.segments),
            key=lambda item: (-float(getattr(view, "segment_target_scores")[item[0]]), self._segment_priority_key(item[1])),
        )
        max_added = 14 if set(view.available_action_ids).issubset({6, 7}) else 10
        added = 0
        total_pixels = max(1, int(view.raw_frame.size))
        rare_thr = max(2, int(round(total_pixels * 0.012)))

        for seg_id, seg in ordered_segments:
            if added >= max_added:
                break
            color = int(seg["color"])
            if color == self.fp.status_bar_color:
                continue
            target_score = float(getattr(view, "segment_target_scores").get(int(seg_id), 0.0))
            if target_score < 0.22 and added >= 6:
                break

            base_group = self.fp.segment_group(seg)
            if target_score >= 1.10:
                base_group = 0
            elif target_score >= 0.70:
                base_group = min(base_group, 1)
            else:
                base_group = min(max(base_group, 1), self.n_groups - 2)

            outside_points = self._outside_points_for_segment(view, seg_id, seg, max_points=4)
            color_count = int(getattr(view, "color_hist").get(color, int(seg["area"])))
            for rep_idx, (x, y) in enumerate(outside_points):
                if added >= max_added:
                    break
                key = (int(x), int(y))
                if key in existing_clicks:
                    continue
                group = min(base_group + (1 if rep_idx >= 2 else 0), self.n_groups - 1)
                feats = self.fp.candidate_features(
                    kind="click",
                    action_id=6,
                    group=group,
                    x=int(x),
                    y=int(y),
                    raw_frame=view.raw_frame,
                    segment=seg,
                    source="neighbor",
                    rep_idx=rep_idx,
                )
                feats["rare_color"] = int(color_count <= rare_thr)
                feats["salient_color"] = int(color in self.fp.salient_colors)
                feats["bg_color"] = int(color == getattr(view, "background_color"))
                feats["target_like"] = int(target_score >= 0.75)
                feats["singleton_color"] = int(
                    int(getattr(view, "color_hist").get(color, 0)) <= int(seg["area"]) + 1 and int(seg.get("number_of_twins", 0)) == 0
                )
                feats["color_count_bucket"] = int(min(7, max(0, int(math.log2(max(1, color_count))))))
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "click",
                        "action_id": 6,
                        "data": {"x": int(x), "y": int(y)},
                        "group": int(group),
                        "tag": ("neighbor", int(seg_id), int(rep_idx), int(x), int(y)),
                        "features": feats,
                    }
                )
                groups[group].add(cand_idx)
                existing_clicks.add(key)
                added += 1

        model_budget = 8 if set(view.available_action_ids).issubset({6, 7}) else 5
        for rank, (x, y, prob) in enumerate(self._model_top_click_points(view, existing_clicks, max_points=model_budget)):
            group = 0 if rank < 2 else (1 if rank < 4 else 2)
            feats = self.fp.candidate_features(
                kind="click",
                action_id=6,
                group=group,
                x=int(x),
                y=int(y),
                raw_frame=view.raw_frame,
                segment=None,
                source="model_top",
                rep_idx=rank,
            )
            feats["model_rank"] = int(rank)
            feats["model_hot"] = int(prob >= 0.62)
            feats["model_score_bucket"] = int(min(7, max(0, round((float(prob) - 0.50) * 20.0))))
            cand_idx = len(candidates)
            candidates.append(
                {
                    "kind": "click",
                    "action_id": 6,
                    "data": {"x": int(x), "y": int(y)},
                    "group": int(group),
                    "tag": ("model_top", int(rank), int(x), int(y)),
                    "features": feats,
                }
            )
            groups[group].add(cand_idx)
            existing_clicks.add((int(x), int(y)))

        new_view = self._clone_view_with_candidates(view, candidates, groups)
        self.view_cache[new_view.node_id] = new_view
        return new_view


    def _model_top_click_points(
        self,
        view: FrameView,
        existing_clicks: set[tuple[int, int]],
        *,
        max_points: int = 5,
    ) -> list[tuple[int, int, float]]:
        if 6 not in view.available_action_ids or max_points <= 0:
            return []
        if len(self.replay) < max(12, self.min_train // 2):
            return []
        _, click_probs = self._get_model_predictions(view)
        if click_probs is None:
            return []

        h, w = view.raw_frame.shape
        if h <= 0 or w <= 0:
            return []

        heat = np.asarray(click_probs[:h, :w], dtype=np.float32).copy()
        if heat.size == 0:
            return []

        if self.status_bar_mask is not None and self.status_bar_mask.shape == heat.shape:
            heat[self.status_bar_mask] = -1.0
        for x, y in existing_clicks:
            if 0 <= int(x) < w and 0 <= int(y) < h:
                heat[int(y), int(x)] = -1.0

        finite = heat[np.isfinite(heat)]
        if finite.size == 0:
            return []
        top_score = float(finite.max())
        median_score = float(np.median(finite))
        if top_score < 0.56 and top_score - median_score < 0.045:
            return []

        search_k = min(int(heat.size), max_points * 18)
        flat = heat.reshape(-1)
        if search_k <= 0:
            return []
        candidate_ids = np.argpartition(flat, -search_k)[-search_k:]
        candidate_ids = candidate_ids[np.argsort(flat[candidate_ids])[::-1]]

        points: list[tuple[int, int, float]] = []
        for idx in candidate_ids.tolist():
            prob = float(flat[idx])
            if not np.isfinite(prob) or prob < max(0.54, top_score - 0.12):
                continue
            y, x = divmod(int(idx), w)
            if any(max(abs(x - px), abs(y - py)) <= 2 for px, py, _ in points):
                continue
            points.append((int(x), int(y), prob))
            if len(points) >= max_points:
                break
        return points

    def _get_target_points(self, view: FrameView, avatar_seg: Optional[dict[str, Any]] = None) -> list[tuple[int, int, float, str]]:
        self._compute_view_metadata(view)
        cache = getattr(view, "target_points_cache", None)
        if cache is not None:
            return cache

        avatar_key = self.game_dynamics.best_entity_key() if self.game_dynamics is not None else None
        points: list[tuple[int, int, float, str]] = []
        seen: set[tuple[int, int, str]] = set()

        for seg_id, seg in enumerate(view.segments):
            if int(seg["color"]) == self.fp.status_bar_color:
                continue
            if avatar_seg is not None and seg is avatar_seg:
                continue
            if avatar_key is not None and self.game_dynamics is not None and self.game_dynamics.segment_signature(seg) == avatar_key:
                continue

            target_score = float(getattr(view, "segment_target_scores").get(int(seg_id), 0.0))
            if target_score < 0.25:
                continue

            reps = self.fp.representative_points(seg)[:1]
            for x, y in reps:
                key = (int(x), int(y), "touch")
                if key not in seen:
                    seen.add(key)
                    points.append((int(x), int(y), target_score, "touch"))

            for x, y in self._outside_points_for_segment(view, seg_id, seg, max_points=2):
                key = (int(x), int(y), "adjacent")
                if key not in seen:
                    seen.add(key)
                    points.append((int(x), int(y), max(0.10, target_score - 0.10), "adjacent"))

        points.sort(key=lambda item: float(item[2]), reverse=True)
        out = points[:14]
        setattr(view, "target_points_cache", out)
        return out

    def _movement_bonus(self, view: FrameView, candidate: dict[str, Any]) -> float:
        if self.game_dynamics is None:
            return 0.0
        if str(candidate.get("kind", "")) != "simple":
            return 0.0
        aid = int(candidate.get("action_id", -1))
        delta, conf = self.game_dynamics.primary_delta(aid)
        if delta is None or conf <= 0.0:
            return 0.0
        avatar_seg = self.game_dynamics.locate_avatar(view)
        if avatar_seg is None:
            return 0.0

        ax, ay = self.game_dynamics.segment_center(avatar_seg)
        dx, dy = delta
        nx = ax + float(dx)
        ny = ay + float(dy)

        targets = self._get_target_points(view, avatar_seg=avatar_seg)
        if not targets:
            return 0.04 * float(conf)

        best_gain = -1.0
        for tx, ty, weight, mode in targets:
            before = abs(ax - float(tx)) + abs(ay - float(ty))
            after = abs(nx - float(tx)) + abs(ny - float(ty))
            gain = (before - after) * float(weight)
            if mode == "adjacent" and after <= 1.0:
                gain += 0.35 * float(weight)
            best_gain = max(best_gain, gain)

        action_change = self.game_dynamics.action_change_rate(aid)
        bonus = 0.09 * float(conf) + 0.07 * float(action_change) + 0.11 * max(0.0, best_gain)
        return float(max(-0.18, min(0.30, bonus)))

    def _interaction_bonus(self, view: FrameView, candidate: dict[str, Any]) -> float:
        if self.game_dynamics is None:
            return 0.0
        if str(candidate.get("kind", "")) != "simple":
            return 0.0
        aid = int(candidate.get("action_id", -1))
        propensity = self.game_dynamics.interaction_propensity(aid)
        if propensity <= 0.55:
            return 0.0
        avatar_seg = self.game_dynamics.locate_avatar(view)
        if avatar_seg is None:
            return 0.02 * propensity if aid == 5 else 0.0

        ax, ay = self.game_dynamics.segment_center(avatar_seg)
        targets = self._get_target_points(view, avatar_seg=avatar_seg)
        if not targets:
            return 0.04 * propensity if aid == 5 else 0.0

        nearest = min(abs(ax - float(tx)) + abs(ay - float(ty)) for tx, ty, _, _ in targets)
        bonus = 0.0
        if nearest <= 1.0:
            bonus += 0.18 * propensity
        elif nearest <= 2.0:
            bonus += 0.09 * propensity
        elif nearest <= 3.0:
            bonus += 0.04 * propensity
        if aid == 5:
            bonus += 0.03 * propensity
        return float(min(0.22, bonus))

    def _click_alignment_bonus(self, view: FrameView, candidate: dict[str, Any]) -> float:
        if str(candidate.get("kind", "")) != "click":
            return 0.0
        feats = candidate.get("features") or {}
        bonus = 0.0
        if int(feats.get("target_like", 0)):
            bonus += 0.12
        if int(feats.get("rare_color", 0)):
            bonus += 0.05
        if int(feats.get("salient_color", 0)):
            bonus += 0.04
        if int(feats.get("singleton_color", 0)):
            bonus += 0.03
        source = str(feats.get("source", ""))
        if source in {"neighbor", "boundary", "outside"}:
            bonus += 0.06
        elif source == "model_top":
            bonus += 0.03 + 0.02 * int(feats.get("model_hot", 0))

        data = candidate.get("data") or {}
        x = int(data.get("x", 0))
        y = int(data.get("y", 0))
        for tx, ty, weight, mode in self._get_target_points(view):
            if x == int(tx) and y == int(ty):
                bonus += 0.07 * float(weight)
                if mode == "adjacent":
                    bonus += 0.03
                break
        return float(min(0.24, bonus))

    def candidate_priority(self, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        base = super().candidate_priority(candidate, path_mode=path_mode, distance=distance)
        if candidate.get("kind") == "undo":
            return base

        game_est = self.game_memory.estimate(candidate)
        game_exp = min(16.0, self.game_memory.exposure(candidate))
        base += 0.24 * (float(game_est) - 0.5)
        if not path_mode:
            base += 0.04 / math.sqrt(1.0 + float(game_exp))

        feats = candidate.get("features") or {}
        kind = str(candidate.get("kind", ""))
        aid = int(candidate.get("action_id", -1))
        if kind == "click":
            if int(feats.get("target_like", 0)):
                base += 0.08
            if int(feats.get("rare_color", 0)):
                base += 0.05
            if int(feats.get("salient_color", 0)):
                base += 0.04
            source = str(feats.get("source", ""))
            if source in {"neighbor", "boundary", "outside"}:
                base += 0.05
            elif source == "model_top":
                base += 0.04 + 0.02 * int(feats.get("model_hot", 0))
        elif kind == "simple":
            if self.game_dynamics is not None and aid in self.game_dynamics.recent_success_actions:
                base += 0.03
            if aid == 5 and self.game_dynamics is not None:
                base += 0.05 * max(0.0, self.game_dynamics.interaction_propensity(aid) - 0.5)

        return float(base)

    def start_level(self, obs: Any, env: Any) -> FrameView:
        if self.game_dynamics is not None:
            self.game_dynamics.start_new_level()
        view = super().start_level(obs, env)
        self.start_node_simple_untried = {
            idx for idx, cand in enumerate(view.candidates)
            if cand.get("kind") == "simple" and int(cand.get("action_id", -1)) != 7
        }
        preferred_clicks = [
            idx for idx, cand in enumerate(view.candidates)
            if cand.get("kind") == "click"
            and (
                int((cand.get("features") or {}).get("target_like", 0)) == 1
                or int(cand.get("group", self.n_groups - 1)) <= 1
                or str((cand.get("features") or {}).get("source", "")) == "model_top"
            )
        ]
        self.start_node_click_untried = preferred_clicks[:12]
        return view

    def observe_transition(
        self,
        prev_view: FrameView,
        candidate_idx: int,
        next_obs: Any,
        env: Any,
        game_over: bool = False,
        level_up: bool = False,
    ) -> Optional[FrameView]:
        candidate = prev_view.candidates[candidate_idx]
        next_view = super().observe_transition(
            prev_view,
            candidate_idx,
            next_obs,
            env,
            game_over=game_over,
            level_up=level_up,
        )
        if self.game_dynamics is not None:
            try:
                self.game_dynamics.observe(
                    prev_view,
                    candidate,
                    next_view,
                    game_over=game_over,
                    level_up=level_up,
                )
            except Exception:
                pass
        return next_view

    def hybrid_score(self, view: FrameView, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        base = self.candidate_priority(candidate, path_mode=path_mode, distance=distance)
        base_prob = 1.0 / (1.0 + math.exp(-1.35 * float(base)))

        replay_factor = 0.0
        if self.min_train > 0:
            replay_factor = max(0.0, min(1.0, (len(self.replay) - 0.30 * self.min_train) / (2.4 * self.min_train)))
        model_prob = self._candidate_model_score(view, candidate)
        model_prob = 0.5 + replay_factor * (float(model_prob) - 0.5)

        local_prior = self.level_memory.estimate(candidate)
        game_prior = self.game_memory.estimate(candidate)
        global_prior = self.global_memory.estimate(candidate)
        prior = 0.50 * local_prior + 0.28 * game_prior + 0.22 * global_prior

        node_failures = self.node_candidate_failures.get(self._node_candidate_key(view, int(candidate.get("candidate_idx", -1))), 0)
        fail_penalty = 0.055 * min(4, int(node_failures))

        kind = str(candidate.get("kind", ""))
        aid = int(candidate.get("action_id", -1))
        warmup_bonus = 0.0
        if not path_mode and view.node_id == self.start_node:
            if kind == "simple" and aid != 7 and self.steps_in_level <= max(5, len(view.available_action_ids) + 1):
                warmup_bonus += 0.08
            elif kind == "click" and self.steps_in_level <= 10:
                feats = candidate.get("features") or {}
                if int(feats.get("target_like", 0)):
                    warmup_bonus += 0.05
                elif int(candidate.get("group", self.n_groups - 1)) <= 1:
                    warmup_bonus += 0.03

        structure_bonus = 0.0
        if kind == "simple":
            structure_bonus += self._movement_bonus(view, candidate)
            structure_bonus += self._interaction_bonus(view, candidate)
        elif kind == "click":
            structure_bonus += self._click_alignment_bonus(view, candidate)

        if path_mode:
            score = 0.50 * base_prob + 0.10 * model_prob + 0.40 * prior + 0.45 * structure_bonus - 0.018 * float(distance) - fail_penalty
        else:
            score = 0.40 * base_prob + 0.16 * model_prob + 0.30 * prior + warmup_bonus + structure_bonus - fail_penalty

        score += 1e-6 * self.py_rng.random()
        return float(score)


    def choose_candidate(self, view: FrameView) -> Optional[dict[str, Any]]:
        if not view.candidates:
            return None

        self._get_model_predictions(view)

        undo_idx = self._undo_candidate_idx(view)
        undo_status = self.graph.edge_status(view.node_id, undo_idx) if undo_idx is not None else -1
        can_try_undo = (
            undo_idx is not None
            and bool(view.undo_available)
            and bool(self.undo_stack)
            and undo_status != -1
        )

        current_dist = self.graph.distance(view.node_id)
        unreachable = (
            view.node_id != self.start_node
            and view.node_id not in self.graph.frontier
            and current_dist >= INFINITY
        )

        level_idx = max(1, int(view.levels_completed) + 1)
        max_resets = 9 if level_idx <= 3 else 11
        new_node_limit = 16 if level_idx <= 2 else (22 if level_idx <= 4 else 30)
        hash_change_limit = 10 if level_idx <= 2 else (14 if level_idx <= 4 else 20)
        stagnated = self.steps_since_new_node >= new_node_limit or self.steps_since_hash_change >= hash_change_limit

        if unreachable:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to recover reachable frontier from {view.node_id}"
                return out
            if view.node_id != self.start_node and self.resets_in_level < max_resets:
                return {"kind": "meta_reset", "reasoning": f"reset to recover frontier from {view.node_id}"}
            return None

        if stagnated and view.node_id != self.start_node:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to escape stagnation from {view.node_id}"
                return out
            if self.resets_in_level < max_resets:
                return {"kind": "meta_reset", "reasoning": f"reset after stagnation at {view.node_id}"}

        if view.node_id == self.start_node and not self.graph.frontier:
            return None

        open_ids = self.graph.open_candidate_ids(view.node_id)
        open_non_undo = [idx for idx in open_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if view.node_id == self.start_node and open_non_undo:
            simple_candidates = [idx for idx in open_non_undo if idx in self.start_node_simple_untried]
            if simple_candidates and self.steps_in_level <= max(5, len(view.available_action_ids) + 1):
                edge_idx = max(
                    simple_candidates,
                    key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                )
                self.start_node_simple_untried.discard(edge_idx)
                cand = view.candidates[edge_idx]
                out = dict(cand)
                out["candidate_idx"] = int(edge_idx)
                return out
            if self.start_node_click_untried and self.steps_in_level <= 10:
                click_candidates = [idx for idx in self.start_node_click_untried if idx in open_non_undo]
                if click_candidates:
                    edge_idx = max(
                        click_candidates,
                        key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                    )
                    self.start_node_click_untried = [idx for idx in self.start_node_click_untried if idx != edge_idx]
                    cand = view.candidates[edge_idx]
                    out = dict(cand)
                    out["candidate_idx"] = int(edge_idx)
                    return out

        if open_non_undo:
            successful_ids = set(self.graph.successful_candidate_ids(view.node_id))
            model_ready = len(self.replay) >= max(12, self.min_train // 2)

            fresh_simple = [
                idx for idx in open_non_undo
                if str(view.candidates[idx].get("kind", "")) == "simple"
                and idx not in successful_ids
            ]
            if fresh_simple and self.steps_since_hash_change <= 2 and self.steps_in_level > 0:
                edge_idx = max(
                    fresh_simple,
                    key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                )
                cand = view.candidates[edge_idx]
                out = dict(cand)
                out["candidate_idx"] = int(edge_idx)
                out["reasoning"] = "fresh_simple_probe"
                return out

            guided_simple = []
            guided_click = []
            for idx in open_non_undo:
                cand = view.candidates[idx]
                kind = str(cand.get("kind", ""))
                score_candidate = {**cand, "candidate_idx": int(idx)}
                model_prob = self._candidate_model_score(view, score_candidate)
                if kind == "simple":
                    guide = self._movement_bonus(view, score_candidate) + self._interaction_bonus(view, score_candidate)
                    if guide >= 0.08 or (model_ready and model_prob >= 0.68):
                        guided_simple.append(idx)
                elif kind == "click":
                    feats = cand.get("features") or {}
                    source = str(feats.get("source", ""))
                    guide = self._click_alignment_bonus(view, score_candidate)
                    if (
                        guide >= 0.08
                        or int(feats.get("target_like", 0))
                        or (model_ready and source == "model_top" and model_prob >= 0.60)
                    ):
                        guided_click.append(idx)

            if guided_simple:
                edge_idx = max(
                    guided_simple,
                    key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                )
                cand = view.candidates[edge_idx]
                out = dict(cand)
                out["candidate_idx"] = int(edge_idx)
                out["reasoning"] = "guided_simple"
                return out

            if guided_click and (self.steps_in_level <= 16 or not any(view.candidates[idx].get("kind") == "simple" for idx in open_non_undo)):
                edge_idx = max(
                    guided_click,
                    key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                )
                cand = view.candidates[edge_idx]
                out = dict(cand)
                out["candidate_idx"] = int(edge_idx)
                out["reasoning"] = "guided_click"
                return out

            edge_idx = max(
                open_non_undo,
                key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if can_try_undo:
            out = dict(view.candidates[undo_idx])
            out["candidate_idx"] = int(undo_idx)
            out["reasoning"] = f"undo to parent/frontier from {view.node_id}"
            return out

        path_ids = self.graph.successful_candidate_ids(view.node_id)
        path_non_undo = [idx for idx in path_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if path_non_undo:
            edge_idx = max(
                path_non_undo,
                key=lambda idx: self.hybrid_score(
                    view,
                    {**view.candidates[idx], "candidate_idx": int(idx)},
                    path_mode=True,
                    distance=self.graph.edge_distance(view.node_id, idx),
                ),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if view.node_id != self.start_node and self.resets_in_level < max_resets:
            return {"kind": "meta_reset", "reasoning": f"reset because no reachable frontier from {view.node_id}"}
        return None



@dataclass
class LocalEnvInfo:
    game_id: str
    class_name: str
    local_dir: str
    baseline_actions: list[int] = field(default_factory=list)
    title: str = ""
    default_fps: int = 5


@dataclass(frozen=True)
class OfflineCandidate:
    action_id: int
    x: int = -1
    y: int = -1
    kind: str = "simple"
    source: str = "simple"
    priority: float = 0.0

    def data(self) -> Optional[dict[str, int]]:
        if self.action_id == 6 and self.x >= 0 and self.y >= 0:
            return {"x": int(self.x), "y": int(self.y)}
        return None

    def token(self) -> tuple[int, int, int]:
        return (int(self.action_id), int(self.x), int(self.y))


@dataclass
class OfflineNode:
    path: tuple[tuple[int, int, int], ...]
    game: Any
    obs: Any
    key: Any
    frame_hash: str
    raw_frame: np.ndarray
    state: str
    levels_completed: int
    last_token: tuple[int, int, int] = (0, -1, -1)
    rank: tuple[Any, ...] = field(default_factory=tuple)


FP = FrameProcessor()
INVERSE_ACTION = {1: 2, 2: 1, 3: 4, 4: 3}


def segment_priority_key(fp: FrameProcessor, seg: dict[str, Any]) -> tuple[Any, ...]:
    group = fp.segment_group(seg)
    color = int(seg["color"])
    salient = 0 if color in fp.salient_colors else 1
    status = 1 if color == fp.status_bar_color else 0
    area = int(seg["area"])
    x1, y1, x2, y2 = seg["bounding_box"]
    width = x2 - x1 + 1
    height = y2 - y1 + 1
    mediumish = 0 if (fp.minimal_width <= width <= fp.maximal_width and fp.minimal_width <= height <= fp.maximal_width) else 1
    return (group, status, salient, mediumish, -min(area, 256), y1, x1)


KEYWORD_PRIORITIES = (
    (0, ("player", "agent", "avatar", "cursor", "hero", "character")),
    (1, ("goal", "target", "exit", "door", "key", "button", "switch", "portal", "flag", "star")),
    (2, ("box", "crate", "block", "ball", "piece", "item", "orb", "gem", "food", "enemy", "npc")),
    (3, ("sprite", "entity", "object", "tile", "cell", "point", "coord", "position", "loc")),
    (4, ("level",)),
)

STATE_SKIP_NAMES = {
    "logger",
    "_logger",
    "renderer",
    "screen",
    "surface",
    "image",
    "frames",
    "frame",
    "_last_response",
    "_recording_filename",
    "scorecard_manager",
    "recording",
    "sprite_sheet",
    "font",
    "text_cache",
    "window",
    "display",
}

OBJECT_POINT_KEYS = (
    ("x", "y"),
    ("col", "row"),
    ("cx", "cy"),
    ("tile_x", "tile_y"),
    ("px", "py"),
    ("left", "top"),
    ("right", "bottom"),
)

POSITION_ATTRS = (
    "position",
    "pos",
    "location",
    "loc",
    "coord",
    "coords",
    "xy",
    "center",
    "point",
    "points",
    "bounding_box",
    "bbox",
    "rect",
)


def score_keyword_priority(path: str) -> int:
    lower = str(path).lower()
    for score, words in KEYWORD_PRIORITIES:
        if any(word in lower for word in words):
            return score
    return 5


def scan_local_environment_infos(root: Path) -> dict[str, LocalEnvInfo]:
    infos: dict[str, LocalEnvInfo] = {}
    if not root.exists():
        return infos
    for meta_path in sorted(root.rglob("metadata.json")):
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(meta, dict):
            continue
        gid = str(meta.get("game_id") or meta.get("id") or meta_path.parent.name)
        class_name = str(meta.get("class_name") or gid[:4].capitalize())
        baseline_actions = []
        for x in meta.get("baseline_actions") or []:
            try:
                baseline_actions.append(int(x))
            except Exception:
                continue
        title = str(meta.get("title") or gid)
        try:
            default_fps = int(meta.get("default_fps") or 5)
        except Exception:
            default_fps = 5
        infos[gid] = LocalEnvInfo(
            game_id=gid,
            class_name=class_name,
            local_dir=str(meta_path.parent),
            baseline_actions=baseline_actions,
            title=title,
            default_fps=default_fps,
        )
    return infos


def build_prefix_index(local_infos: dict[str, LocalEnvInfo]) -> dict[str, list[LocalEnvInfo]]:
    out: dict[str, list[LocalEnvInfo]] = defaultdict(list)
    for info in local_infos.values():
        out[info.game_id[:4]].append(info)
    return out


def resolve_local_env_info(
    game_id: str,
    local_infos: dict[str, LocalEnvInfo],
    prefix_index: dict[str, list[LocalEnvInfo]],
) -> Optional[LocalEnvInfo]:
    if game_id in local_infos:
        return local_infos[game_id]
    matches = prefix_index.get(game_id[:4], [])
    if len(matches) == 1:
        return matches[0]
    for info in matches:
        if info.game_id.startswith(game_id) or game_id.startswith(info.game_id[:4]):
            return info
    return None


def token_to_data(token: tuple[int, int, int]) -> Optional[dict[str, int]]:
    aid, x, y = token
    if int(aid) == 6 and int(x) >= 0 and int(y) >= 0:
        return {"x": int(x), "y": int(y)}
    return None


def make_action_input(action_token: Any, data: Optional[dict[str, int]] = None, reasoning: Optional[str] = None) -> Any:
    payload = data or {}
    attempts = [
        {"id": action_token, "data": payload, "reasoning": reasoning},
        {"id": action_token, "data": payload},
        {"id": action_token},
        {"action": action_token, "data": payload, "reasoning": reasoning},
        {"action": action_token, "data": payload},
        {"action": action_token},
    ]
    last_exc = None
    for kwargs in attempts:
        try:
            clean_kwargs = {k: v for k, v in kwargs.items() if v is not None}
            return ActionInput(**clean_kwargs)
        except TypeError as exc:
            last_exc = exc
        except Exception:
            raise
    raise last_exc or RuntimeError("Could not construct ActionInput")


def perform_action_game(
    game: Any,
    action_id: int,
    data: Optional[dict[str, int]] = None,
    reasoning: Optional[str] = None,
) -> Any:
    action_enum = RESET_ACTION if int(action_id) == 0 else ACTION_ID_TO_ENUM.get(int(action_id), int(action_id))
    tokens = [action_enum]
    name = getattr(action_enum, "name", None)
    if name:
        tokens.append(name)
    value = getattr(action_enum, "value", None)
    if value is not None:
        tokens.append(value)
    tokens.append(int(action_id))
    tokens = unique_preserve_order(tokens)

    last_exc = None
    methods = []
    if hasattr(game, "perform_action"):
        methods.append(game.perform_action)
    if hasattr(game, "step"):
        methods.append(game.step)

    for token in tokens:
        try:
            action_input = make_action_input(token, data=data, reasoning=reasoning)
        except TypeError as exc:
            last_exc = exc
            continue
        for method in methods:
            for kwargs in ({"raw": True}, {}, {"data": data or {}, "reasoning": reasoning}):
                try:
                    clean_kwargs = {k: v for k, v in kwargs.items() if v is not None}
                    out = method(action_input, **clean_kwargs)
                    return unwrap_observation(out)
                except TypeError as exc:
                    last_exc = exc
                except Exception:
                    raise
            try:
                out = method(token)
                return unwrap_observation(out)
            except TypeError as exc:
                last_exc = exc
            except Exception:
                raise
    raise last_exc or RuntimeError(f"Could not step local game with action {action_id}")


def try_clone_game(game: Any) -> Any:
    if game is None:
        return None
    try:
        return copy.deepcopy(game)
    except Exception:
        pass
    try:
        return pickle.loads(pickle.dumps(game, protocol=pickle.HIGHEST_PROTOCOL))
    except Exception:
        return None


def shallow_hash_bytes(data: bytes, salt: str = "") -> str:
    return hashlib.blake2b(data, digest_size=16, person=salt.encode()[:16] if salt else b"").hexdigest()


def array_digest(arr: np.ndarray) -> str:
    arr = np.asarray(arr)
    return hashlib.blake2b(arr.tobytes(), digest_size=16, person=repr(arr.shape).encode()[:16]).hexdigest()


def serialize_hidden_state(obj: Any, depth: int = 4, max_items: int = 18) -> Any:
    if depth < 0:
        return None
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        arr = np.asarray(obj)
        if arr.size <= 128 and arr.ndim <= 2:
            try:
                return {"__ndarray__": arr.astype(int).tolist(), "shape": list(arr.shape)}
            except Exception:
                return {"__ndhash__": array_digest(arr), "shape": list(arr.shape)}
        return {"__ndhash__": array_digest(arr), "shape": list(arr.shape)}
    if isinstance(obj, dict):
        out: dict[str, Any] = {}
        items = sorted(obj.items(), key=lambda kv: str(kv[0]))[:max_items]
        for key, value in items:
            key_text = str(key)
            if key_text in STATE_SKIP_NAMES:
                continue
            try:
                out[key_text] = serialize_hidden_state(value, depth - 1, max_items)
            except Exception:
                out[key_text] = repr(value)
        return out
    if isinstance(obj, (list, tuple, set, frozenset)):
        seq = list(obj)[:max_items]
        return [serialize_hidden_state(v, depth - 1, max_items) for v in seq]
    if is_dataclass(obj):
        try:
            return serialize_hidden_state(asdict(obj), depth - 1, max_items)
        except Exception:
            pass
    if hasattr(obj, "model_dump"):
        try:
            return serialize_hidden_state(obj.model_dump(), depth - 1, max_items)
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        out: dict[str, Any] = {"__class__": obj.__class__.__name__}
        try:
            attrs = sorted(vars(obj).items(), key=lambda kv: score_keyword_priority(kv[0]))[:max_items]
        except Exception:
            attrs = []
        for key, value in attrs:
            if key in STATE_SKIP_NAMES:
                continue
            if callable(value) or inspect.ismodule(value) or inspect.isclass(value):
                continue
            try:
                out[key] = serialize_hidden_state(value, depth - 1, max_items)
            except Exception:
                out[key] = repr(value)
        return out
    return repr(obj)


def game_hidden_digest(game: Any) -> str:
    if game is None:
        return ""
    try:
        dumped = pickle.dumps(game, protocol=pickle.HIGHEST_PROTOCOL)
        if len(dumped) <= 200_000:
            return shallow_hash_bytes(dumped, salt=game.__class__.__name__)
    except Exception:
        pass
    try:
        rep = serialize_hidden_state(game, depth=4, max_items=18)
        dumped = json.dumps(rep, sort_keys=True, ensure_ascii=False, default=repr).encode("utf-8")
        return shallow_hash_bytes(dumped, salt=game.__class__.__name__)
    except Exception:
        return repr(type(game))


def build_offline_key(game: Any, obs: Any, fp: FrameProcessor) -> tuple[Any, str, np.ndarray]:
    raw_frame, _ = extract_latest_frame(obs)
    frame_hash = fp.hash_frame(raw_frame)
    key = (
        levels_completed(obs),
        state_name(obs),
        frame_hash,
        game_hidden_digest(game),
    )
    return key, frame_hash, raw_frame


def offline_node_from(game: Any, obs: Any, path: tuple[tuple[int, int, int], ...], last_token: tuple[int, int, int]) -> OfflineNode:
    key, frame_hash, raw_frame = build_offline_key(game, obs, FP)
    return OfflineNode(
        path=path,
        game=game,
        obs=obs,
        key=key,
        frame_hash=frame_hash,
        raw_frame=raw_frame,
        state=state_name(obs),
        levels_completed=levels_completed(obs),
        last_token=last_token,
    )


def ast_int(node: Any) -> Optional[int]:
    if isinstance(node, ast.Constant) and isinstance(node.value, int):
        return int(node.value)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        inner = ast_int(node.operand)
        if inner is not None:
            return -int(inner)
    return None


def parse_literal_points(source_text: str) -> list[tuple[int, ...]]:
    out: list[tuple[int, ...]] = []
    try:
        tree = ast.parse(source_text)
    except Exception:
        return out

    for node in ast.walk(tree):
        if isinstance(node, (ast.Tuple, ast.List)):
            vals = [ast_int(e) for e in node.elts]
            if len(vals) in (2, 4) and all(v is not None for v in vals):
                out.append(tuple(int(v) for v in vals))
        elif isinstance(node, ast.Dict):
            raw: dict[str, int] = {}
            for k, v in zip(node.keys, node.values):
                if isinstance(k, ast.Constant) and isinstance(k.value, str):
                    iv = ast_int(v)
                    if iv is not None:
                        raw[str(k.value).lower()] = int(iv)
            for kx, ky in OBJECT_POINT_KEYS:
                if kx in raw and ky in raw:
                    out.append((int(raw[kx]), int(raw[ky])))
        elif isinstance(node, ast.Call):
            raw_kw: dict[str, int] = {}
            for kw in node.keywords or []:
                if kw.arg is None:
                    continue
                iv = ast_int(kw.value)
                if iv is not None:
                    raw_kw[str(kw.arg).lower()] = int(iv)
            for kx, ky in OBJECT_POINT_KEYS:
                if kx in raw_kw and ky in raw_kw:
                    out.append((int(raw_kw[kx]), int(raw_kw[ky])))
    dedup: list[tuple[int, ...]] = []
    seen = set()
    for item in out:
        if item not in seen:
            seen.add(item)
            dedup.append(item)
    return dedup


@dataclass
class GameSourceFactory:
    env_info: LocalEnvInfo
    source_path: Path = field(init=False)
    source_text: str = field(init=False)
    module_name: str = field(init=False)
    module: Any = field(init=False)
    cls: Any = field(init=False)
    init_sig: inspect.Signature = field(init=False)
    literal_specs: list[tuple[int, ...]] = field(default_factory=list, init=False)

    def __post_init__(self) -> None:
        local_dir = Path(self.env_info.local_dir)
        candidates = [
            local_dir / f"{self.env_info.class_name.lower()}.py",
            local_dir / f"{self.env_info.class_name}.py",
        ]
        source_path = next((p for p in candidates if p.exists()), None)
        if source_path is None:
            py_files = [p for p in local_dir.glob("*.py") if p.name != "__init__.py"]
            if len(py_files) == 1:
                source_path = py_files[0]
        if source_path is None:
            raise FileNotFoundError(f"Source file not found in {local_dir}")
        self.source_path = source_path
        self.source_text = self.source_path.read_text(encoding="utf-8")
        safe_name = re.sub(r"[^0-9a-zA-Z_]+", "_", self.env_info.game_id)
        self.module_name = f"_arc_hidden_{safe_name}_{hashlib.md5(str(self.source_path).encode()).hexdigest()[:8]}"
        module = types.ModuleType(self.module_name)
        module.__file__ = str(self.source_path)
        sys.modules[self.module_name] = module
        exec(compile(self.source_text, str(self.source_path), "exec"), module.__dict__)
        self.module = module
        if not hasattr(module, self.env_info.class_name):
            fallback_name = self.env_info.game_id[:4].capitalize()
            if hasattr(module, fallback_name):
                self.env_info.class_name = fallback_name
        self.cls = getattr(module, self.env_info.class_name)
        self.init_sig = inspect.signature(self.cls)
        self.literal_specs = parse_literal_points(self.source_text)

    def new_game(self, seed: int = 0) -> Any:
        kwargs: dict[str, Any] = {}
        if "seed" in self.init_sig.parameters:
            kwargs["seed"] = seed
        return self.cls(**kwargs)

    def reset(self, game: Any) -> Any:
        return perform_action_game(game, 0, None, None)

    def replay(self, full_prefix: tuple[tuple[int, int, int], ...]) -> tuple[Any, Any]:
        game = self.new_game(seed=SEED)
        obs = self.reset(game)
        for token in full_prefix:
            obs = perform_action_game(game, int(token[0]), token_to_data(token), None)
        return game, obs

    def literal_points(self, width: int, height: int) -> list[tuple[int, int]]:
        out: list[tuple[int, int]] = []
        seen: set[tuple[int, int]] = set()

        def add(x: int, y: int) -> None:
            pt = (int(x), int(y))
            if 0 <= pt[0] < width and 0 <= pt[1] < height and pt not in seen:
                seen.add(pt)
                out.append(pt)

        for spec in self.literal_specs:
            if len(spec) == 2:
                add(spec[0], spec[1])
            elif len(spec) == 4:
                x1, y1, x2, y2 = spec
                add(x1, y1)
                add(x2, y2)
                add(x1, y2)
                add(x2, y1)
                add((x1 + x2) // 2, (y1 + y2) // 2)
        return out


def map_point_to_display(game: Any, x: int, y: int, width: int, height: int) -> list[tuple[int, int]]:
    candidates: list[tuple[int, int]] = []

    def add(px: Any, py: Any) -> None:
        try:
            ix = int(round(float(px)))
            iy = int(round(float(py)))
        except Exception:
            return
        if 0 <= ix < width and 0 <= iy < height:
            candidates.append((ix, iy))

    add(x, y)
    add(y, x)

    camera = maybe_get(game, "camera")
    if camera is None:
        return unique_preserve_order(candidates)

    methods = [
        "grid_to_display",
        "grid_to_display_coords",
        "grid_to_display_position",
        "to_display",
        "grid_to_screen",
    ]
    for method_name in methods:
        method = getattr(camera, method_name, None)
        if method is None or not callable(method):
            continue
        patterns = [
            ((x, y),),
            ((y, x),),
            (x, y),
            (y, x),
        ]
        for args in patterns:
            try:
                res = method(*args)
            except Exception:
                continue
            if isinstance(res, (tuple, list)) and len(res) >= 2:
                add(res[0], res[1])
            else:
                rx = maybe_get(res, "x", "col", "px")
                ry = maybe_get(res, "y", "row", "py")
                if rx is not None and ry is not None:
                    add(rx, ry)
    return unique_preserve_order(candidates)


def points_from_value(value: Any, path: str, limit: int = 8) -> list[tuple[int, int, int]]:
    results: list[tuple[int, int, int]] = []

    def add_pair(a: Any, b: Any, bias: int = 0) -> None:
        try:
            ia = int(a)
            ib = int(b)
        except Exception:
            return
        results.append((ia, ib, bias))
        if ia != ib:
            results.append((ib, ia, bias + 1))

    if value is None:
        return results
    if isinstance(value, np.ndarray):
        arr = np.asarray(value)
        if arr.ndim == 1 and arr.size == 2:
            add_pair(arr[0], arr[1], 0)
        elif arr.ndim == 2 and arr.shape[1] == 2:
            for row in arr[:limit]:
                add_pair(row[0], row[1], 0)
        return results
    if isinstance(value, (tuple, list)):
        if len(value) == 2:
            add_pair(value[0], value[1], 0)
        elif len(value) == 4 and all(isinstance(v, (int, np.integer)) for v in value):
            x1, y1, x2, y2 = [int(v) for v in value]
            results.append(((x1 + x2) // 2, (y1 + y2) // 2, 0))
            results.append((x1, y1, 1))
            results.append((x2, y2, 1))
            results.append((x1, y2, 2))
            results.append((x2, y1, 2))
        elif value and all(isinstance(v, (tuple, list, np.ndarray)) for v in value[: min(len(value), limit)]):
            for item in value[:limit]:
                results.extend(points_from_value(item, path, limit=2))
        return results
    if isinstance(value, dict):
        lower = {str(k).lower(): v for k, v in value.items()}
        for kx, ky in OBJECT_POINT_KEYS:
            if kx in lower and ky in lower:
                add_pair(lower[kx], lower[ky], 0)
        if all(k in lower for k in ("x1", "y1", "x2", "y2")):
            try:
                x1, y1, x2, y2 = int(lower["x1"]), int(lower["y1"]), int(lower["x2"]), int(lower["y2"])
                results.append(((x1 + x2) // 2, (y1 + y2) // 2, 0))
            except Exception:
                pass
        for attr in POSITION_ATTRS:
            if attr in lower:
                results.extend(points_from_value(lower[attr], path + "." + attr, limit=limit))
        return results
    for attr in POSITION_ATTRS:
        if hasattr(value, attr):
            try:
                results.extend(points_from_value(getattr(value, attr), path + "." + attr, limit=limit))
            except Exception:
                continue
    lower = {attr.lower(): attr for attr in dir(value)[:64] if isinstance(attr, str)}
    for kx, ky in OBJECT_POINT_KEYS:
        if kx in lower and ky in lower:
            try:
                add_pair(getattr(value, lower[kx]), getattr(value, lower[ky]), 0)
            except Exception:
                pass
    return results


def collect_game_points(game: Any, width: int, height: int, max_depth: int = 4, max_points: int = 32) -> list[tuple[int, int, int, str]]:
    out: dict[tuple[int, int], tuple[int, str]] = {}
    queue: deque[tuple[Any, int, str]] = deque([(game, 0, "root")])
    seen: set[int] = set()

    def add_point(x: int, y: int, priority: int, source: str) -> None:
        if 0 <= x < width and 0 <= y < height:
            cur = out.get((x, y))
            if cur is None or priority < cur[0]:
                out[(x, y)] = (priority, source)

    while queue and len(out) < max_points * 3:
        obj, depth, path = queue.popleft()
        if obj is None:
            continue
        try:
            oid = id(obj)
            if oid in seen:
                continue
            seen.add(oid)
        except Exception:
            pass

        base_priority = score_keyword_priority(path) * 10 + depth

        try:
            for px, py, bias in points_from_value(obj, path, limit=6):
                for mx, my in map_point_to_display(game, px, py, width, height):
                    add_point(mx, my, base_priority + bias, path)
        except Exception:
            pass

        if depth >= max_depth:
            continue

        try:
            if isinstance(obj, dict):
                items = sorted(obj.items(), key=lambda kv: (score_keyword_priority(str(kv[0])), str(kv[0])))[:16]
                for key, value in items:
                    queue.append((value, depth + 1, f"{path}.{key}"))
                continue
            if isinstance(obj, (list, tuple, set, frozenset)):
                for idx, value in enumerate(list(obj)[:12]):
                    queue.append((value, depth + 1, f"{path}[{idx}]"))
                continue
            attrs = []
            if hasattr(obj, "__dict__"):
                for key, value in vars(obj).items():
                    if key in STATE_SKIP_NAMES:
                        continue
                    if callable(value) or inspect.ismodule(value) or inspect.isclass(value):
                        continue
                    attrs.append((key, value))
                attrs = sorted(attrs, key=lambda kv: (score_keyword_priority(kv[0]), kv[0]))[:20]
                for key, value in attrs:
                    queue.append((value, depth + 1, f"{path}.{key}"))
        except Exception:
            continue

    ranked = [((x, y), pr, src) for (x, y), (pr, src) in out.items()]
    ranked.sort(key=lambda item: (item[1], abs(item[0][1] - height / 2.0) + abs(item[0][0] - width / 2.0)))
    final: list[tuple[int, int, int, str]] = []
    for (x, y), pr, src in ranked[:max_points]:
        final.append((x, y, pr, src))
    return final


@dataclass
class OfflineLevelContext:
    env_info: LocalEnvInfo
    factory: GameSourceFactory
    full_prefix_before_level: tuple[tuple[int, int, int], ...]
    root_game: Any
    root_obs: Any
    baseline_actions: int
    start_levels: int = field(init=False)
    root_frame: np.ndarray = field(init=False)
    root_frame_hash: str = field(init=False)
    root_key: Any = field(init=False)

    def __post_init__(self) -> None:
        self.start_levels = levels_completed(self.root_obs)
        self.root_frame, _ = extract_latest_frame(self.root_obs)
        self.root_frame_hash = FP.hash_frame(self.root_frame)
        self.root_key, _, _ = build_offline_key(self.root_game, self.root_obs, FP)

    def replay_path(self, relative_path: tuple[tuple[int, int, int], ...]) -> tuple[Any, Any]:
        return self.factory.replay(self.full_prefix_before_level + relative_path)


def extract_frame_objects_for_transfer(frame: np.ndarray) -> list[dict[str, Any]]:
    arr = np.asarray(frame, dtype=np.uint8)
    if arr.ndim != 2 or arr.size == 0:
        return []
    height, width = arr.shape
    bg = int(np.bincount(arr.flatten(), minlength=16).argmax())
    labels, segments = FP.segment_frame(arr)
    status_mask = FP.identify_status_bars(labels, segments)
    objects: list[dict[str, Any]] = []
    for seg in segments:
        color = int(seg["color"])
        area = int(seg["area"])
        pts = np.asarray(list(seg["points"]), dtype=int)
        if pts.size == 0:
            continue
        py = pts[:, 0]
        px = pts[:, 1]
        if float(np.mean(status_mask[py, px])) > 0.5:
            continue
        if color == bg and area < max(4, (height * width) // 64):
            continue
        x1, y1, x2, y2 = [int(v) for v in seg["bounding_box"]]
        objects.append(
            {
                "color": color,
                "area": area,
                "cx": float(np.mean(px)),
                "cy": float(np.mean(py)),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "w": int(x2 - x1 + 1),
                "h": int(y2 - y1 + 1),
                "group": int(FP.segment_group(seg)),
            }
        )
    if not objects:
        for color in range(16):
            if color == bg:
                continue
            mask = arr == color
            area = int(np.count_nonzero(mask))
            if area < 2:
                continue
            ys, xs = np.where(mask)
            objects.append(
                {
                    "color": int(color),
                    "area": area,
                    "cx": float(np.mean(xs)),
                    "cy": float(np.mean(ys)),
                    "x1": int(xs.min()),
                    "y1": int(ys.min()),
                    "x2": int(xs.max()),
                    "y2": int(ys.max()),
                    "w": int(xs.max() - xs.min() + 1),
                    "h": int(ys.max() - ys.min() + 1),
                    "group": 0,
                }
            )
    objects.sort(key=lambda o: (o["color"], o["group"], -o["area"], o["y1"], o["x1"]))
    return objects[:32]


def estimate_click_transform(prev_frame: np.ndarray, curr_frame: np.ndarray) -> list[dict[str, Any]]:
    prev_arr = np.asarray(prev_frame, dtype=np.uint8)
    curr_arr = np.asarray(curr_frame, dtype=np.uint8)
    h0, w0 = prev_arr.shape
    h1, w1 = curr_arr.shape
    prev_objects = extract_frame_objects_for_transfer(prev_arr)
    curr_objects = extract_frame_objects_for_transfer(curr_arr)

    specs: list[dict[str, Any]] = []

    def add_spec(name: str, ax: float, bx: float, ay: float, by: float) -> None:
        ax = float(ax)
        bx = float(bx)
        ay = float(ay)
        by = float(by)
        if not np.isfinite(ax) or not np.isfinite(bx) or not np.isfinite(ay) or not np.isfinite(by):
            return
        key = (round(ax, 4), round(bx, 4), round(ay, 4), round(by, 4))
        if any(s.get("_key") == key for s in specs):
            return
        specs.append(
            {
                "name": name,
                "mode": "affine",
                "ax": ax,
                "bx": bx,
                "ay": ay,
                "by": by,
                "_key": key,
            }
        )

    add_spec("identity", 1.0, 0.0, 1.0, 0.0)
    add_spec(
        "resize",
        float((w1 - 1) / max(1, w0 - 1)),
        0.0,
        float((h1 - 1) / max(1, h0 - 1)),
        0.0,
    )
    add_spec(
        "resize_mirror_x",
        float(-(w1 - 1) / max(1, w0 - 1)),
        float(w1 - 1),
        float((h1 - 1) / max(1, h0 - 1)),
        0.0,
    )
    add_spec(
        "resize_mirror_y",
        float((w1 - 1) / max(1, w0 - 1)),
        0.0,
        float(-(h1 - 1) / max(1, h0 - 1)),
        float(h1 - 1),
    )
    add_spec(
        "resize_mirror_xy",
        float(-(w1 - 1) / max(1, w0 - 1)),
        float(w1 - 1),
        float(-(h1 - 1) / max(1, h0 - 1)),
        float(h1 - 1),
    )

    matched: list[tuple[dict[str, Any], dict[str, Any]]] = []
    used_curr: set[int] = set()
    for prev_obj in prev_objects:
        best_idx = None
        best_score = None
        for curr_idx, curr_obj in enumerate(curr_objects):
            if curr_idx in used_curr:
                continue
            if curr_obj["color"] != prev_obj["color"]:
                continue
            size_ratio = max(prev_obj["area"], curr_obj["area"]) / max(1.0, min(prev_obj["area"], curr_obj["area"]))
            if size_ratio > 3.5:
                continue
            shape_penalty = abs(prev_obj["w"] - curr_obj["w"]) + abs(prev_obj["h"] - curr_obj["h"])
            dist = abs(prev_obj["cx"] - curr_obj["cx"]) + abs(prev_obj["cy"] - curr_obj["cy"])
            score = (shape_penalty, dist, -min(prev_obj["area"], curr_obj["area"]))
            if best_score is None or score < best_score:
                best_score = score
                best_idx = curr_idx
        if best_idx is not None:
            used_curr.add(best_idx)
            matched.append((prev_obj, curr_objects[best_idx]))

    if matched:
        dx = float(np.median([curr["cx"] - prev["cx"] for prev, curr in matched]))
        dy = float(np.median([curr["cy"] - prev["cy"] for prev, curr in matched]))
        add_spec("translate", 1.0, dx, 1.0, dy)
        add_spec("mirror_x_translate", -1.0, float(w1 - 1) + dx, 1.0, dy)
        add_spec("mirror_y_translate", 1.0, dx, -1.0, float(h1 - 1) + dy)
        add_spec("mirror_xy_translate", -1.0, float(w1 - 1) + dx, -1.0, float(h1 - 1) + dy)

    if len(matched) >= 2:
        prev_xs = np.array([prev["cx"] for prev, _ in matched], dtype=float)
        prev_ys = np.array([prev["cy"] for prev, _ in matched], dtype=float)
        curr_xs = np.array([curr["cx"] for _, curr in matched], dtype=float)
        curr_ys = np.array([curr["cy"] for _, curr in matched], dtype=float)

        def affine_from_extents(src: np.ndarray, dst: np.ndarray, mirror: bool = False) -> Optional[tuple[float, float]]:
            s_min = float(np.min(src))
            s_max = float(np.max(src))
            d_min = float(np.min(dst))
            d_max = float(np.max(dst))
            if abs(s_max - s_min) < 1.0:
                return None
            if mirror:
                a = (d_min - d_max) / (s_max - s_min)
                b = d_max - a * s_min
            else:
                a = (d_max - d_min) / (s_max - s_min)
                b = d_min - a * s_min
            if not np.isfinite(a) or not np.isfinite(b):
                return None
            a = float(np.clip(a, -4.0, 4.0))
            return a, float(b)

        x_direct = affine_from_extents(prev_xs, curr_xs, mirror=False)
        y_direct = affine_from_extents(prev_ys, curr_ys, mirror=False)
        x_mirror = affine_from_extents(prev_xs, curr_xs, mirror=True)
        y_mirror = affine_from_extents(prev_ys, curr_ys, mirror=True)

        if x_direct and y_direct:
            add_spec("matched_affine", x_direct[0], x_direct[1], y_direct[0], y_direct[1])
        if x_mirror and y_direct:
            add_spec("matched_mirror_x", x_mirror[0], x_mirror[1], y_direct[0], y_direct[1])
        if x_direct and y_mirror:
            add_spec("matched_mirror_y", x_direct[0], x_direct[1], y_mirror[0], y_mirror[1])
        if x_mirror and y_mirror:
            add_spec("matched_mirror_xy", x_mirror[0], x_mirror[1], y_mirror[0], y_mirror[1])

    for spec in specs:
        spec.pop("_key", None)
    return specs


def apply_click_transform(
    token: tuple[int, int, int],
    spec: Optional[dict[str, Any]],
    width: int,
    height: int,
) -> tuple[int, int, int]:
    aid, x, y = [int(v) for v in token]
    if aid != 6 or x < 0 or y < 0 or not spec:
        return (aid, x, y)
    mode = str(spec.get("mode") or "affine")
    nx = float(x)
    ny = float(y)
    if mode == "affine":
        nx = float(spec.get("ax", 1.0)) * nx + float(spec.get("bx", 0.0))
        ny = float(spec.get("ay", 1.0)) * ny + float(spec.get("by", 0.0))
    elif mode == "translate":
        nx = nx + float(spec.get("dx", 0.0))
        ny = ny + float(spec.get("dy", 0.0))
    ix = int(round(nx))
    iy = int(round(ny))
    ix = max(0, min(width - 1, ix))
    iy = max(0, min(height - 1, iy))
    return (aid, ix, iy)


def try_transfer_path(
    ctx: OfflineLevelContext,
    path: tuple[tuple[int, int, int], ...],
    deadline: float,
    spec: Optional[dict[str, Any]] = None,
) -> Optional[list[tuple[int, int, int]]]:
    if not path or time.time() >= deadline:
        return None
    game = try_clone_game(ctx.root_game)
    if game is None:
        try:
            game, obs = ctx.replay_path(tuple())
        except Exception:
            return None
    else:
        obs = ctx.root_obs
    height, width = ctx.root_frame.shape
    out: list[tuple[int, int, int]] = []
    for token in path:
        if time.time() >= deadline:
            break
        transformed = apply_click_transform(token, spec, width, height)
        try:
            obs = perform_action_game(game, int(transformed[0]), token_to_data(transformed), None)
        except Exception:
            return None
        out.append(transformed)
        if state_name(obs) == "WIN" or levels_completed(obs) > ctx.start_levels:
            return out
        if state_name(obs) == "GAME_OVER":
            return None
    return None


def try_transfer_from_previous_level(
    ctx: OfflineLevelContext,
    previous_level_summary: Optional[dict[str, Any]],
    deadline: float,
) -> Optional[list[tuple[int, int, int]]]:
    if not previous_level_summary or time.time() >= deadline:
        return None
    path = tuple(tuple(int(v) for v in token) for token in previous_level_summary.get("path") or [])
    if not path:
        return None

    direct = try_transfer_path(ctx, path, deadline=deadline, spec=None)
    if direct:
        return direct

    if not any(int(token[0]) == 6 for token in path):
        return None

    prev_frame = previous_level_summary.get("root_frame")
    if prev_frame is None:
        return None

    for spec in estimate_click_transform(np.asarray(prev_frame, dtype=np.uint8), ctx.root_frame):
        if time.time() >= deadline:
            break
        if spec.get("name") == "identity":
            continue
        transferred = try_transfer_path(ctx, path, deadline=deadline, spec=spec)
        if transferred:
            return transferred
    return None


def probe_simple_actions(ctx: OfflineLevelContext) -> dict[int, dict[str, Any]]:
    avail = sorted(a for a in extract_available_action_ids(ctx.root_obs) if a != 6)
    out: dict[int, dict[str, Any]] = {}
    root = offline_node_from(ctx.root_game, ctx.root_obs, tuple(), (0, -1, -1))
    root_state = state_name(ctx.root_obs)
    for aid in avail:
        child = simulate_offline_child(
            ctx,
            root,
            OfflineCandidate(action_id=aid, kind="simple", source="probe", priority=0.0),
        )
        if child is None:
            continue
        changed = int(np.count_nonzero(child.raw_frame != ctx.root_frame))
        visible_same = (
            child.frame_hash == ctx.root_frame_hash
            and child.levels_completed == ctx.start_levels
            and child.state == root_state
        )
        hidden_only = child.key != ctx.root_key and visible_same
        effective = child.key != ctx.root_key or changed > 0 or child.levels_completed > ctx.start_levels or child.state == "WIN"
        out[aid] = {
            "changed": changed,
            "game_over": child.state == "GAME_OVER",
            "level_up": child.levels_completed > ctx.start_levels,
            "win": child.state == "WIN",
            "noop": child.key == ctx.root_key,
            "hidden_only": hidden_only,
            "effective": effective,
        }
    return out


def probe_click_actions(
    ctx: OfflineLevelContext,
    max_points: int = 48,
    max_keep: int = 18,
) -> list[dict[str, Any]]:
    avail = extract_available_action_ids(ctx.root_obs)
    if 6 not in avail:
        return []

    root = offline_node_from(ctx.root_game, ctx.root_obs, tuple(), (0, -1, -1))
    raw_frame = ctx.root_frame
    height, width = raw_frame.shape
    labels, segments = FP.segment_frame(raw_frame)
    status_mask = FP.identify_status_bars(labels, segments)
    bg = int(np.bincount(raw_frame.flatten(), minlength=16).argmax())
    point_scores: dict[tuple[int, int], tuple[float, str]] = {}

    def add_point(x: int, y: int, pr: float, source: str) -> None:
        if not (0 <= x < width and 0 <= y < height):
            return
        if status_mask[y, x]:
            return
        current = point_scores.get((x, y))
        if current is None or pr > current[0]:
            point_scores[(x, y)] = (float(pr), source)

    obj_points = collect_game_points(ctx.root_game, width, height, max_depth=5, max_points=40)
    for idx, (x, y, pri, src) in enumerate(obj_points):
        add_point(x, y, 430.0 - float(pri) - 1.5 * idx, f"obj:{src}")
        if idx < 12:
            for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                add_point(x + dx, y + dy, 390.0 - float(pri) - 1.5 * idx, f"obj-n:{src}")

    for idx, (x, y) in enumerate(ctx.factory.literal_points(width, height)[:28]):
        add_point(x, y, 370.0 - 2.5 * idx, "ast")

    ordered_segments = sorted(segments, key=lambda seg: segment_priority_key(FP, seg))
    for seg_idx, seg in enumerate(ordered_segments[:20]):
        color = int(seg["color"])
        base = 340.0 - 12.0 * FP.segment_group(seg) - 0.5 * seg_idx
        if color in FP.salient_colors:
            base += 12.0
        if color == bg:
            base -= 28.0
        reps = FP.representative_points(seg)
        for rep_idx, (x, y) in enumerate(reps[:6]):
            add_point(x, y, base - 4.0 * rep_idx, f"seg:{seg_idx}:{rep_idx}")
        x1, y1, x2, y2 = [int(v) for v in seg["bounding_box"]]
        add_point((x1 + x2) // 2, (y1 + y2) // 2, base - 2.0, f"segc:{seg_idx}")

    only_click = avail.issubset({6}) or avail.issubset({6, 7})
    invalid_mask = status_mask.astype(bool)
    grid_n = 6 if only_click else 5
    for idx, (x, y) in enumerate(FP.coarse_grid_points(raw_frame, invalid_mask, grid_n=grid_n)):
        pr = 230.0 - idx
        if int(raw_frame[y, x]) != bg:
            pr += 8.0
        add_point(x, y, pr, "grid")

    step = 1 if only_click and width * height <= 144 else 2 if width * height <= 400 else 3
    center_x = (width - 1) / 2.0
    center_y = (height - 1) / 2.0
    for y in range(0, height, step):
        for x in range(0, width, step):
            if invalid_mask[y, x]:
                continue
            color = int(raw_frame[y, x])
            pr = 170.0 - 0.6 * (abs(x - center_x) + abs(y - center_y))
            if color != bg:
                pr += 8.0
            if color in FP.salient_colors:
                pr += 10.0
            add_point(x, y, pr, "scan")

    ranked_points = sorted(
        point_scores.items(),
        key=lambda item: (
            item[1][0],
            -abs(item[0][0] - width / 2.0) - abs(item[0][1] - height / 2.0),
        ),
        reverse=True,
    )[:max_points]

    best_by_effect: dict[Any, dict[str, Any]] = {}
    root_state = state_name(ctx.root_obs)
    for (x, y), (base_pr, source) in ranked_points:
        cand = OfflineCandidate(action_id=6, x=int(x), y=int(y), kind="click", source=f"probe:{source}", priority=float(base_pr))
        child = simulate_offline_child(ctx, root, cand)
        if child is None:
            continue
        changed = int(np.count_nonzero(child.raw_frame != ctx.root_frame))
        visible_same = (
            child.frame_hash == ctx.root_frame_hash
            and child.levels_completed == ctx.start_levels
            and child.state == root_state
        )
        hidden_only = child.key != ctx.root_key and visible_same
        level_up = child.levels_completed > ctx.start_levels
        win = child.state == "WIN"
        effective = child.key != ctx.root_key or changed > 0 or level_up or win
        if not effective and child.state != "GAME_OVER":
            continue
        pr = float(base_pr)
        pr += float(min(80, changed))
        if hidden_only:
            pr += 90.0
        if effective:
            pr += 18.0
        if level_up or win:
            pr += 1000.0
        if child.state == "GAME_OVER":
            pr -= 220.0
        info = {
            "x": int(x),
            "y": int(y),
            "priority": float(pr),
            "source": str(source),
            "changed": changed,
            "game_over": child.state == "GAME_OVER",
            "level_up": level_up,
            "win": win,
            "hidden_only": hidden_only,
            "effective": effective,
            "key": child.key,
        }
        key = child.key
        prev = best_by_effect.get(key)
        if prev is None or info["priority"] > prev["priority"]:
            best_by_effect[key] = info

    out = list(best_by_effect.values())
    out.sort(
        key=lambda item: (
            int(item.get("win", False)),
            int(item.get("level_up", False)),
            float(item.get("priority", 0.0)),
        ),
        reverse=True,
    )
    if only_click:
        max_keep = max(max_keep, 24)
    return out[:max_keep]


def build_offline_candidates(
    ctx: OfflineLevelContext,
    node: OfflineNode,
    depth: int,
    simple_probe: Optional[dict[int, dict[str, Any]]] = None,
    click_probe: Optional[list[dict[str, Any]]] = None,
) -> list[OfflineCandidate]:
    raw_frame = node.raw_frame
    height, width = raw_frame.shape
    avail = extract_available_action_ids(node.obs)
    candidates: list[OfflineCandidate] = []

    for aid in sorted(a for a in avail if a != 6):
        pr = 300.0
        info = simple_probe.get(aid, {}) if simple_probe else {}
        if info.get("win") or info.get("level_up"):
            pr += 500.0
        pr += float(min(80, info.get("changed", 0)))
        if info.get("hidden_only"):
            pr += 36.0
        if info.get("effective"):
            pr += 14.0
        if info.get("noop"):
            pr -= 100.0
        if info.get("game_over"):
            pr -= 200.0
        if aid == 7:
            pr += 5.0
        candidates.append(OfflineCandidate(action_id=aid, kind="simple", source="simple", priority=pr))

    if 6 in avail:
        if click_probe:
            for idx, info in enumerate(click_probe):
                pr = float(info.get("priority", 0.0))
                if info.get("win") or info.get("level_up"):
                    pr += 500.0
                candidates.append(
                    OfflineCandidate(
                        action_id=6,
                        x=int(info["x"]),
                        y=int(info["y"]),
                        kind="click",
                        source=f"click-probe:{info.get('source', 'probe')}",
                        priority=pr - 0.1 * idx,
                    )
                )

        labels, segments = FP.segment_frame(raw_frame)
        status_mask = FP.identify_status_bars(labels, segments)
        point_scores: dict[tuple[int, int], tuple[float, str]] = {}

        def add_point(x: int, y: int, pr: float, source: str) -> None:
            if not (0 <= x < width and 0 <= y < height):
                return
            if status_mask[y, x]:
                return
            current = point_scores.get((x, y))
            if current is None or pr > current[0]:
                point_scores[(x, y)] = (float(pr), source)

        obj_points = collect_game_points(node.game, width, height, max_depth=4, max_points=28)
        for idx, (x, y, pri, src) in enumerate(obj_points):
            add_point(x, y, 360.0 - pri - 2.0 * idx, f"obj:{src}")
            if idx < 10:
                for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    add_point(x + dx, y + dy, 320.0 - pri - 2.0 * idx, f"obj-n:{src}")

        for idx, (x, y) in enumerate(ctx.factory.literal_points(width, height)[:20]):
            add_point(x, y, 300.0 - 3.0 * idx, "ast")

        ordered_segments = sorted(segments, key=lambda seg: segment_priority_key(FP, seg))
        for seg_idx, seg in enumerate(ordered_segments[:16]):
            base = 280.0 - 12.0 * FP.segment_group(seg) - 0.4 * seg_idx
            if int(seg["color"]) == FP.status_bar_color:
                base -= 90.0
            if int(seg["color"]) in FP.salient_colors:
                base += 12.0
            reps = FP.representative_points(seg)
            for rep_idx, (x, y) in enumerate(reps):
                add_point(x, y, base - 4.0 * rep_idx, f"seg:{seg_idx}:{rep_idx}")
            if int(seg["area"]) <= 4:
                for py, px in list(seg["points"])[:4]:
                    add_point(int(px), int(py), base - 10.0, f"segpt:{seg_idx}")

        only_click = avail.issubset({6}) or avail.issubset({6, 7})
        invalid_mask = status_mask.astype(bool)
        grid_n = 5 if only_click else 4
        for idx, (x, y) in enumerate(FP.coarse_grid_points(raw_frame, invalid_mask, grid_n=grid_n)):
            add_point(x, y, 170.0 - idx, "grid")
        if only_click and width * height <= 144:
            center_x = (width - 1) / 2.0
            center_y = (height - 1) / 2.0
            for y in range(height):
                for x in range(width):
                    if invalid_mask[y, x]:
                        continue
                    pr = 140.0 - 0.8 * (abs(x - center_x) + abs(y - center_y))
                    add_point(x, y, pr, "fullgrid")

        click_candidates: list[OfflineCandidate] = []
        for (x, y), (pr, source) in point_scores.items():
            color = int(raw_frame[y, x])
            if color in FP.salient_colors:
                pr += 10.0
            if color == 0:
                pr -= 6.0
            click_candidates.append(
                OfflineCandidate(action_id=6, x=x, y=y, kind="click", source=source, priority=pr)
            )

        click_candidates.sort(
            key=lambda c: (
                c.priority,
                -abs(c.x - width / 2.0) - abs(c.y - height / 2.0),
            ),
            reverse=True,
        )
        click_limit = 18 if depth == 0 else 14 if depth <= 2 else 10 if depth <= 4 else 8
        if only_click:
            click_limit = max(click_limit, 24 if depth == 0 else 14)
        candidates.extend(click_candidates[:click_limit])

    candidates.sort(key=lambda c: (c.priority, 1 if c.kind == "simple" else 0), reverse=True)
    dedup: list[OfflineCandidate] = []
    seen = set()
    for cand in candidates:
        key = (cand.action_id, cand.x, cand.y)
        if key in seen:
            continue
        seen.add(key)
        dedup.append(cand)
    return dedup


def simulate_offline_child(ctx: OfflineLevelContext, node: OfflineNode, cand: OfflineCandidate) -> Optional[OfflineNode]:
    child_path = node.path + (cand.token(),)
    child_game = try_clone_game(node.game)
    child_obs = None
    if child_game is not None:
        try:
            child_obs = perform_action_game(child_game, cand.action_id, cand.data(), None)
        except Exception:
            child_game = None
            child_obs = None
    if child_game is None or child_obs is None:
        try:
            child_game, child_obs = ctx.replay_path(child_path)
        except Exception:
            return None
    return offline_node_from(child_game, child_obs, child_path, cand.token())


def rank_offline_child(
    ctx: OfflineLevelContext,
    parent: OfflineNode,
    child: OfflineNode,
    cand: OfflineCandidate,
) -> tuple[Any, ...]:
    changed_parent = int(np.count_nonzero(child.raw_frame != parent.raw_frame))
    changed_root = int(np.count_nonzero(child.raw_frame != ctx.root_frame))
    same_visible = (
        child.frame_hash == parent.frame_hash
        and child.levels_completed == parent.levels_completed
        and child.state == parent.state
    )
    same_key = child.key == parent.key
    hidden_only = int(same_visible and not same_key)
    backtrack = int(cand.action_id != 6 and INVERSE_ACTION.get(int(parent.last_token[0])) == int(cand.action_id))
    same_click = int(
        cand.action_id == 6
        and int(parent.last_token[0]) == 6
        and int(parent.last_token[1]) == int(cand.x)
        and int(parent.last_token[2]) == int(cand.y)
    )
    return (
        int(child.state == "WIN" or child.levels_completed > ctx.start_levels),
        int(child.state != "GAME_OVER"),
        int(not same_key),
        hidden_only,
        int(not same_visible),
        min(changed_parent, 255),
        min(changed_root, 255),
        int(round(cand.priority)),
        -backtrack,
        -same_click,
        -len(child.path),
        random.random(),
    )


def select_diverse_nodes(nodes: list[OfflineNode], beam_width: int) -> list[OfflineNode]:
    out: list[OfflineNode] = []
    seen_keys = set()
    frame_counts: dict[str, int] = defaultdict(int)
    for node in nodes:
        if node.key in seen_keys:
            continue
        if frame_counts[node.frame_hash] >= 4:
            continue
        seen_keys.add(node.key)
        frame_counts[node.frame_hash] += 1
        out.append(node)
        if len(out) >= beam_width:
            return out
    for node in nodes:
        if node.key in seen_keys:
            continue
        seen_keys.add(node.key)
        out.append(node)
        if len(out) >= beam_width:
            break
    return out


def bfs_simple_level(
    ctx: OfflineLevelContext,
    deadline: float,
    max_depth: int,
    max_nodes: int = 2500,
) -> Optional[list[tuple[int, int, int]]]:
    root = offline_node_from(ctx.root_game, ctx.root_obs, tuple(), (0, -1, -1))
    queue: deque[OfflineNode] = deque([root])
    visited: dict[Any, int] = {root.key: 0}
    simple_probe = probe_simple_actions(ctx)
    for aid, info in simple_probe.items():
        if info.get("win") or info.get("level_up"):
            return [(int(aid), -1, -1)]
    ordered = sorted(
        [aid for aid in extract_available_action_ids(ctx.root_obs) if aid != 6],
        key=lambda aid: (
            int(simple_probe.get(aid, {}).get("win", False)),
            int(simple_probe.get(aid, {}).get("level_up", False)),
            int(simple_probe.get(aid, {}).get("hidden_only", False)),
            int(simple_probe.get(aid, {}).get("effective", False)),
            int(simple_probe.get(aid, {}).get("changed", 0)),
            -int(simple_probe.get(aid, {}).get("game_over", False)),
        ),
        reverse=True,
    )
    expanded = 0
    while queue and time.time() < deadline and expanded < max_nodes:
        node = queue.popleft()
        if len(node.path) >= max_depth:
            continue
        for aid in ordered:
            cand = OfflineCandidate(action_id=int(aid), kind="simple", source="bfs", priority=0.0)
            child = simulate_offline_child(ctx, node, cand)
            if child is None:
                continue
            if child.state == "WIN" or child.levels_completed > ctx.start_levels:
                return list(child.path)
            if child.state == "GAME_OVER":
                continue
            child_depth = len(child.path)
            if visited.get(child.key, 10 ** 9) <= child_depth:
                continue
            visited[child.key] = child_depth
            queue.append(child)
            expanded += 1
            if expanded >= max_nodes:
                break
    return None


def beam_search_level(
    ctx: OfflineLevelContext,
    deadline: float,
    max_depth: int,
    beam_width: int,
) -> Optional[list[tuple[int, int, int]]]:
    root = offline_node_from(ctx.root_game, ctx.root_obs, tuple(), (0, -1, -1))
    simple_probe = probe_simple_actions(ctx)
    for aid, info in simple_probe.items():
        if info.get("win") or info.get("level_up"):
            return [(int(aid), -1, -1)]

    click_probe = probe_click_actions(ctx) if 6 in extract_available_action_ids(ctx.root_obs) else []
    for info in click_probe:
        if info.get("win") or info.get("level_up"):
            return [(6, int(info["x"]), int(info["y"]))]

    visited: dict[Any, int] = {root.key: 0}
    beam: list[OfflineNode] = [root]

    for depth in range(max_depth):
        if time.time() >= deadline:
            break
        next_nodes: list[OfflineNode] = []
        for node in beam:
            if time.time() >= deadline:
                break
            candidates = build_offline_candidates(
                ctx,
                node,
                depth,
                simple_probe=simple_probe if depth == 0 else None,
                click_probe=click_probe if depth == 0 else None,
            )
            node_cap = 18 if depth == 0 else 14 if depth <= 2 else 10
            for cand in candidates[:node_cap]:
                child = simulate_offline_child(ctx, node, cand)
                if child is None:
                    continue
                if child.state == "WIN" or child.levels_completed > ctx.start_levels:
                    return list(child.path)
                if child.state == "GAME_OVER":
                    continue
                child_depth = len(child.path)
                if visited.get(child.key, 10 ** 9) <= child_depth:
                    continue
                visited[child.key] = child_depth
                child.rank = rank_offline_child(ctx, node, child, cand)
                next_nodes.append(child)
        if not next_nodes:
            break
        next_nodes.sort(key=lambda n: n.rank, reverse=True)
        beam = select_diverse_nodes(next_nodes, beam_width)
    return None


def rollout_search_level(
    ctx: OfflineLevelContext,
    deadline: float,
    max_depth: int,
    episodes: int = 48,
) -> Optional[list[tuple[int, int, int]]]:
    root = offline_node_from(ctx.root_game, ctx.root_obs, tuple(), (0, -1, -1))
    seen_best_depth: dict[Any, int] = {root.key: 0}
    simple_probe = probe_simple_actions(ctx)
    click_probe = probe_click_actions(ctx) if 6 in extract_available_action_ids(ctx.root_obs) else []
    episode = 0

    while time.time() < deadline and episode < episodes:
        node = root
        local_seen = {node.key}
        for depth in range(max_depth):
            if time.time() >= deadline:
                break
            candidates = build_offline_candidates(
                ctx,
                node,
                depth,
                simple_probe=simple_probe if depth == 0 else None,
                click_probe=click_probe if depth == 0 else None,
            )
            scored: list[tuple[float, OfflineNode]] = []
            for cand in candidates[:10]:
                child = simulate_offline_child(ctx, node, cand)
                if child is None or child.state == "GAME_OVER":
                    continue
                if child.state == "WIN" or child.levels_completed > ctx.start_levels:
                    return list(child.path)
                if child.key in local_seen:
                    continue
                rank = rank_offline_child(ctx, node, child, cand)
                scalar = (
                    100000.0 * rank[0]
                    + 1000.0 * rank[1]
                    + 40.0 * rank[2]
                    + 18.0 * rank[3]
                    + 8.0 * rank[4]
                    + 1.0 * rank[5]
                    + 0.2 * rank[6]
                    + 0.05 * rank[7]
                )
                best_depth = seen_best_depth.get(child.key)
                if best_depth is not None and best_depth <= len(child.path):
                    scalar -= 50.0
                scored.append((scalar, child))
            if not scored:
                break
            scored.sort(key=lambda x: x[0], reverse=True)
            top = scored[: min(5, len(scored))]
            values = np.array([max(0.001, s - top[-1][0] + 0.5) for s, _ in top], dtype=float)
            probs = values / values.sum()
            idx = int(np.random.choice(len(top), p=probs))
            node = top[idx][1]
            local_seen.add(node.key)
            seen_best_depth[node.key] = min(seen_best_depth.get(node.key, 10 ** 9), len(node.path))
        episode += 1
    return None


def offline_level_time_budget(level_idx: int, baseline: int, has_click: bool) -> float:
    baseline = int(max(1, baseline))
    budget = 3.5 + 0.22 * baseline + 0.8 * max(0, level_idx - 1)
    if has_click:
        budget += 1.5
    return float(max(4.0, min(18.0 if has_click else 14.0, budget)))


def offline_game_time_budget(game_index: int, total_games: int) -> float:
    elapsed = time.time() - GLOBAL_START
    remaining = max(120.0, GLOBAL_WALLCLOCK_LIMIT_S - elapsed - 180.0)
    per_game = remaining / max(1, total_games - game_index)
    return float(max(16.0, min(90.0, per_game * 0.85)))


def solve_level_offline(
    ctx: OfflineLevelContext,
    deadline: float,
    previous_level_summary: Optional[dict[str, Any]] = None,
) -> Optional[list[tuple[int, int, int]]]:
    transferred = try_transfer_from_previous_level(ctx, previous_level_summary=previous_level_summary, deadline=min(deadline, time.time() + 2.0))
    if transferred:
        return transferred

    avail = extract_available_action_ids(ctx.root_obs)
    has_click = 6 in avail
    baseline = int(max(1, ctx.baseline_actions))
    if not has_click or len(avail) <= 3:
        bfs_depth = int(max(8, min(36, baseline + 3)))
        path = bfs_simple_level(ctx, deadline=min(deadline, time.time() + min(8.0, max(0.0, deadline - time.time()))), max_depth=bfs_depth)
        if path:
            return path
    beam_depth = int(max(10, min(28 if has_click else 44, baseline + 4)))
    beam_width = 32 if has_click else 96
    path = beam_search_level(ctx, deadline=deadline, max_depth=beam_depth, beam_width=beam_width)
    if path:
        return path
    if time.time() < deadline - 0.5:
        deeper_depth = int(max(beam_depth + 4, min(40 if has_click else 56, int(2.2 * baseline) + 4)))
        path = beam_search_level(ctx, deadline=deadline, max_depth=deeper_depth, beam_width=48 if has_click else 128)
        if path:
            return path
    if time.time() < deadline - 0.5:
        rollout_depth = int(max(beam_depth, min(36 if has_click else 52, int(2.0 * baseline) + 4)))
        path = rollout_search_level(ctx, deadline=deadline, max_depth=rollout_depth, episodes=36 if has_click else 20)
        if path:
            return path
    return None


def estimate_game_score(baseline_actions: list[int], level_action_counts: list[int]) -> float:
    if not baseline_actions:
        return 0.0
    ai_counts = [int(x) for x in level_action_counts]
    if ai_counts:
        ai_counts[0] += 1
    total = 0.0
    denom = 0.0
    for idx, human in enumerate(baseline_actions, start=1):
        denom += idx
        if idx - 1 < len(ai_counts) and ai_counts[idx - 1] > 0 and human > 0:
            score = min((float(human) / float(ai_counts[idx - 1])) ** 2, 1.0)
        else:
            score = 0.0
        total += idx * score
    return float(total / denom) if denom > 0 else 0.0


def plan_game_offline(
    env_info: LocalEnvInfo,
    game_index: int,
    total_games: int,
) -> dict[str, Any]:
    result: dict[str, Any] = {
        "tokens": [],
        "expected_trace": [],
        "level_action_counts": [],
        "levels_completed": 0,
        "final_state": "",
        "solved": False,
        "error": "",
        "estimated_score": 0.0,
    }
    game_deadline = time.time() + offline_game_time_budget(game_index, total_games)
    try:
        factory = GameSourceFactory(env_info)
    except Exception as exc:
        result["error"] = f"factory: {exc!r}"
        return result

    try:
        game = factory.new_game(seed=SEED)
        obs = factory.reset(game)
    except Exception as exc:
        result["error"] = f"boot: {exc!r}"
        return result

    tokens: list[tuple[int, int, int]] = []
    expected_trace: list[dict[str, Any]] = []
    level_action_counts: list[int] = []
    previous_level_summary: Optional[dict[str, Any]] = None

    while time.time() < game_deadline and time.time() - GLOBAL_START < GLOBAL_WALLCLOCK_LIMIT_S - 180.0:
        current_state = state_name(obs)
        if current_state == "WIN":
            break
        if current_state == "GAME_OVER":
            break

        level_idx = levels_completed(obs) + 1
        baseline = env_info.baseline_actions[level_idx - 1] if level_idx - 1 < len(env_info.baseline_actions) else max(6, 4 + 4 * level_idx)
        has_click = 6 in extract_available_action_ids(obs)
        level_deadline = min(
            game_deadline,
            time.time() + offline_level_time_budget(level_idx, baseline, has_click),
        )
        ctx = OfflineLevelContext(
            env_info=env_info,
            factory=factory,
            full_prefix_before_level=tuple(tokens),
            root_game=game,
            root_obs=obs,
            baseline_actions=baseline,
        )
        path = solve_level_offline(ctx, deadline=level_deadline, previous_level_summary=previous_level_summary)
        if not path:
            break

        start_levels = levels_completed(obs)
        used = 0
        progressed = False
        for token in path:
            try:
                obs = perform_action_game(game, int(token[0]), token_to_data(token), None)
            except Exception:
                progressed = False
                break
            tokens.append(token)
            used += 1
            raw_frame, _ = extract_latest_frame(obs)
            expected_trace.append(
                {
                    "frame_hash": FP.hash_frame(raw_frame),
                    "levels_completed": levels_completed(obs),
                    "state": state_name(obs),
                }
            )
            if state_name(obs) == "WIN" or levels_completed(obs) > start_levels:
                progressed = True
                break
            if state_name(obs) == "GAME_OVER":
                progressed = False
                break
        if not progressed:
            break
        level_action_counts.append(used)
        previous_level_summary = {
            "path": tuple(path[:used]),
            "root_frame": ctx.root_frame.copy(),
            "level_idx": int(level_idx),
        }

    result["tokens"] = tokens
    result["expected_trace"] = expected_trace
    result["level_action_counts"] = level_action_counts
    result["levels_completed"] = levels_completed(obs)
    result["final_state"] = state_name(obs)
    result["solved"] = state_name(obs) == "WIN"
    result["estimated_score"] = estimate_game_score(env_info.baseline_actions, level_action_counts)
    return result


def replay_plan_online(
    env: Any,
    plan_tokens: list[tuple[int, int, int]],
    expected_trace: list[dict[str, Any]],
) -> tuple[Any, int, int, int, bool]:
    obs, used = perform_reset(env, reason="initial reset", max_attempts=3)
    actions_in_game = int(used)
    current_level_actions = 0
    last_completed = levels_completed(obs)
    replayed = 0
    diverged = False

    for idx, token in enumerate(plan_tokens):
        if state_name(obs) in {"WIN", "GAME_OVER"}:
            break
        obs = take_action(env, int(token[0]), data=token_to_data(token), reasoning=None)
        actions_in_game += 1
        replayed += 1
        current_level_actions += 1

        new_completed = levels_completed(obs)
        if new_completed > last_completed:
            last_completed = new_completed
            current_level_actions = 0

        if idx < len(expected_trace):
            expected = expected_trace[idx]
            try:
                raw_frame, _ = extract_latest_frame(obs)
                frame_hash = FP.hash_frame(raw_frame)
            except Exception:
                frame_hash = None
            if (
                expected.get("levels_completed") != levels_completed(obs)
                or expected.get("state") != state_name(obs)
                or (frame_hash is not None and expected.get("frame_hash") != frame_hash)
            ):
                diverged = True
                break
    return obs, actions_in_game, current_level_actions, replayed, diverged


# ==================== Compact hidden-state BFS solver ====================

class BFSSolver:
    """Offline BFS solver using direct source instantiation.

    Best used as a rescue solver for sparse / hidden-state games after the
    stronger source planner and object-centric offline planner fail.
    """

    def __init__(self, game_path: str, game_class_name: str, scan_timeout: float = 1.2, bfs_timeout: float = 10.0) -> None:
        self.game_path = str(game_path)
        self.class_name = str(game_class_name)
        self.scan_timeout = float(scan_timeout)
        self.bfs_timeout = float(bfs_timeout)
        self.game_cls = None
        self.solutions: dict[int, list[tuple[int, Optional[dict[str, int]]]]] = {}

    def load(self) -> bool:
        try:
            self.game_cls = _load_game_class(self.game_path, self.class_name)
            return self.game_cls is not None
        except Exception:
            return False

    def _state_hash(
        self,
        game: Any,
        frame: np.ndarray,
        hidden_fields: Optional[list[str]] = None,
    ) -> str:
        fh = hashlib.md5(np.asarray(frame, dtype=np.uint8).tobytes()).hexdigest()[:16]
        if not hidden_fields:
            return fh
        extras: list[str] = []
        for field_name in hidden_fields:
            try:
                value = getattr(game, field_name, None)
            except Exception:
                value = None
            if isinstance(value, (int, float, bool)):
                extras.append(f"{field_name}={value}")
        if extras:
            return fh + "|" + "|".join(extras)
        return fh

    def _probe_hidden_fields(
        self,
        game: Any,
        actions: list[tuple[int, Optional[dict[str, int]]]],
    ) -> list[str]:
        if not actions:
            return []
        initial: dict[str, Any] = {}
        for key, value in getattr(game, "__dict__", {}).items():
            if isinstance(value, (int, float, bool)) and not str(key).startswith("__"):
                initial[str(key)] = value

        changing: set[str] = set()
        frame0 = None
        try:
            frame0 = _local_frame_from_result(None, game)
        except Exception:
            frame0 = None
        if frame0 is None and hasattr(game, "get_pixels"):
            try:
                frame0 = game.get_pixels(0, 0, 64, 64)
            except Exception:
                frame0 = None
        if frame0 is None:
            return []
        frame0 = np.asarray(frame0, dtype=np.uint8)

        for act_id, data in actions[:12]:
            g = try_clone_game(game)
            if g is None:
                continue
            try:
                _local_step(g, int(act_id), data=data)
            except Exception:
                continue
            for key, value in getattr(g, "__dict__", {}).items():
                if isinstance(value, (int, float, bool)) and key in initial and value != initial[key]:
                    if key not in {"_action_count", "_full_reset", "_action_complete"}:
                        changing.add(str(key))

        hidden = []
        for field_name in sorted(changing):
            if field_name.startswith("_") and field_name not in {"_current_level_index", "_score"}:
                continue
            hidden.append(field_name)
        return hidden

    def _scan_actions(
        self,
        game: Any,
        frame0: np.ndarray,
        background_color: int,
    ) -> list[tuple[int, Optional[dict[str, int]]]]:
        avail = []
        try:
            avail = list(_local_available_ids(game))
        except Exception:
            try:
                avail = [normalize_action_id(a) for a in getattr(game, "_available_actions", [])]
            except Exception:
                avail = []
        actions: list[tuple[int, Optional[dict[str, int]]]] = []

        for aid in [a for a in avail if a in {1, 2, 3, 4, 5, 7}]:
            g = try_clone_game(game)
            if g is None:
                continue
            try:
                result = _local_step(g, int(aid), data=None)
                frame = _local_frame_from_result(result, g)
            except Exception:
                frame = None
            if frame is None:
                continue
            if np.sum(np.asarray(frame0, dtype=np.uint8) != np.asarray(frame, dtype=np.uint8)) > 0:
                actions.append((int(aid), None))

        if 6 in avail:
            seen_effects: set[str] = set()
            h, w = map(int, frame0.shape[:2])
            step = 2 if max(h, w) <= 32 else 3
            t0 = time.time()
            for y in range(0, h, step):
                if time.time() - t0 > self.scan_timeout:
                    break
                for x in range(0, w, step):
                    if time.time() - t0 > self.scan_timeout:
                        break
                    if int(frame0[y, x]) == int(background_color):
                        continue
                    g = try_clone_game(game)
                    if g is None:
                        continue
                    data = {"x": int(x), "y": int(y)}
                    try:
                        result = _local_step(g, 6, data=data)
                        frame = _local_frame_from_result(result, g)
                    except Exception:
                        frame = None
                    if frame is None:
                        continue
                    diff = int(np.sum(np.asarray(frame0, dtype=np.uint8) != np.asarray(frame, dtype=np.uint8)))
                    if diff <= 0:
                        continue
                    effect_hash = hashlib.md5(np.asarray(frame, dtype=np.uint8).tobytes()).hexdigest()[:12]
                    if effect_hash in seen_effects:
                        continue
                    seen_effects.add(effect_hash)
                    actions.append((6, data))

        return actions

    def solve_level(
        self,
        level_idx: int,
        max_states: int = 200000,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        if self.game_cls is None:
            return None

        try:
            game = self.game_cls()
        except Exception:
            return None
        if hasattr(game, "set_level"):
            try:
                game.set_level(int(level_idx))
            except Exception:
                pass

        result = None
        frame0 = None
        for _ in range(3):
            try:
                result = _local_step(game, 0, data=None)
            except Exception:
                result = None
            frame0 = _local_frame_from_result(result, game)
            if frame0 is not None:
                break
        if frame0 is None:
            return None
        frame0 = np.asarray(frame0, dtype=np.uint8)
        background_color = int(np.bincount(frame0.flatten(), minlength=16).argmax())

        if prev_solution and level_idx > 0:
            transfer = self._try_transfer(game, level_idx, prev_solution, frame0)
            if transfer:
                self.solutions[int(level_idx)] = transfer
                return transfer

        actions = self._scan_actions(game, frame0, background_color)
        if not actions:
            return None

        visited: set[str] = set()
        queue: deque[tuple[Any, list[tuple[int, Optional[dict[str, int]]]], int]] = deque()
        h0 = self._state_hash(game, frame0, None)
        visited.add(h0)
        queue.append((try_clone_game(game), [], 0))
        start_completed = _local_levels_completed(result, game)
        t0 = time.time()
        explored = 0

        while queue and explored < int(max_states) and (time.time() - t0) < self.bfs_timeout:
            current_game, hist, depth = queue.popleft()
            if current_game is None:
                continue

            for act_id, data in actions:
                g2 = try_clone_game(current_game)
                if g2 is None:
                    continue
                try:
                    result2 = _local_step(g2, int(act_id), data=data)
                except Exception:
                    continue
                explored += 1
                frame2 = _local_frame_from_result(result2, g2)
                if frame2 is None:
                    continue
                frame2 = np.asarray(frame2, dtype=np.uint8)
                h2 = self._state_hash(g2, frame2, None)
                if h2 in visited:
                    continue
                visited.add(h2)

                new_hist = hist + [(int(act_id), dict(data) if isinstance(data, dict) else None)]
                new_state = _local_state_name(result2, g2)
                new_completed = _local_levels_completed(result2, g2)
                if new_completed > start_completed or new_state == "WIN":
                    self.solutions[int(level_idx)] = new_hist
                    return new_hist
                if new_state == "GAME_OVER":
                    continue
                if depth < 30:
                    queue.append((g2, new_hist, depth + 1))

        if len(visited) < 50 and (time.time() - t0) < self.bfs_timeout * 0.85:
            hidden_fields = self._probe_hidden_fields(game, actions)
            if hidden_fields:
                visited2: set[str] = set()
                queue2: deque[tuple[Any, list[tuple[int, Optional[dict[str, int]]]], int]] = deque()
                try:
                    game2 = self.game_cls()
                except Exception:
                    return None
                if hasattr(game2, "set_level"):
                    try:
                        game2.set_level(int(level_idx))
                    except Exception:
                        pass
                result_start = None
                frame_start = None
                for _ in range(3):
                    try:
                        result_start = _local_step(game2, 0, data=None)
                    except Exception:
                        result_start = None
                    frame_start = _local_frame_from_result(result_start, game2)
                    if frame_start is not None:
                        break
                if frame_start is None:
                    return None
                frame_start = np.asarray(frame_start, dtype=np.uint8)
                visited2.add(self._state_hash(game2, frame_start, hidden_fields))
                queue2.append((try_clone_game(game2), [], 0))
                start_completed2 = _local_levels_completed(result_start, game2)
                t1 = time.time()
                explored2 = 0
                remaining = max(2.0, self.bfs_timeout - (time.time() - t0))

                while queue2 and explored2 < int(max_states) and (time.time() - t1) < remaining:
                    current_game, hist, depth = queue2.popleft()
                    if current_game is None:
                        continue
                    for act_id, data in actions:
                        g2 = try_clone_game(current_game)
                        if g2 is None:
                            continue
                        try:
                            result2 = _local_step(g2, int(act_id), data=data)
                        except Exception:
                            continue
                        explored2 += 1
                        frame2 = _local_frame_from_result(result2, g2)
                        if frame2 is None:
                            continue
                        frame2 = np.asarray(frame2, dtype=np.uint8)
                        h2 = self._state_hash(g2, frame2, hidden_fields)
                        if h2 in visited2:
                            continue
                        visited2.add(h2)
                        new_hist = hist + [(int(act_id), dict(data) if isinstance(data, dict) else None)]
                        new_state = _local_state_name(result2, g2)
                        new_completed = _local_levels_completed(result2, g2)
                        if new_completed > start_completed2 or new_state == "WIN":
                            self.solutions[int(level_idx)] = new_hist
                            return new_hist
                        if new_state == "GAME_OVER":
                            continue
                        if depth < 30:
                            queue2.append((g2, new_hist, depth + 1))
        return None

    def _try_transfer(
        self,
        game: Any,
        level_idx: int,
        prev_solution: list[tuple[int, Optional[dict[str, int]]]],
        current_frame: np.ndarray,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        try:
            g = try_clone_game(game)
            if g is not None:
                for step_idx, (act_id, data) in enumerate(prev_solution):
                    try:
                        result = _local_step(g, int(act_id), data=data)
                    except Exception:
                        break
                    new_state = _local_state_name(result, g)
                    new_completed = _local_levels_completed(result, g)
                    if new_completed > level_idx or new_state == "WIN":
                        return list(prev_solution[: step_idx + 1])

            if level_idx <= 0:
                return None

            try:
                prev_game = self.game_cls()
            except Exception:
                return None
            if hasattr(prev_game, "set_level"):
                try:
                    prev_game.set_level(int(level_idx - 1))
                except Exception:
                    pass
            prev_result = None
            prev_frame = None
            for _ in range(3):
                try:
                    prev_result = _local_step(prev_game, 0, data=None)
                except Exception:
                    prev_result = None
                prev_frame = _local_frame_from_result(prev_result, prev_game)
                if prev_frame is not None:
                    break
            if prev_frame is None:
                return None
            prev_frame = np.asarray(prev_frame, dtype=np.uint8)
            bg = int(np.bincount(prev_frame.flatten(), minlength=16).argmax())
            current_frame = np.asarray(current_frame, dtype=np.uint8)

            def get_objects(frame: np.ndarray, bg_color: int) -> list[dict[str, float]]:
                objs: list[dict[str, float]] = []
                for color in range(16):
                    if int(color) == int(bg_color):
                        continue
                    mask = frame == int(color)
                    n = int(np.sum(mask))
                    if n < 2:
                        continue
                    ys, xs = np.where(mask)
                    objs.append(
                        {
                            "color": int(color),
                            "cx": float(np.mean(xs)),
                            "cy": float(np.mean(ys)),
                            "n": float(n),
                        }
                    )
                return sorted(objs, key=lambda obj: (obj["color"], -obj["n"]))

            objs_prev = get_objects(prev_frame, bg)
            objs_curr = get_objects(current_frame, bg)
            if not objs_prev or not objs_curr:
                return None

            matched: list[tuple[dict[str, float], dict[str, float]]] = []
            for obj_prev in objs_prev:
                best = None
                best_dist = float("inf")
                for obj_curr in objs_curr:
                    if obj_curr["color"] != obj_prev["color"]:
                        continue
                    if abs(obj_curr["n"] - obj_prev["n"]) > max(obj_curr["n"], obj_prev["n"]) * 0.5:
                        continue
                    dist = abs(obj_curr["cx"] - obj_prev["cx"]) + abs(obj_curr["cy"] - obj_prev["cy"])
                    if dist < best_dist:
                        best_dist = dist
                        best = obj_curr
                if best is not None:
                    matched.append((obj_prev, best))
            if not matched:
                return None

            dx = float(np.mean([pair[1]["cx"] - pair[0]["cx"] for pair in matched]))
            dy = float(np.mean([pair[1]["cy"] - pair[0]["cy"] for pair in matched]))

            transferred: list[tuple[int, Optional[dict[str, int]]]] = []
            for act_id, data in prev_solution:
                if isinstance(data, dict) and "x" in data and "y" in data:
                    transferred.append(
                        (
                            int(act_id),
                            {
                                "x": max(0, min(63, int(round(float(data["x"]) + dx)))),
                                "y": max(0, min(63, int(round(float(data["y"]) + dy)))),
                            },
                        )
                    )
                else:
                    transferred.append((int(act_id), dict(data) if isinstance(data, dict) else None))

            g = try_clone_game(game)
            if g is None:
                return None
            for step_idx, (act_id, data) in enumerate(transferred):
                try:
                    result = _local_step(g, int(act_id), data=data)
                except Exception:
                    break
                new_state = _local_state_name(result, g)
                new_completed = _local_levels_completed(result, g)
                if new_completed > level_idx or new_state == "WIN":
                    return transferred[: step_idx + 1]
        except Exception:
            return None
        return None



# ==================== Strategic online fallback + solver ensembling ====================

SOLVER_LOG = logging.getLogger("arc_submission")
SOLVER_LOG.setLevel(logging.WARNING)

def empty_result(reason: str = "") -> dict[str, Any]:
    return {
        "tokens": [],
        "expected_trace": [],
        "level_action_counts": [],
        "levels_completed": 0,
        "final_state": "",
        "solved": False,
        "estimated_score": 0.0,
        "error": str(reason or ""),
        "level_plans": [],
        "sources_by_level": [],
    }


def action_pair_to_token(aid: int, data: Optional[dict[str, Any]] = None) -> tuple[int, int, int]:
    x = -1
    y = -1
    if isinstance(data, dict):
        try:
            x = int(data.get("x", -1))
        except Exception:
            x = -1
        try:
            y = int(data.get("y", -1))
        except Exception:
            y = -1
    return (int(aid), int(x), int(y))


def token_to_action_pair(token: tuple[int, int, int]) -> tuple[int, Optional[dict[str, int]]]:
    aid = int(token[0])
    return aid, token_to_data(token)


def split_by_counts(seq: list[Any], counts: list[int]) -> list[list[Any]]:
    out: list[list[Any]] = []
    pos = 0
    items = list(seq or [])
    for raw_count in counts or []:
        count = max(0, int(raw_count))
        out.append(items[pos:pos + count])
        pos += count
    return out


def level_plan_value(level_idx: int, action_count: int, baseline_actions: list[int]) -> float:
    count = max(0, int(action_count))
    if level_idx == 0:
        count += 1
    if level_idx < len(baseline_actions):
        human = max(0, int(baseline_actions[level_idx]))
        if human > 0 and count > 0:
            return float(min((float(human) / float(count)) ** 2, 1.0))
    return float(1.0 / max(1, count))


def make_level_plan(
    level_idx: int,
    source: str,
    tokens: list[tuple[int, int, int]],
    expected_trace: Optional[list[dict[str, Any]]],
    baseline_actions: list[int],
    action_pairs: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
) -> dict[str, Any]:
    toks = [tuple(int(v) for v in tok) for tok in (tokens or [])]
    if action_pairs is None:
        action_pairs = [token_to_action_pair(tok) for tok in toks]
    trace = list(expected_trace or [])
    return {
        "level_idx": int(level_idx),
        "source": str(source),
        "tokens": toks,
        "expected_trace": trace,
        "action_pairs": [
            (int(aid), dict(data) if isinstance(data, dict) else None)
            for aid, data in action_pairs
        ],
        "action_count": int(len(toks)),
        "score": level_plan_value(int(level_idx), len(toks), baseline_actions),
    }


def result_to_level_plans(
    result: Optional[dict[str, Any]],
    baseline_actions: list[int],
    default_source: str = "offline",
) -> list[dict[str, Any]]:
    if not isinstance(result, dict):
        return []
    tokens = [tuple(int(v) for v in tok) for tok in (result.get("tokens") or [])]
    counts = [max(0, int(x)) for x in (result.get("level_action_counts") or [])]
    traces = list(result.get("expected_trace") or [])
    token_segments = split_by_counts(tokens, counts)
    trace_segments = split_by_counts(traces, counts)
    source_names = list(result.get("sources_by_level") or [])
    out: list[dict[str, Any]] = []
    for level_idx, seg_tokens in enumerate(token_segments):
        if not seg_tokens:
            continue
        seg_trace = trace_segments[level_idx] if level_idx < len(trace_segments) else []
        source_name = source_names[level_idx] if level_idx < len(source_names) else default_source
        out.append(
            make_level_plan(
                level_idx=level_idx,
                source=source_name,
                tokens=seg_tokens,
                expected_trace=seg_trace,
                baseline_actions=baseline_actions,
            )
        )
    return out


def build_result_from_level_plans(
    level_plans: list[dict[str, Any]],
    baseline_actions: list[int],
    error: str = "",
) -> dict[str, Any]:
    tokens: list[tuple[int, int, int]] = []
    expected_trace: list[dict[str, Any]] = []
    counts: list[int] = []
    sources: list[str] = []
    clean_level_plans: list[dict[str, Any]] = []
    for plan in level_plans or []:
        toks = [tuple(int(v) for v in tok) for tok in (plan.get("tokens") or [])]
        if not toks:
            break
        trace = list(plan.get("expected_trace") or [])
        counts.append(int(len(toks)))
        sources.append(str(plan.get("source") or "unknown"))
        tokens.extend(toks)
        expected_trace.extend(trace)
        clean_level_plans.append(
            make_level_plan(
                level_idx=int(plan.get("level_idx", len(clean_level_plans))),
                source=str(plan.get("source") or "unknown"),
                tokens=toks,
                expected_trace=trace,
                baseline_actions=baseline_actions,
                action_pairs=plan.get("action_pairs"),
            )
        )
    estimated_score = estimate_game_score(baseline_actions, counts)
    final_state = "WIN" if baseline_actions and len(clean_level_plans) >= len(baseline_actions) else ""
    return {
        "tokens": tokens,
        "expected_trace": expected_trace,
        "level_action_counts": counts,
        "levels_completed": int(len(clean_level_plans)),
        "final_state": final_state,
        "solved": bool(final_state == "WIN"),
        "estimated_score": float(estimated_score),
        "error": str(error or ""),
        "level_plans": clean_level_plans,
        "sources_by_level": sources,
    }


SOURCE_PRIORITY = {
    "offline18": 1,
    "sourceplanner": 2,
    "bfs": 0,
}


def merge_level_plan_sets(
    plan_sets: list[list[dict[str, Any]]],
    baseline_actions: list[int],
) -> dict[str, Any]:
    max_levels = max((len(plans) for plans in plan_sets if plans is not None), default=0)
    chosen: list[dict[str, Any]] = []
    for level_idx in range(max_levels):
        candidates: list[dict[str, Any]] = []
        for plans in plan_sets:
            if not plans or level_idx >= len(plans):
                continue
            plan = plans[level_idx]
            toks = plan.get("tokens") or []
            if toks:
                candidates.append(plan)
        if not candidates:
            break
        best = max(
            candidates,
            key=lambda p: (
                float(p.get("score", 0.0) or 0.0),
                -int(len(p.get("tokens") or [])),
                int(SOURCE_PRIORITY.get(str(p.get("source") or ""), 0)),
            ),
        )
        chosen.append(best)
    return build_result_from_level_plans(chosen, baseline_actions)


def should_try_bfs(result: dict[str, Any], baseline_actions: list[int]) -> bool:
    solved = int(result.get("levels_completed", 0) or 0)
    est = float(result.get("estimated_score", 0.0) or 0.0)
    if solved <= 0:
        return True
    if solved < min(2, max(1, len(baseline_actions))):
        return True
    return est < 0.20



# --- Conservative offline planner selection overrides for Submission 25 ---
ALT_SOURCE_PENALTY = {
    "offline18": 0.000,
    "bfs": 0.012,
    "sourceplanner": 0.040,
}
ALT_SOURCE_MARGIN = {
    "offline18": 0.000,
    "bfs": 0.010,
    "sourceplanner": 0.030,
}

EXTRA_SEMANTIC_PATH_TOKENS = {
    "pickup": 6,
    "drop": 6,
    "carry": 6,
    "inventory": 6,
    "collect": 5,
    "bridge": 6,
    "lever": 6,
    "toggle": 6,
    "teleport": 7,
    "warp": 7,
    "bomb": 4,
    "laser": 4,
    "beam": 4,
    "mirror": 4,
    "rail": 4,
    "cart": 4,
    "train": 4,
    "water": 4,
    "fire": 4,
    "ice": 4,
    "lava": 4,
    "hole": 4,
    "pit": 4,
    "spike": 4,
    "health": 4,
    "hp": 4,
    "fuel": 4,
    "energy": 4,
    "charge": 4,
    "ammo": 4,
    "open": 4,
    "locked": 5,
    "unlock": 5,
    "facing": 3,
    "direction": 3,
    "dir": 3,
    "dx": 2,
    "dy": 2,
}
try:
    SEMANTIC_PATH_TOKENS.update(EXTRA_SEMANTIC_PATH_TOKENS)
except Exception:
    pass

def should_try_bfs(result: dict[str, Any], baseline_actions: list[int]) -> bool:
    solved = int(result.get("levels_completed", 0) or 0)
    est = float(result.get("estimated_score", 0.0) or 0.0)
    target = min(2, max(1, len(baseline_actions)))
    if solved <= 0:
        return True
    if solved < target and est < 0.30:
        return True
    return est < 0.10

def should_try_source_planner_offline(result: dict[str, Any], baseline_actions: list[int]) -> bool:
    solved = int(result.get("levels_completed", 0) or 0)
    est = float(result.get("estimated_score", 0.0) or 0.0)
    target = min(2, max(1, len(baseline_actions)))
    if solved <= 0:
        return True
    if solved < target and est < 0.28:
        return True
    return est < 0.08

def result_rank_tuple(result: dict[str, Any]) -> tuple[float, int, int, int]:
    counts = [max(0, int(x)) for x in (result.get("level_action_counts") or [])]
    total_actions = int(sum(counts)) if counts else int(len(result.get("tokens") or []))
    return (
        float(result.get("estimated_score", 0.0) or 0.0),
        int(result.get("levels_completed", 0) or 0),
        -total_actions,
        -int(len(result.get("tokens") or [])),
    )

def conservative_merge_level_plan_sets(
    plan_sets: list[list[dict[str, Any]]],
    baseline_actions: list[int],
) -> dict[str, Any]:
    if not plan_sets:
        return build_result_from_level_plans([], baseline_actions)

    base_set = plan_sets[0] if plan_sets else []
    max_levels = max((len(plans) for plans in plan_sets if plans is not None), default=0)
    chosen: list[dict[str, Any]] = []

    for level_idx in range(max_levels):
        candidates: list[dict[str, Any]] = []
        for plans in plan_sets:
            if not plans or level_idx >= len(plans):
                continue
            plan = plans[level_idx]
            if plan.get("tokens"):
                candidates.append(plan)
        if not candidates:
            break

        base_plan = base_set[level_idx] if base_set and level_idx < len(base_set) else None
        if base_plan is None or not base_plan.get("tokens"):
            best = max(
                candidates,
                key=lambda p: (
                    float(p.get("score", 0.0) or 0.0) - ALT_SOURCE_PENALTY.get(str(p.get("source") or ""), 0.02),
                    -int(p.get("action_count", len(p.get("tokens") or [])) or 0),
                    int(SOURCE_PRIORITY.get(str(p.get("source") or ""), 0)),
                ),
            )
            chosen.append(best)
            continue

        base_score = float(base_plan.get("score", 0.0) or 0.0)
        base_len = max(1, int(base_plan.get("action_count", len(base_plan.get("tokens") or [])) or 0))
        best = base_plan
        best_adj = base_score

        for plan in candidates:
            if plan is base_plan:
                continue
            src = str(plan.get("source") or "")
            raw_score = float(plan.get("score", 0.0) or 0.0)
            adj_score = raw_score - ALT_SOURCE_PENALTY.get(src, 0.02)
            margin = ALT_SOURCE_MARGIN.get(src, 0.02)
            cand_len = max(1, int(plan.get("action_count", len(plan.get("tokens") or [])) or 0))

            if cand_len > max(base_len * 2 + 2, base_len + 10):
                continue

            clearly_better = adj_score > base_score + margin + 1e-9
            slightly_shorter_same_score = (
                raw_score >= base_score - 1e-9
                and cand_len + 1 < base_len
                and adj_score >= base_score - 0.005
            )
            if clearly_better or slightly_shorter_same_score:
                if adj_score > best_adj + 1e-9 or (
                    abs(adj_score - best_adj) <= 0.005
                    and cand_len < int(best.get("action_count", len(best.get("tokens") or [])) or 0)
                ):
                    best = plan
                    best_adj = adj_score

        chosen.append(best)

    return build_result_from_level_plans(chosen, baseline_actions)

def choose_conservative_offline_result(
    offline18_result: dict[str, Any],
    source_result: dict[str, Any],
    bfs_result: dict[str, Any],
    baseline_actions: list[int],
) -> dict[str, Any]:
    plan_sets: list[list[dict[str, Any]]] = []
    candidates: list[dict[str, Any]] = []

    base_plans = result_to_level_plans(offline18_result, baseline_actions, default_source="offline18")
    if base_plans:
        plan_sets.append(base_plans)
    if isinstance(offline18_result, dict):
        candidates.append(dict(offline18_result))

    if isinstance(bfs_result, dict) and (bfs_result.get("tokens") or bfs_result.get("level_plans")):
        bfs_plans = result_to_level_plans(bfs_result, baseline_actions, default_source="bfs")
        if bfs_plans:
            plan_sets.append(bfs_plans)
        candidates.append(dict(bfs_result))

    if isinstance(source_result, dict) and (source_result.get("tokens") or source_result.get("level_plans")):
        source_plans = result_to_level_plans(source_result, baseline_actions, default_source="sourceplanner")
        if source_plans:
            plan_sets.append(source_plans)
        candidates.append(dict(source_result))

    if plan_sets:
        merged = conservative_merge_level_plan_sets(plan_sets, baseline_actions)
        if merged.get("tokens"):
            candidates.append(merged)

    candidates = [cand for cand in candidates if isinstance(cand, dict)]
    if not candidates:
        return empty_result()

    return max(candidates, key=result_rank_tuple)

def resolve_source_path_from_env_info(env_info: Optional[LocalEnvInfo]) -> tuple[Optional[str], Optional[str]]:
    if env_info is None:
        return None, None
    local_dir = Path(env_info.local_dir)
    candidates = [
        local_dir / f"{env_info.class_name.lower()}.py",
        local_dir / f"{env_info.class_name}.py",
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate), str(env_info.class_name)
    py_files = [p for p in local_dir.glob("*.py") if p.name != "__init__.py"]
    if len(py_files) == 1:
        return str(py_files[0]), str(env_info.class_name)
    if py_files:
        py_files = sorted(py_files)
        return str(py_files[0]), str(env_info.class_name)
    return None, None


def trace_action_pairs_with_game_cls(
    game_cls: Any,
    level_idx: int,
    action_pairs: Optional[list[tuple[int, Optional[dict[str, int]]]]],
    fp: Optional[FrameProcessor] = None,
) -> Optional[dict[str, Any]]:
    if game_cls is None or not action_pairs:
        return None
    fp = fp or FrameProcessor()
    try:
        game = game_cls()
    except Exception:
        return None
    if hasattr(game, "set_level"):
        try:
            game.set_level(int(level_idx))
        except Exception:
            pass
    result = None
    frame = None
    for _ in range(3):
        try:
            result = _local_step(game, 0, data=None)
        except Exception:
            result = None
        frame = _local_frame_from_result(result, game)
        if frame is not None:
            break
    if frame is None:
        return None
    start_completed = _local_levels_completed(result, game)
    tokens: list[tuple[int, int, int]] = []
    trace: list[dict[str, Any]] = []
    pairs_used: list[tuple[int, Optional[dict[str, int]]]] = []
    for aid, data in action_pairs:
        try:
            result = _local_step(game, int(aid), data=data)
        except Exception:
            return None
        frame = _local_frame_from_result(result, game)
        if frame is None:
            return None
        token = action_pair_to_token(int(aid), data)
        tokens.append(token)
        pairs_used.append((int(aid), dict(data) if isinstance(data, dict) else None))
        trace.append(
            {
                "frame_hash": fp.hash_frame(np.asarray(frame, dtype=np.uint8)),
                "levels_completed": _local_levels_completed(result, game),
                "state": _local_state_name(result, game),
            }
        )
        new_state = _local_state_name(result, game)
        new_completed = _local_levels_completed(result, game)
        if new_state == "GAME_OVER":
            return None
        if new_completed > start_completed or new_state == "WIN":
            return {"tokens": tokens, "expected_trace": trace, "action_pairs": pairs_used}
    return None


def trace_planner_solution(
    planner: Optional[SourcePlanner],
    level_idx: int,
    plan: Optional[list[tuple[int, Optional[dict[str, int]]]]],
) -> Optional[dict[str, Any]]:
    if planner is None or not plan:
        return None
    try:
        game, result, frame = planner._new_game(level_idx)
    except Exception:
        return None
    if game is None or frame is None:
        return None
    start_completed = _local_levels_completed(result, game)
    tokens: list[tuple[int, int, int]] = []
    trace: list[dict[str, Any]] = []
    pairs_used: list[tuple[int, Optional[dict[str, int]]]] = []
    for aid, data in plan:
        try:
            result = _local_step(game, int(aid), data=data)
        except Exception:
            return None
        frame = _local_frame_from_result(result, game)
        if frame is None:
            return None
        tokens.append(action_pair_to_token(int(aid), data))
        pairs_used.append((int(aid), dict(data) if isinstance(data, dict) else None))
        trace.append(
            {
                "frame_hash": planner.fp.hash_frame(np.asarray(frame, dtype=np.uint8)),
                "levels_completed": _local_levels_completed(result, game),
                "state": _local_state_name(result, game),
            }
        )
        new_state = _local_state_name(result, game)
        new_completed = _local_levels_completed(result, game)
        if new_state == "GAME_OVER":
            return None
        if new_completed > start_completed or new_state == "WIN":
            return {"tokens": tokens, "expected_trace": trace, "action_pairs": pairs_used}
    return None


def plan_game_with_source_planner(
    game_id: str,
    env_info: Optional[LocalEnvInfo],
    source_path: Optional[str],
    class_name: Optional[str],
    game_index: int,
    total_games: int,
) -> tuple[dict[str, Any], Optional[SourcePlanner]]:
    if not source_path or not class_name:
        return empty_result("missing source"), None
    planner = SourcePlanner(game_id, source_path, class_name)
    try:
        if not planner.load():
            return empty_result("planner load failed"), None
    except Exception as exc:
        return empty_result(f"planner load failed: {exc}"), None

    baseline_actions = list(env_info.baseline_actions) if env_info is not None else []
    level_target = len(baseline_actions) if baseline_actions else 8

    start_t = time.time()
    remaining_global = max(0.0, GLOBAL_WALLCLOCK_LIMIT_S - (start_t - GLOBAL_START))
    games_left = max(1, total_games - game_index)
    total_budget = min(50.0, max(10.0, remaining_global / games_left * 0.25))

    level_plans: list[dict[str, Any]] = []
    for level_idx in range(level_target):
        elapsed = time.time() - start_t
        remaining = total_budget - elapsed
        if remaining < 1.5:
            break
        prev_solution = planner.solution_cache.get(level_idx - 1)
        budget = planner_time_budget_seconds(level_idx, has_transfer=prev_solution is not None)
        budget = min(float(budget), max(2.0, remaining))
        plan = None
        try:
            plan = planner.solve_level(level_idx, prev_solution=prev_solution, time_budget_s=budget)
        except Exception:
            traceback.print_exc()
            plan = None
        traced = trace_planner_solution(planner, level_idx, plan)
        if not traced:
            break
        level_plans.append(
            make_level_plan(
                level_idx=level_idx,
                source="sourceplanner",
                tokens=traced["tokens"],
                expected_trace=traced["expected_trace"],
                baseline_actions=baseline_actions,
                action_pairs=traced["action_pairs"],
            )
        )

    result = build_result_from_level_plans(level_plans, baseline_actions)
    result["level_plans"] = level_plans
    return result, planner


def plan_game_with_bfs(
    game_id: str,
    env_info: Optional[LocalEnvInfo],
    source_path: Optional[str],
    class_name: Optional[str],
    game_index: int,
    total_games: int,
) -> tuple[dict[str, Any], Optional["BFSSolver"]]:
    if not source_path or not class_name:
        return empty_result("missing source"), None
    baseline_actions = list(env_info.baseline_actions) if env_info is not None else []
    level_target = len(baseline_actions) if baseline_actions else 8

    start_t = time.time()
    remaining_global = max(0.0, GLOBAL_WALLCLOCK_LIMIT_S - (start_t - GLOBAL_START))
    games_left = max(1, total_games - game_index)
    total_budget = min(26.0, max(6.0, remaining_global / games_left * 0.12))

    solver = BFSSolver(source_path, class_name, scan_timeout=1.2, bfs_timeout=max(4.0, total_budget))
    try:
        if not solver.load():
            return empty_result("bfs load failed"), None
    except Exception as exc:
        return empty_result(f"bfs load failed: {exc}"), None

    fp = FrameProcessor()
    level_plans: list[dict[str, Any]] = []
    for level_idx in range(level_target):
        elapsed = time.time() - start_t
        remaining = total_budget - elapsed
        if remaining < 1.5:
            break
        prev_solution = solver.solutions.get(level_idx - 1)
        solver.scan_timeout = min(2.4, max(0.8, 0.9 + 0.18 * level_idx))
        solver.bfs_timeout = max(2.5, min(18.0, remaining))
        max_states = int(min(260000, 90000 + 30000 * level_idx))
        plan = None
        try:
            plan = solver.solve_level(level_idx, max_states=max_states, prev_solution=prev_solution)
        except Exception:
            traceback.print_exc()
            plan = None
        traced = trace_action_pairs_with_game_cls(solver.game_cls, level_idx, plan, fp=fp)
        if not traced:
            break
        level_plans.append(
            make_level_plan(
                level_idx=level_idx,
                source="bfs",
                tokens=traced["tokens"],
                expected_trace=traced["expected_trace"],
                baseline_actions=baseline_actions,
                action_pairs=traced["action_pairs"],
            )
        )
    result = build_result_from_level_plans(level_plans, baseline_actions)
    result["level_plans"] = level_plans
    return result, solver


def continue_with_best_online_agent(
    env: Any,
    game_id: str,
    obs: Any,
    actions_in_game: int,
    current_level_actions: int,
) -> tuple[Any, int, bool]:
    agent_cls = StrategicHybridAgent if "StrategicHybridAgent" in globals() else (
        HybridLevelExplorerAgent if "HybridLevelExplorerAgent" in globals() else LevelExplorerAgent
    )
    agent = agent_cls(game_id, global_memory=GLOBAL_FEATURE_MEMORY)
    current_view = agent.start_level(obs, env)
    agent.steps_in_level = int(current_level_actions)
    completed_flag = state_name(obs) == "WIN"

    while time.time() - GLOBAL_START <= GLOBAL_WALLCLOCK_LIMIT_S:
        st = state_name(obs)
        current_level_completed = levels_completed(obs)

        if st == "WIN":
            completed_flag = True
            break

        if actions_in_game >= game_budget(current_level_completed + 1):
            break

        if agent.steps_in_level >= level_budget(current_level_completed + 1):
            break

        if st == "GAME_OVER":
            if agent.resets_in_level >= 12:
                break
            obs, used = perform_reset(env, reason="retry level after game over", max_attempts=2)
            actions_in_game += used
            agent.note_external_reset(used)
            current_view = agent._build_view(obs, env)
            continue

        if current_view is None:
            current_view = agent._build_view(obs, env)

        choice = agent.choose_candidate(current_view)
        if choice is None:
            break

        if choice["kind"] == "meta_reset":
            obs, used = perform_reset(env, reason=choice.get("reasoning", "strategic reset"), max_attempts=2)
            actions_in_game += used
            agent.note_external_reset(used)
            current_view = agent._build_view(obs, env)
            continue

        prev_view = current_view
        candidate_idx = int(choice["candidate_idx"])
        reasoning = json.dumps(
            {
                "candidate_idx": candidate_idx,
                "kind": choice["kind"],
                "action_id": int(choice["action_id"]),
                "tag": list(choice.get("tag", [])) if isinstance(choice.get("tag"), tuple) else choice.get("tag"),
            },
            ensure_ascii=False,
        )

        try:
            obs = take_action(
                env,
                int(choice["action_id"]),
                data=choice.get("data"),
                reasoning=reasoning,
            )
        except Exception:
            agent.graph.record_test(prev_view.node_id, candidate_idx, -1, None)
            try:
                agent.remember(prev_view.candidates[candidate_idx], changed=False, errored=True)
            except Exception:
                pass
            obs, used = perform_reset(env, reason="recover after step exception", max_attempts=2)
            actions_in_game += used
            agent.note_external_reset(used)
            current_view = agent._build_view(obs, env)
            continue

        actions_in_game += 1

        new_state = state_name(obs)
        new_levels = levels_completed(obs)
        leveled_up = new_levels > prev_view.levels_completed
        game_over = new_state == "GAME_OVER"

        current_view = agent.observe_transition(
            prev_view,
            candidate_idx,
            obs,
            env,
            game_over=game_over,
            level_up=leveled_up,
        )

        if leveled_up:
            current_view = agent.start_level(obs, env)

        if new_state == "WIN":
            completed_flag = True
            break

    return obs, actions_in_game, completed_flag




def continue_with_source_then_best_online_agent(
    env: Any,
    game_id: str,
    obs: Any,
    actions_in_game: int,
    current_level_actions: int,
    source_planner: Optional[SourcePlanner] = None,
    game_index: int = 0,
    total_games: int = 1,
    diverged: bool = False,
) -> tuple[Any, int, bool, dict[str, Any]]:
    planner_meta: dict[str, Any] = {
        "planner_used": bool(source_planner is not None),
        "planner_solved_levels": 0,
        "planner_attempted_levels": 0,
        "planner_level_action_counts": [],
        "planner_error": "",
    }
    completed_flag = state_name(obs) == "WIN"

    if (
        source_planner is not None
        and not diverged
        and state_name(obs) not in {"WIN", "GAME_OVER"}
        and int(current_level_actions) == 0
    ):
        while time.time() - GLOBAL_START <= GLOBAL_WALLCLOCK_LIMIT_S - 2.0:
            level_idx = int(levels_completed(obs))
            if state_name(obs) in {"WIN", "GAME_OVER"}:
                break
            if actions_in_game >= game_budget(level_idx + 1):
                break

            prev_solution = source_planner.solution_cache.get(level_idx - 1)
            remaining_global = max(0.0, GLOBAL_WALLCLOCK_LIMIT_S - (time.time() - GLOBAL_START))
            expected_games_left = max(1, int(total_games) - int(game_index))
            level_budget_s = planner_time_budget_seconds(level_idx, has_transfer=prev_solution is not None)
            level_budget_s = min(level_budget_s, max(4.0, remaining_global / expected_games_left))

            planner_meta["planner_attempted_levels"] += 1
            try:
                plan = source_planner.solve_level(level_idx, prev_solution=prev_solution, time_budget_s=level_budget_s)
            except Exception as exc:
                planner_meta["planner_error"] = repr(exc)
                traceback.print_exc()
                plan = None

            if not plan:
                break

            replay_solved = False
            used_this_level = 0

            for step_id, (aid, data) in enumerate(plan):
                if actions_in_game >= game_budget(level_idx + 1):
                    break

                reasoning = json.dumps(
                    {
                        "solver": "source_planner_online",
                        "game_id": game_id,
                        "level_idx": int(level_idx),
                        "step_id": int(step_id),
                        "action_id": int(aid),
                        "data": data,
                    },
                    ensure_ascii=False,
                )
                try:
                    obs = take_action(env, int(aid), data=data, reasoning=reasoning)
                except Exception as exc:
                    planner_meta["planner_error"] = repr(exc)
                    break

                actions_in_game += 1
                used_this_level += 1
                current_level_actions += 1

                new_state = state_name(obs)
                new_levels = levels_completed(obs)

                if new_state == "GAME_OVER":
                    replay_solved = False
                    break
                if new_state == "WIN":
                    planner_meta["planner_solved_levels"] += 1
                    planner_meta["planner_level_action_counts"].append(int(used_this_level))
                    completed_flag = True
                    current_level_actions = 0
                    replay_solved = True
                    break
                if new_levels > level_idx:
                    planner_meta["planner_solved_levels"] += 1
                    planner_meta["planner_level_action_counts"].append(int(used_this_level))
                    current_level_actions = 0
                    replay_solved = True
                    break

            if completed_flag:
                return obs, actions_in_game, True, planner_meta

            if replay_solved:
                current_level_actions = 0
                continue

            break

    obs, actions_in_game, completed_flag = continue_with_best_online_agent(
        env,
        game_id,
        obs,
        actions_in_game=actions_in_game,
        current_level_actions=current_level_actions,
    )
    return obs, actions_in_game, completed_flag, planner_meta

GLOBAL_START = time.time()
GLOBAL_WALLCLOCK_LIMIT_S = 5.55 * 60 * 60

arc = make_arcade()

games_raw = arc.get_environments()
game_ids = normalize_game_ids(games_raw)
if not game_ids:
    raise RuntimeError("No games returned by ARC toolkit / gateway.")

LOCAL_ENV_INFOS = scan_local_environment_infos(ENVIRONMENTS_DIR)
LOCAL_PREFIX_INDEX = build_prefix_index(LOCAL_ENV_INFOS)

internal_rows: dict[str, dict[str, Any]] = {}
game_summaries: list[dict[str, Any]] = []

for game_index, game_id in enumerate(game_ids):
    if time.time() - GLOBAL_START > GLOBAL_WALLCLOCK_LIMIT_S - 60.0:
        break

    env = None
    obs = None
    actions_in_game = 0
    completed_flag = False
    replayed_count = 0
    current_level_actions = 0
    diverged = False

    env_info = None
    source_planner_obj: Optional[SourcePlanner] = None
    planner_meta: dict[str, Any] = {
        "planner_used": False,
        "planner_solved_levels": 0,
        "planner_attempted_levels": 0,
        "planner_level_action_counts": [],
        "planner_error": "",
    }

    offline18_result = empty_result()
    source_result = empty_result()
    bfs_result = empty_result()
    offline_result = empty_result()

    try:
        env_info = resolve_local_env_info(game_id, LOCAL_ENV_INFOS, LOCAL_PREFIX_INDEX)
        baseline_actions = list(env_info.baseline_actions) if env_info is not None else []

        source_path = None
        class_name = None
        if env_info is not None:
            try:
                source_path, class_name = resolve_source_path_from_env_info(env_info)
            except Exception:
                source_path, class_name = None, None
        if not source_path or not class_name:
            try:
                source_path, class_name = _find_game_source_and_class(game_id, env=None)
            except Exception:
                source_path, class_name = None, None

        if env_info is not None:
            try:
                offline18_result = plan_game_offline(env_info, game_index=game_index, total_games=len(game_ids))
            except Exception:
                traceback.print_exc()
                offline18_result = empty_result("offline18 failed")

        weak_for_source = should_try_source_planner_offline(offline18_result, baseline_actions)
        weak_for_bfs = should_try_bfs(offline18_result, baseline_actions)

        if source_path and class_name:
            if weak_for_source:
                try:
                    source_result, source_planner_obj = plan_game_with_source_planner(
                        game_id=game_id,
                        env_info=env_info,
                        source_path=source_path,
                        class_name=class_name,
                        game_index=game_index,
                        total_games=len(game_ids),
                    )
                except Exception:
                    traceback.print_exc()
                    source_result = empty_result("source planner failed")
                    source_planner_obj = None

            if source_planner_obj is None:
                try:
                    source_planner_obj = SourcePlanner(game_id, source_path, class_name)
                    if not source_planner_obj.load():
                        source_planner_obj = None
                except Exception:
                    source_planner_obj = None

            if weak_for_bfs:
                try:
                    bfs_result, _bfs_solver = plan_game_with_bfs(
                        game_id=game_id,
                        env_info=env_info,
                        source_path=source_path,
                        class_name=class_name,
                        game_index=game_index,
                        total_games=len(game_ids),
                    )
                except Exception:
                    traceback.print_exc()
                    bfs_result = empty_result("bfs failed")

        offline_result = choose_conservative_offline_result(
            offline18_result=offline18_result,
            source_result=source_result,
            bfs_result=bfs_result,
            baseline_actions=baseline_actions,
        )

        env = make_env(arc, game_id)

        obs, actions_in_game, current_level_actions, replayed_count, diverged = replay_plan_online(
            env,
            offline_result.get("tokens", []) or [],
            offline_result.get("expected_trace", []) or [],
        )

        if diverged and state_name(obs) not in {"WIN", "GAME_OVER"} and current_level_actions > 0:
            try:
                obs, used = perform_reset(env, reason="recover after offline divergence", max_attempts=1)
                actions_in_game += used
                current_level_actions = 0
            except Exception:
                pass

        if state_name(obs) == "WIN":
            completed_flag = True
        elif time.time() - GLOBAL_START <= GLOBAL_WALLCLOCK_LIMIT_S - 5.0:
            obs, actions_in_game, completed_flag, planner_meta = continue_with_source_then_best_online_agent(
                env,
                game_id,
                obs,
                actions_in_game=actions_in_game,
                current_level_actions=current_level_actions,
                source_planner=source_planner_obj,
                game_index=game_index,
                total_games=len(game_ids),
                diverged=diverged,
            )

    except Exception:
        traceback.print_exc()
    finally:
        if env is not None and hasattr(env, "close"):
            try:
                env.close()
            except Exception:
                pass

    final_levels = levels_completed(obs) if obs is not None else int(offline_result.get("levels_completed", 0) or 0)
    final_state = state_name(obs) if obs is not None else str(offline_result.get("final_state", "") or "")
    completed_flag = bool(completed_flag or final_state == "WIN")

    estimated_score = 0.0
    try:
        estimated_score = float(offline_result.get("estimated_score", 0.0) or 0.0)
    except Exception:
        estimated_score = 0.0

    if env_info is not None:
        try:
            combined_counts = list(offline_result.get("level_action_counts", []) or [])
            combined_counts.extend(list(planner_meta.get("planner_level_action_counts", []) or []))
            if combined_counts:
                estimated_score = max(
                    float(estimated_score),
                    float(estimate_game_score(env_info.baseline_actions, combined_counts)),
                )
        except Exception:
            pass
    internal_rows[game_id] = {
        "row_id": f"{game_id}_0",
        "game_id": game_id,
        "end_of_game": bool(completed_flag),
        "score": estimated_score,
        "levels_completed": int(final_levels),
        "actions": int(actions_in_game),
        "state": final_state,
        "offline_actions": int(len(offline_result.get("tokens", []) or [])),
        "replayed_actions": int(replayed_count),
        "offline_error": " | ".join(
            x
            for x in [
                str(offline18_result.get("error", "") or ""),
                str(source_result.get("error", "") or ""),
                str(bfs_result.get("error", "") or ""),
                str(offline_result.get("error", "") or ""),
                str(planner_meta.get("planner_error", "") or ""),
            ]
            if x
        ),
        "offline_sources": ",".join(offline_result.get("sources_by_level") or []),
        "diverged": bool(diverged),
        "planner_used": bool(planner_meta.get("planner_used", False)),
        "planner_solved_levels": int(planner_meta.get("planner_solved_levels", 0) or 0),
        "planner_attempted_levels": int(planner_meta.get("planner_attempted_levels", 0) or 0),
    }
    game_summaries.append(internal_rows[game_id])

scorecard = None
if hasattr(arc, "close_scorecard"):
    try:
        scorecard = arc.close_scorecard()
    except TypeError:
        try:
            scorecard = arc.close_scorecard(None)
        except Exception:
            scorecard = None
    except Exception:
        scorecard = None

plain_scorecard = to_plain(scorecard)

if isinstance(plain_scorecard, dict):
    env_entries = plain_scorecard.get("environments") or plain_scorecard.get("scores") or []
    for env_item in env_entries:
        if not isinstance(env_item, dict):
            continue
        gid = str(env_item.get("id") or env_item.get("game_id") or "")
        if not gid:
            continue
        row = internal_rows.get(
            gid,
            {
                "row_id": f"{gid}_0",
                "game_id": gid,
                "end_of_game": False,
                "score": 0.0,
            },
        )
        score_val = env_item.get("score", row.get("score", 0.0))
        completed_val = env_item.get("completed", row.get("end_of_game", False))
        try:
            score_val = float(score_val or 0.0)
        except Exception:
            score_val = 0.0
        row["score"] = max(0.0, score_val)
        row["end_of_game"] = bool(completed_val)
        internal_rows[gid] = row

rows = []
for gid in game_ids:
    row = internal_rows.get(
        gid,
        {"row_id": f"{gid}_0", "game_id": gid, "end_of_game": False, "score": 0.0},
    )
    rows.append(
        {
            "row_id": row["row_id"],
            "game_id": row["game_id"],
            "end_of_game": bool(row["end_of_game"]),
            "score": float(row["score"]),
        }
    )

submission = pd.DataFrame(rows, columns=["row_id", "game_id", "end_of_game", "score"])
submission.to_parquet(SUBMISSION_PATH, index=False)
submission = pd.read_parquet(SUBMISSION_PATH)


In [ ]:
print(submission.head(25))
print(f"Average score of first 25 games: {float(submission.head(25)['score'].mean()):.6f}")